# 다음 식사 메뉴 추천 (Group 33)

## 문제 한 줄
한양대 왕십리 주변에서 지금 영업 중이고 걸어서 15분 안에 갈 수 있는 식당을 골라, 최근 6끼와 덜 겹치는 메뉴를 서로 다른 식당에서 3개 추천합니다.

## 실행 방법
1. 상단 메뉴에서 `런타임 → 모두 실행`을 누릅니다. Google에서 작성하지 않은 노트북이라는 경고 창이 나오면 계속 실행을 선택합니다.
2. 맨 아래 `C4 상위 메뉴 선택, 출력` 섹션의 코드 셀 출력에서 추천 메뉴 3개와 추천 이유를 확인합니다.

API 키 없이도 마지막 셀까지 실행됩니다. 키가 없으면 예시 식사 기록과 미리 저장한 식당 3곳으로 추천합니다. 추천 기준 끼니는 실행한 한국 시각으로 정합니다. 10:30부터 16:00 전까지는 점심, 16:00부터 22:30 전까지는 저녁, 그 밖의 시간은 아침입니다.

실제 위치로 검색하려면 Kakao REST API 키가 필요합니다. 왼쪽 열쇠 아이콘(보안 비밀)에 `KAKAO_REST_KEY`라는 이름으로 키를 추가하고 노트북 액세스를 켠 뒤 다시 모두 실행하면, 3번 셀에서 출발지 주소를 묻습니다. 입력한 주소와 좌표는 Kakao로 전송됩니다.

내 식사 기록으로 추천받으려면 0번 셀의 `TEST_MODE = True`를 `False`로 바꿉니다. 그러면 2번 셀에서 직전 6끼를 차례로 묻습니다. 기억나지 않는 끼니는 Enter, 거른 끼니는 `안 먹음`을 입력합니다. 이때 키를 등록하지 않았다면 3번 셀에서 키를 직접 입력하는 칸이 나옵니다.

## 팀원별 담당 함수
- 장민엽: `load_test_menu_csv`, `walking_route`, `normalize_data`, `prepare_history`, `straight_distance_m`, `location_from_address`, `csv_opening_status`, `csv_restaurant_candidates`
- 노경민: `validate_data`, `filter_candidates`
- 안영빈: `clean_category`, `lookup_category`, `overlap_points`, `meal_number`, `score_candidates`, `ranking_order`, `score_sort`, `select_top3`, `display_results`

## 데이터 설명과 출처
| 데이터 | 내용 | 출처 |
|---|---|---|
| 식당 메뉴 CSV | 한양대 왕십리 주변 식당 314곳의 메뉴 3,795행입니다. 식당명, 주소, 좌표, 업종, 메뉴명, 가격, 메뉴 분류, 요일별 영업시간 등 20개 열이 있습니다. 0번 셀에 문자열로 들어 있어 따로 업로드할 필요가 없습니다. | 지도 서비스의 식당 정보로 뼈대를 만들고 나머지 값은 팀이 임의로 채운 더미 데이터 |
| 최근 6끼 식사 기록 | `TEST_MODE = True`이면 김밥, 된장찌개, 짜장면, 비빔밥 등이 들어 있는 예시 기록을 씁니다. `False`이면 실행할 때 직접 입력합니다. | 예시 기록은 직접 만든 더미 데이터 |
| 출발지 좌표 | 실행 중 입력한 도로명 주소나 지번 주소를 좌표로 바꿉니다. | Kakao Local API 주소 검색 |
| 도보 거리와 시간 | 출발지에서 가까운 식당부터 최대 500곳의 도보 경로를 조회합니다. | Kakao 도보 경로 조회 API |
| 키가 없을 때 쓰는 예시 식당 | 왕십리로 222에서 미리 조회한 식당 3곳의 도보 거리와 시간입니다. 영업 여부는 영업 중으로 고정합니다. | Kakao 도보 경로 조회 결과를 저장한 값 |

CSV에는 임의로 채운 값이 섞여 있어 실제 메뉴, 가격, 영업시간과 다를 수 있습니다. 영업 여부는 CSV의 요일별 영업시간과 실행 시각을 비교해 판단합니다.


## 0. 실행 설정과 식당 CSV

- `TEST_MODE`: 최근 6끼만 예시로 채우는 설정
- 두 모드 모두 같은 CSV 사용. 업로드나 Google Drive 연결 불필요

### `load_test_menu_csv`
#### 입력
- `TEST_MENU_CSV_TEXT`: 노트북에 포함된 최종 CSV 문자열 (함수 인자 없음)

#### 출력
- `test_menu_data`: 원본 열을 모두 보존한 `DataFrame`
  - 모든 값은 문자열, 빈 값은 `''`


In [ ]:
# @title 테스트용 CSV 원본 { display-mode: "form" }
# 최종 CSV 원문 (hanyang_wangsimni_building_coordinates.csv)
TEST_MENU_CSV_TEXT = (
    '\ufeff식당_ID,식당명,주소,위도,경도,업종,메뉴명,가격_원,가격_상태,메뉴_대분류,음식유형,주문형태,메뉴역할,월요일,화요일,수요일,목요일,금요일,토요일,일요일\r\n'
    'Rdd1c4f5c648b,프레퍼스 한양대플라자점,서울 성동구 왕십리로 222 학생복지관 3층 (우)04763,37.5563210,127.0439713,샐러드,비프 플레이트 샐러드,12900,추정가,샐러드,샐러드,단품,식사,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,09:30-20:30,09:30-20:30\r\n'
    'Rdd1c4f5c648b,프레퍼스 한양대플라자점,서울 성동구 왕십리로 222 학생복지관 3층 (우)04763,37.5563210,127.0439713,샐러드,치킨 콥 플레이트,11900,추정가,튀김·커틀릿,치킨,단품,식사,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,09:30-20:30,09:30-20:30\r\n'
    'Rdd1c4f5c648b,프레퍼스 한양대플라자점,서울 성동구 왕십리로 222 학생복지관 3층 (우)04763,37.5563210,127.0439713,샐러드,치킨 플레이트 샐러드,10900,추정가,튀김·커틀릿,치킨,단품,식사,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,09:30-20:30,09:30-20:30\r\n'
    'Rdd1c4f5c648b,프레퍼스 한양대플라자점,서울 성동구 왕십리로 222 학생복지관 3층 (우)04763,37.5563210,127.0439713,샐러드,포크 플레이트 샐러드,11900,추정가,샐러드,샐러드,단품,식사,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,08:30-20:30,09:30-20:30,09:30-20:30\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,반계탕,8500,추정가,국물요리,삼계탕·백숙,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,삼계탕,12000,추정가,국물요리,삼계탕·백숙,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,김치전골,18000,추정가,국물요리,전골,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,된장찌개,6000,추정가,국물요리,찌개류,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,부대찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,순두부찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,떡만두,6000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,비빔밥,6000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,냉면(물냉면),6000,표시 가격 확인,국수류,냉면,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,김치덮밥,6000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,카레덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,제육덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,오징어덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,특라면,4000,표시 가격 확인,국수류,라면,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,궁금한 오늘의 메뉴,6000,표시 가격 확인,여러 요리 구성,백반,세트,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,삼겹살(셋트),12000,표시 가격 확인,구이·볶음·찜,돼지고기구이,세트,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,닭도리탕,25000,추정가,구이·볶음·찜,닭볶음탕,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,두부김치,10000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,제육볶음,10000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,계란말이,7000,표시 가격 확인,곁들임,계란,단품,사이드,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,김치전,7000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'Rcf0b6cca7289,밥한술 (참새와방앗간),서울 성동구 광나루로 156 신양빌딩 1층,37.5514275,127.0475642,한식,부추전,7000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,08:00-14:30,휴무,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,원조김밥,3300,표시 가격 확인,분식,김밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,참치김밥,4000,표시 가격 확인,분식,김밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,소고기김밥,4300,표시 가격 확인,분식,김밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,치즈김밥,3700,표시 가격 확인,분식,김밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,고추김밥,3700,표시 가격 확인,분식,김밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,고추참치김밥,4500,표시 가격 확인,분식,김밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,라면,3800,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,치즈라면,4300,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,떡라면,4800,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,만두라면,4800,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,김치라면,4300,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,매콤참치라면,5000,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,해물라면,6000,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,너~구리!!,4300,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,짜파게티,4300,표시 가격 확인,국수류,라면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,냉메밀국수,7000,표시 가격 확인,국수류,모밀·소바,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,어묵우동,5500,표시 가격 확인,국수류,우동,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,김치우동,6000,표시 가격 확인,국수류,우동,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,떡국,6500,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,만둣국,6500,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,떡만둣국,6500,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,쫄면,6500,표시 가격 확인,국수류,쫄면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,물냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,비빔냉면,7500,표시 가격 확인,국수류,냉면,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,비빔국수,6000,표시 가격 확인,국수류,국수,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,콩국수,7000,표시 가격 확인,국수류,국수,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,메밀비빔막국수,7500,표시 가격 확인,국수류,막국수,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,국물떡볶이,4900,표시 가격 확인,분식,떡볶이,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,왕푸짐쌀떡볶이,6900,표시 가격 확인,분식,떡볶이,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,로제떡볶이,12000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,궁중떡볶이,5500,표시 가격 확인,분식,떡볶이,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,삼삼떡볶이,14900,표시 가격 확인,분식,떡볶이,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,스페셜라볶이,7900,표시 가격 확인,분식,떡볶이,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,모듬튀김,5900,표시 가격 확인,튀김·커틀릿,튀김,모둠,사이드,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,야채비빔밥,6500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,소고기비빔밥,7000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,제육비빔밥,7000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,참치비빔밥,7000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,돈까스,7500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,치즈돈까스,8000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,고구마치즈돈까스,8000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,철판제육,8500,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,참치덮밥,7000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,오징어덮밥,8500,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,햄참치마요,7000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,햄치킨마요,7500,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,카레라이스,7000,표시 가격 확인,커리·스튜,커리,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,김치볶음밥,6000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,오므라이스,7000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,뚝불고기,9000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,차돌된장,7500,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,순두부찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,김치찌개,6500,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,참치찌개,6500,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,부대찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,갈비탕,9000,표시 가격 확인,국물요리,갈비탕,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,육개장,9000,표시 가격 확인,국물요리,육개장,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,후라이,800,표시 가격 확인,곁들임,달걀,추가,추가옵션,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,치즈,500,표시 가격 확인,곁들임,치즈,추가,추가옵션,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,곱배기,2000,표시 가격 확인,추가옵션,곱빼기,추가,추가옵션,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,라면사리,1000,표시 가격 확인,국수류,라면,추가,추가옵션,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,만두,2000,표시 가격 확인,분식,만두,추가,추가옵션,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R5d66493feb24,면24 한양대점,서울 성동구 왕십리로 222 한양대학교 학생복지관 1층 5호,37.5563210,127.0439713,한식·분식,공깃밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,11:00-16:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,까르보나라,25000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,뇨끼,23000,표시 가격 확인,파스타류,뇨끼,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,리가토니 라구,26000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,바라문디,35000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,봉골레,26000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,부라타,20000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,비스크 꼰길리에,26000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,뽈뽀,25000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,새우 세비체,23000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,시저샐러드,17000,표시 가격 확인,샐러드,샐러드,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,앤쵸비 보타르가,25000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,우니 딸리올리니,32000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,젤라또,8000,표시 가격 확인,디저트,아이스크림,단품,디저트,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,트러플,30000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,티라미슈,10000,표시 가격 확인,디저트,케이크,단품,디저트,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,포르치니 리조또,24000,표시 가격 확인,밥·죽류,리조또,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,푸타네스카,23000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,풍기,20000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,한우 채끝등심 200g,80000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R86c646cae74a,파리나,서울 성동구 광나루로 144 더스페이스타워 1층 110호 (우)04788,37.5513018,127.0464325,이탈리안,한우 타르타르,24000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[고소한] 전복&가리비관자 솥밥,16800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[냉] 고등어 오차즈케,15800,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[냉] 명란 오차즈케,14800,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[냉] 연어 오차즈케 (기름NO),15800,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[담백] 가자미&가리비관자 솥밥,16800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[매콤] 쭈꾸미 솥밥,16800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[버터] 옥수수 솥밥,10800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[버터로구운] 고등어구이,7800,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[사장님추천] 부타동 솥밥,15300,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[소대창] 호르몬동 솥밥,17800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[시그니처] 스테이크 솥밥,16800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[알이톡톡] 명란 솥밥,16200,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[염장다시마] 시오콘부 솥밥,10800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[재주문1위] 민물장어 솥밥,25800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[제로]코카콜라 355mL,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접구운] 연어 솥밥 (기름NO),16800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 연어카츠4P+타르타르소스+생와사비,12800,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 온천계란튀김2P,2800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 온천계란튀김4P,4800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 치즈계란튀김2P,2800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 치즈계란튀김4P,4800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 하나계란튀김2P,2800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[직접만든] 하나계란튀김4P,4800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,[클래식] 온센베이컨 솥밥,11800,표시 가격 확인,밥·죽류,솥밥,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,유린기,9800,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,칠성사이다 355mL,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,코카콜라 355mL,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,통새우튀김1P+직접만든타르타르소스,2800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,통새우튀김2P+직접만든타르타르소스,5200,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Reb3288eb8b88,호호솥밥 서울성수점,서울 성동구 상원길 82 1층 (우)04791,37.5512924,127.0475415,한식,통새우튀김3P+직접만든타르타르소스,7200,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R300801e631e2,성수뚝배기,서울 성동구 상원길 82 광성빌딩 1층 (우)04791,37.5512924,127.0475415,"찌개,전골",고추장찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,휴무,휴무\r\n'
    'R300801e631e2,성수뚝배기,서울 성동구 상원길 82 광성빌딩 1층 (우)04791,37.5512924,127.0475415,"찌개,전골",김치찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,휴무,휴무\r\n'
    'R300801e631e2,성수뚝배기,서울 성동구 상원길 82 광성빌딩 1층 (우)04791,37.5512924,127.0475415,"찌개,전골",뚝배기제육,10000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,휴무,휴무\r\n'
    'R300801e631e2,성수뚝배기,서울 성동구 상원길 82 광성빌딩 1층 (우)04791,37.5512924,127.0475415,"찌개,전골",순두부찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,휴무,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,고등어 파스타,19000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,뇨끼 제노베제,23000,표시 가격 확인,파스타류,뇨끼,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,딸기콤포트 판나코타,7000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,레드와인,73000,추정가,음료,와인,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,봉골레,19000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,비프 라구 라자냐,25000,표시 가격 확인,파스타류,라자냐,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,시저샐러드,18000,표시 가격 확인,샐러드,샐러드,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,아페롤스프리츠,10000,표시 가격 확인,국물요리,수프,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,아페르티보 플레이트,18000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,양고기 라구 탈리아텔레,23000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,어니언슾,10000,표시 가격 확인,국물요리,수프,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,와규 플랭크 스테이크,39000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,이베리코 갈비살 탈리아텔레,25000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,참송이 크림 리소토,23000,표시 가격 확인,밥·죽류,리조또,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,초당 옥수수 스프 (cold),10000,표시 가격 확인,국물요리,수프,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,치킨룰라드,29000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,푸타네스카,19000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'Rd82b5081aa99,모음,서울 성동구 살곶이8길 22 Moeum,37.5563145,127.0429786,양식,화이트와인,55000,추정가,음료,와인,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,고등어자반구이반상,17000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,물회,22000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,보리굴비반상,24000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,소고기미역국,13000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,소불고기반상,17000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,오색나물비빔밥,14000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,한우투뿔육회비빔밥,20000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R2987bbf9d8e2,이십사절기 한양대점,서울 성동구 왕십리로 220 한양대학교 HIT 6층,37.5577278,127.0467977,한정식,회덮밥,20000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'Rfbf13a5613f6,엄마손한식뷔페,서울 성동구 왕십리로 213 지하1층,37.5565554,127.0430160,한식뷔페,기본 백반,8000,표시 가격 확인,여러 요리 구성,백반,단품,식사,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30\r\n'
    'Rfbf13a5613f6,엄마손한식뷔페,서울 성동구 왕십리로 213 지하1층,37.5565554,127.0430160,한식뷔페,제육볶음,6000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30,08:00-19:30\r\n'
    'R7cc7976de08b,밥플러스 아이티캐슬점,서울 성동구 광나루로 130,37.5509781,127.0447234,한식뷔페,,8500,대표가 추정(메뉴 미등록),,,,,11:00-19:30,11:00-19:30,11:00-19:30,11:00-19:30,11:00-14:00,휴무,휴무\r\n'
    'R3ceb41504cfb,본초돈가스,서울 성동구 왕십리로 215,37.5567272,127.0429387,돈가스,라돈세트,10000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,휴무\r\n'
    'R3ceb41504cfb,본초돈가스,서울 성동구 왕십리로 215,37.5567272,127.0429387,돈가스,모돈세트,10000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,휴무\r\n'
    'R3ceb41504cfb,본초돈가스,서울 성동구 왕십리로 215,37.5567272,127.0429387,돈가스,본초돈가스,8000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,휴무\r\n'
    'R3ceb41504cfb,본초돈가스,서울 성동구 왕십리로 215,37.5567272,127.0429387,돈가스,우돈세트,10000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,휴무\r\n'
    'R3ceb41504cfb,본초돈가스,서울 성동구 왕십리로 215,37.5567272,127.0429387,돈가스,치라돈세트,10500,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,휴무\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",가마타마,9000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",가조쿠우동,9000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",라유멘,10000,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",면추가,2500,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",붓가케우동,7500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",새우튀김,2500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",육육우동,9500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",치즈육육우동,11500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",치즈카레우동,12000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",카레우동,10000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rc176fe1fa7f6,우동가조쿠,서울 성동구 왕십리로 215-1,37.5567679,127.0429100,"우동,소바",키츠네우동,8000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Ra8b24bdbb2fc,의정부부대찌개 한양대점,서울 성동구 왕십리로 217 1층,37.5568263,127.0427918,"찌개,전골",오뎅만두 부대찌개,8500,표시 가격 확인,분식,어묵,단품,사이드,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무\r\n'
    'Ra8b24bdbb2fc,의정부부대찌개 한양대점,서울 성동구 왕십리로 217 1층,37.5568263,127.0427918,"찌개,전골",의정부 부대찌개,7500,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,고르곤졸라,11600,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,까르보나라,10600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,닭안심 샐러드,10600,표시 가격 확인,샐러드,샐러드,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,로제새우날치알파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,로제파스타,10600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,로제해산물리조또,11600,표시 가격 확인,밥·죽류,리조또,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,마레 로제파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,마레크림파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,마레토마토파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,베이컨로제파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,불닭까르보나라,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,뽀모도로,10600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,엔초비루꼴라오일파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,우삼겹토마토파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,우삼겹할라피뇨오일파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,카레토마토파스타,11600,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,크림베이컨리조또,11600,표시 가격 확인,밥·죽류,리조또,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'R98fddb119c3a,제이스플레이트,서울 성동구 왕십리로 219-1,37.5569885,127.0427125,양식,해산물 샐러드,11600,표시 가격 확인,샐러드,샐러드,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-19:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,디너 테이스팅 코스,138000,추정가,여러 요리 구성,세부유형 미정,코스,식사,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,문어 감자,26000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,스텔라 생맥주 330ml,7000,표시 가격 확인,음료,맥주,단품,음료,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,평일 런치 코스,34000,추정가,여러 요리 구성,세부유형 미정,코스,식사,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,홍새우 크루도,19000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,Etna,14000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,Pasta limone,21000,표시 가격 확인,파스타류,파스타,단품,식사,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'Rffe97e7b5e39,뻬따,서울 성동구 왕십리로16가길 10-30 1층,37.5506175,127.0453343,이탈리아음식,Petta ragu,14000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,18:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,국밥,6500,표시 가격 확인,국물요리,국밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,껌승,8000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,닭덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,매운쌀국수,10000,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,반꾸온,7000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,반미,6500,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,반쎄오,14000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,볶음밥,8000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,볶음쌀국수,8000,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,분짜,6900,표시 가격 확인,국수류,국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,비빔쌀국수,6900,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,쌀국수,8000,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,오징어덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,제육덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R21065afaebcf,포사이공베트남쌀국수,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,짜조,4000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,갈비만두,3500,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,김치피만두,3500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,도토리냉맛면,6000,표시 가격 확인,국수류,국수,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,도토리닭강정 1박스,6500,표시 가격 확인,튀김·커틀릿,닭강정,단품,사이드,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,도토리닭강정 특,9500,표시 가격 확인,튀김·커틀릿,닭강정,단품,사이드,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,도토리온맛면,6000,표시 가격 확인,국수류,국수,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,매생이감자옹심이,7000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,매생이굴국밥,6500,표시 가격 확인,국물요리,국밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,매생이떡국,6000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd1b3d701b460,묵사발 한양대점,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,한식,묵사발,6000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,껌스언,9500,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,매운쌀국수,8000,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,반꾸온,8000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,반미,7500,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,반쎄오,14000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,볶음밥,6500,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,볶음쌀국수,7000,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,분팃싸오,7900,표시 가격 확인,국수류,국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,쌀국수,7900,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,오징어덮밥,7000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,제육덮밥,7000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rd3d6f10a299d,포싸이공,서울 성동구 왕십리로 221-5 1층 (우)04764,37.5569294,127.0426312,베트남음식,짜조,4500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-18:00,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,가지버섯 파니니,13000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,사과감자 샌드위치,10000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,시저치킨 샌드위치,10000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,월넛 애플 루꼴라 샐러드,12000,표시 가격 확인,샐러드,샐러드,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,잠봉 모짜렐라 파니니,13000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,잠봉고다 샌드위치,10000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,치폴레치킨 파니니,13000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,클래식 루벤샌드위치,16000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,프로슈토 바질페스토 파나니,13000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'Rf5176e055c9c,멜드 샌드위치샵,서울 성동구 상원10나길 23 1층,37.5502686,127.0482215,샌드위치,허브치킨 샌드위치,11000,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,휴무,07:00-15:30,07:00-15:30,07:00-15:30,07:00-15:30,09:00-15:30,휴무\r\n'
    'R54cd5c5670ab,예반,서울 성동구 상원12길 12-1 일층,37.5504448,127.0491592,한식,한식 백반,8000,표시 가격 확인,여러 요리 구성,백반,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-14:00,휴무\r\n'
    'R9615982fb06a,정오아카데미 한양대점,서울 성동구 왕십리로 222,37.5575609,127.0463899,한식,,7000,대표가 추정(메뉴 미등록),,,,,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,휴무,휴무\r\n'
    'R96958b0535dc,피그,서울 성동구 왕십리로16가길 4 1층,37.5501042,127.0447003,양식,,16000,대표가 추정(메뉴 미등록),,,,,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,돼지 양념갈비,17000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,돼지고기 모둠(1kg),69900,표시 가격 확인,구이·볶음·찜,돼지고기구이,모둠,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,돼지고기 반모둠(500g),54900,표시 가격 확인,구이·볶음·찜,돼지고기구이,모둠,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,돼지목살 김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,생삼겹살(150g),11000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,소고기 모둠(1kg),79900,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,소고기 반모둠(500g),64900,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,소고기 육회,25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,소불고기,10000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rf53a9cf9a423,우땅 본점,서울 성동구 살곶이길 346,37.5562194,127.0411276,한식,제육 불고기,9000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,(리뷰두 찜두),1,표시 가격 확인,비메뉴,안내·비용,해당 없음,비메뉴,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합] 고기만두튀김 2개,3000,표시 가격 확인,분식,만두,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합] 고기만두튀김 4개,5500,표시 가격 확인,분식,만두,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합] 곤드레까만볶음밥,4900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합] 묵은지김치볶음밥,4900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합세트] 까만찜닭,35900,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합세트] 묵은지찜닭,38900,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합세트] 실비한우곱찜닭,42900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합세트] 한우대창곱도리탕,41900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[꿀조합세트] 한우대창맵닭발(무뼈),36900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,[원조K로제] 로제찜닭,30800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,180mm 긴말이 2개,3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,가지튀김 15개,3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,공기밥,1500,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,까만찜닭 반마리,20800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,꿔바로우 3개(유자소스 포함),4500,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"나홀로세트 (긴말이1, 통새우1, 공기밥1)",4500,표시 가격 확인,튀김·커틀릿,모둠튀김,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"납작세트 (돈까스1, 납작만두5)",4500,표시 가격 확인,튀김·커틀릿,모둠튀김,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,눈꽃감자 1봉,1000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,닭도리탕,30800,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,돈까스 1개,3000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,두찜 까만찜닭,27800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,두찜 실비한우곱찜닭,34800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,두찜 의성마늘곱도리탕,34800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,두찜 찜닭 + 찜닭,48800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,두찜 칼낙지찜닭 (절단낙지),34800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,두찜 한우대창곱도리탕,33800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"둥근세트 (치즈볼3, 감자만두5)",5000,표시 가격 확인,튀김·커틀릿,모둠튀김,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"든든세트 (고기만두2, 옛날통닭(반마리)",9000,표시 가격 확인,튀김·커틀릿,치킨,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,로제닭발(무뼈),27800,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,로제찜닭 반마리,21800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"롱롱세트 (긴말이2, 통새우튀김2)",6500,표시 가격 확인,튀김·커틀릿,모둠튀김,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,마라곤약(향라맛),700,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,마라로제찜닭,26800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"만두세트 (고기만두4, 납작만두5)",7000,표시 가격 확인,분식,만두,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,모치치(모짜렐라치즈치킨) 1개 추가,2500,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,모치치(모짜렐라치즈치킨) 2개 추가,4500,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,묵은지 김치찜 닭,30800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,불닭로제찜닭,26800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,빨간찜닭,27800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,빨간찜닭 반마리,20800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,사각사각 사각무 추가,1000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,서울식 납작만두 5개,2000,표시 가격 확인,분식,만두,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,스팸부대찜닭,30800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,스페셜+스페셜,48800,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,스프라이트 추가 1.5L,3500,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,스프라이트 추가 500ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,시래기찜닭,30800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,쌈무 추가,1300,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,의성마늘갈비찜닭,34800,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,찜닭+스페셜,48800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,치즈몽땅찜닭,31800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,치즈볼(3개),3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,치즈볼(5개),4500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,코카콜라 제로 추가 1.25L,3500,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,코카콜라 제로 추가 510ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,코카콜라 추가 1.25L,3500,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,코카콜라 추가 510ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,통새우튀김 2개,4000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,"튀김세트 (돈까스1, 긴말이2, 치즈볼3)",8000,표시 가격 확인,튀김·커틀릿,모둠튀김,세트,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,튀김오뎅 5개,2000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,트러플크림찜닭,30800,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Rd7f159e825ad,두찜 한양대점,서울 성동구 살곶이길 346-1,37.5563180,127.0410367,찜닭,한우대창맵닭발(무뼈),28800,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,[25% 할인] 레잇 디너 세트 2인,169000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,6가지 코스 메뉴,65000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,그랑마니에르 토닉,13000,표시 가격 확인,음료,칵테일,단품,음료,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,만다린 에이드,7000,표시 가격 확인,음료,에이드,단품,음료,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,본 인 립아이 스테이크 (700g),149000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,부드럽게 조리한 문어와 알감자,29000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,브로콜리 & 후무스,10000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,새우 로제 비스크 파스타,24000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,새우 리조또 튀김 (아란치니),12000,표시 가격 확인,밥·죽류,리조또,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,생면 라자냐 롤,24000,표시 가격 확인,파스타류,라자냐,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,스파이시 크림 파스타,17000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,시그니쳐 와규 스테이크,49000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,오렌지 & 케일 샐러드,17000,표시 가격 확인,샐러드,샐러드,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,오미자 하이볼,9000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,유자 하이볼,8000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,치즈 플레이트,22000,표시 가격 확인,곁들임,곁들임,단품,사이드,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,코너룸 카프레제 샐러드,17000,표시 가격 확인,샐러드,샐러드,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,포르치니 버섯 크림 뇨끼,24000,표시 가격 확인,파스타류,뇨끼,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,"프렌치 토스트, 메이플 앙글레이즈",14000,표시 가격 확인,분식,토스트,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,한우 1++ 티본 스테이크,169000,추정가,구이·볶음·찜,소고기구이,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,한치 오일 파스타,21000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'Ra5a48818bc57,코너룸 성수,서울 성동구 상원길 64 2층,37.5497944,127.0480180,양식,홈메이드 아이스크림,5000,표시 가격 확인,디저트,아이스크림,단품,디저트,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:30,11:30-22:30,11:30-22:00\r\n'
    'R2bb100f8e4f2,24시회마트,서울 동대문구 전농로4길 42 (답십리동) 지상 1층,37.5666943,127.0588703,횟집,광어,20000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00\r\n'
    'R2bb100f8e4f2,24시회마트,서울 동대문구 전농로4길 42 (답십리동) 지상 1층,37.5666943,127.0588703,횟집,광어+우럭,30000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00\r\n'
    'R2bb100f8e4f2,24시회마트,서울 동대문구 전농로4길 42 (답십리동) 지상 1층,37.5666943,127.0588703,횟집,우럭,20000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00,15:30-24:00\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,354김밥,4500,표시 가격 확인,분식,김밥,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,계란라면,4000,표시 가격 확인,국수류,라면,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,김치말이국수,8000,표시 가격 확인,국수류,국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,돈가스김밥,5800,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,묵참김밥,5500,표시 가격 확인,분식,김밥,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,바지락칼국수,8500,표시 가격 확인,국수류,칼국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,비빔국수,8000,표시 가격 확인,국수류,국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,소고기 김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,순두부찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,스팸땡초김밥,5500,표시 가격 확인,분식,김밥,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,어린이김밥,4000,표시 가격 확인,분식,김밥,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,잔치국수,7500,표시 가격 확인,국수류,국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,제육볶음,9000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,칼비빔,8500,표시 가격 확인,국수류,국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'R1a0aa33ae35f,354김밥국수,살곶이길 354 1층,37.5569171,127.0410035,김밥/만두/분식,콩국수,9000,표시 가격 확인,국수류,국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무,휴무\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,고구마돈까스,11000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,돈쫄세트,11000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,등심돈까스,9000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,모듬돈까스,12000,표시 가격 확인,튀김·커틀릿,돈가스,모둠,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,모듬롤까스,14000,표시 가격 확인,튀김·커틀릿,돈가스,모둠,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,반반까스,10000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,생선까스,10000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,치즈돈까스,11000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'Rf66ad6dc7133,5오돈까스,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,기타,치즈롤돈까스,11000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,휴무,10:00-21:00\r\n'
    'R46155e227e8e,가온길,"서울 성동구 독서당로62길 4, 1층 (응봉동)",37.5496683,127.0311180,한식,뼈해장국,10000,표시 가격 확인,국물요리,해장국,단품,식사,10:30-21:00,10:30-21:00,휴무,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R46155e227e8e,가온길,"서울 성동구 독서당로62길 4, 1층 (응봉동)",37.5496683,127.0311180,한식,제육볶음,10000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,10:30-21:00,10:30-21:00,휴무,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R9782ba2e8ebf,가조쿠식당,서울 성동구 왕십리로 213-1 1층,37.5566239,127.0429966,중국식,멘치카츠정식,11000,표시 가격 확인,튀김·커틀릿,함박스테이크,세트,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,휴무,휴무\r\n'
    'R9782ba2e8ebf,가조쿠식당,서울 성동구 왕십리로 213-1 1층,37.5566239,127.0429966,중국식,소스카츠동(일반/매운맛),10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,휴무,휴무\r\n'
    'R9782ba2e8ebf,가조쿠식당,서울 성동구 왕십리로 213-1 1층,37.5566239,127.0429966,중국식,카츠동,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,휴무,휴무\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,"갈비깍두기볶음밥(치즈추가+3,000)",10000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,갈포 시그니처 生갈비 세트,58000,표시 가격 확인,구이·볶음·찜,갈비구이,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,갈포 시그니처 양념갈비 세트,56000,표시 가격 확인,구이·볶음·찜,갈비구이,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,갈포 한상 세트 (4인 추천),117500,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,갈포화합세트,15000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,묵은지비빔국수,7000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,묵은지육회비빔밥(점심메뉴),12000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,바질된장술밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,생갈비김치찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,양념갈비정식(점심메뉴/2인이상),19500,표시 가격 확인,구이·볶음·찜,갈비구이,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,에그매쉬,9000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,오이지냉국수,7000,표시 가격 확인,국수류,국수,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제육쌈밥정식(점심메뉴/2인이상),14000,표시 가격 확인,구이·볶음·찜,제육볶음,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제육정식볶음(점심메뉴),11000,표시 가격 확인,구이·볶음·찜,제육볶음,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제철쌈채소 6~7종 + 갈포생갈비,21500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제철쌈채소 6~7종 + 갈포양념갈비,20500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제철쌈채소 6~7종 + 특상生갈비,23000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제철쌈채소 6~7종 + 화향삼겹살,18500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,제철쌈채소 6~7종 + LA갈비,28500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,투뿔한우육회,28000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,트러플 청어알,3000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,한우된장뚝배기,9000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R645bd2b93477,갈포갈비 왕십리점,"서울 성동구 행당로17길 22, 1층 (행당동, 292-45)",37.5602260,127.0332678,한식,해물쌈장,6000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R7a7203a0a3e2,갓잇 왕십리엔터식스점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 1층 (행당동)",37.5612890,127.0383818,기타,치킨브리또,9900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7a7203a0a3e2,갓잇 왕십리엔터식스점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 1층 (행당동)",37.5612890,127.0383818,기타,콤보보울,10900,표시 가격 확인,샐러드,포케,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7a7203a0a3e2,갓잇 왕십리엔터식스점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 1층 (행당동)",37.5612890,127.0383818,기타,포크브리또,9900,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R00503cf5ba15,강금옥쭈꾸미,서울 성동구 왕십리로24길 5 1층,37.5630101,127.0337404,해산물 구이/찜,눈꽃치즈쭈꾸미,17000,추정가,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R00503cf5ba15,강금옥쭈꾸미,서울 성동구 왕십리로24길 5 1층,37.5630101,127.0337404,해산물 구이/찜,쭈꾸미,16000,추정가,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R00503cf5ba15,강금옥쭈꾸미,서울 성동구 왕십리로24길 5 1층,37.5630101,127.0337404,해산물 구이/찜,쭈꾸미닭발,17000,추정가,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R00503cf5ba15,강금옥쭈꾸미,서울 성동구 왕십리로24길 5 1층,37.5630101,127.0337404,해산물 구이/찜,쭈꾸미대창,19000,추정가,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R00503cf5ba15,강금옥쭈꾸미,서울 성동구 왕십리로24길 5 1층,37.5630101,127.0337404,해산물 구이/찜,쭈꾸미삼겹,17000,추정가,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R00503cf5ba15,강금옥쭈꾸미,서울 성동구 왕십리로24길 5 1층,37.5630101,127.0337404,해산물 구이/찜,쭈꾸미새우,17000,추정가,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R7d4c9811bbca,강떡,"서울 성동구 사근동10길 4, 1층 (사근동)",37.5608561,127.0480429,분식,기본떡볶이세트,16000,표시 가격 확인,분식,떡볶이,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-22:00,11:00-22:00\r\n'
    'R7d4c9811bbca,강떡,"서울 성동구 사근동10길 4, 1층 (사근동)",37.5608561,127.0480429,분식,짜장떡볶이세트,16000,표시 가격 확인,분식,떡볶이,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-22:00,11:00-22:00\r\n'
    'R7d4c9811bbca,강떡,"서울 성동구 사근동10길 4, 1층 (사근동)",37.5608561,127.0480429,분식,치즈즉석떡볶이세트,18000,표시 가격 확인,분식,떡볶이,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-22:00,11:00-22:00\r\n'
    'R6517deb0b9a5,강떡즉석떡볶이,왕십리로21길 11 1층,37.5613084,127.0331460,김밥/만두/분식,기본즉석떡볶이,14000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-21:30,휴무,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30\r\n'
    'R6517deb0b9a5,강떡즉석떡볶이,왕십리로21길 11 1층,37.5613084,127.0331460,김밥/만두/분식,떡마니세트,14000,표시 가격 확인,분식,떡볶이,세트,식사,10:00-21:30,휴무,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30\r\n'
    'R6517deb0b9a5,강떡즉석떡볶이,왕십리로21길 11 1층,37.5613084,127.0331460,김밥/만두/분식,부대즉석떡볶이,16000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-21:30,휴무,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30\r\n'
    'R6517deb0b9a5,강떡즉석떡볶이,왕십리로21길 11 1층,37.5613084,127.0331460,김밥/만두/분식,아이짜장즉석떡볶이 1인분,8000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-21:30,휴무,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[꽃게폭탄] 꽃도리탕 닭 + 꽃게 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,34900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[모짜치즈] 닭도리탕 닭 + 모짜치즈 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,27900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[묵은지] 닭도리탕 닭 + 묵은지 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,28900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[새우폭탄] 새도리탕 닭 + 대하새우 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,34900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[순두부] 닭도리탕 닭 + 순두부 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,27900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[시그니처] 불향가득 제육볶음 고추장 목전지 + 양파 + 대파 + 공기밥,11900,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[오리지널 한식] 닭도리탕 닭 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,25900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[찹쌀누룽지] 닭도리탕 닭 + 찹쌀누룽지 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 /공기밥 별도,27900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[트간찜] 트러플 간장찜닭 닭 + 트러플소스 + 숏파스타 + 감자뇨끼 + 납작당면 + 야채사리 / 공기밥 별도,29900,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[트간찜] 트러플 간장찜닭(1인)닭 순살(200g) + 트러플소스 + 숏파스타 + 감자뇨끼 + 납작당면 + 야채사리,18900,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[특제간장소스] 간장찜닭 닭 + 감자 + 구멍쌀떡 + 감자수제비 + 납작당면 + 신선한 야채사리 / 공기밥 별도,25900,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[특제간장소스] 간장찜닭(1인)닭 순살(200g) + 감자 + 구멍쌀떡 + 감자수제비 + 납작당면 + 신선한 야채사리,17900,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,[한우대창] 곱도리탕 닭 + 한우대창 + 감자 + 구멍쌀떡 + 감자수제비 + 야채사리 + 반찬 / 공기밥 별도,32900,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,기본 순살닭도리탕 백반정식 닭 순살(200g) + 감자 + 구멍쌀떡 + 야채사리 + 공기밥 + 반찬,13000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,꽃게폭탄 순살꽃도리탕 백반정식 닭 순살(200g) + 꽃게 + 감자 + 구멍쌀떡 + 야채사리 + 공기밥 + 반찬,17900,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,모짜치즈 순살닭도리탕 백반정식 닭 순살(200g) + 모짜렐라치즈 + 감자 + 구멍쌀떡 + 야채사리 + 공기밥 + 반찬,15000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,순두부 순살닭도리탕 백반정식 닭 순살(200g) + 순두부 + 감자 + 구멍쌀떡 + 야채사리 + 공기밥 + 반찬,14000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,찹쌀누룽지 순살닭도리탕 백반정식 닭 순살(200g) + 찹쌀누룽지+ 감자 + 구멍쌀떡 + 야채사리 + 공기밥 + 반찬,14000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R75747aad420c,계근상 왕십리점,"서울 성동구 무학봉28길 4-1, B201호 (하왕십리동)",37.5622223,127.0324799,백반/한정식,한우대창 순살곱도리탕 백반정식 닭 순살(200g) + 한우대창 + 감자 + 구멍쌀떡 + 야채사리 + 공기밥 + 반찬,18000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,곁들임 닭무침,8000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,곁들임 닭수육,8000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,계월 탁주(병/잔술),15000,표시 가격 확인,음료,막걸리,단품,음료,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,닭무침 (홍),18000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,닭수육 (한정 수량),18000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,백곰탕,11000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6728df45c4be,계월,"서울 성동구 성덕정3길 8, 1층 (성수동1가)",37.5395491,127.0476322,한식,홍곰탕,12000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,간장닭갈비,18000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,간장닭갈비덮밥,10000,추정가,밥·죽류,덮밥,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,감자전,9000,추정가,곁들임,전·부침,단품,사이드,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,강원막국수,9000,추정가,국수류,막국수,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,계화로제닭갈비(+파스타),17000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,계화치즈닭갈비,15000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,계화치즈닭갈비덮밥,11000,추정가,밥·죽류,덮밥,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,닭가슴살개장,10000,추정가,미확인,미확인,단품,미확인,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,닭가슴살곰탕,10000,추정가,국물요리,곰탕·설렁탕,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,닭개장칼국수,10000,추정가,국수류,칼국수,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,닭목살구이,13000,추정가,구이·볶음·찜,돼지고기구이,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,닭연골구이,13000,추정가,미확인,미확인,단품,미확인,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,먹태구이,17000,추정가,곁들임,곁들임,단품,사이드,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,모듬포,15000,추정가,곁들임,곁들임,모둠,사이드,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,살얼음 동치미 묵사발,3000,추정가,곁들임,반찬·무침,단품,사이드,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,소금구이닭갈비,13000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,얼큰바지락탕,18000,추정가,국물요리,국·탕,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,옛날 구수한 누룽지,5000,추정가,미확인,미확인,단품,미확인,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,옛날춘천닭갈비,14000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,쥐치포,10000,추정가,곁들임,곁들임,단품,사이드,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화닭갈비덮밥,10000,추정가,밥·죽류,덮밥,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화석쇠무뼈닭발,20000,추정가,구이·볶음·찜,직화닭발,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화석쇠소금구이닭갈비,13000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화석쇠양념닭갈비,14000,추정가,구이·볶음·찜,닭갈비,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화석쇠오돌뼈,18000,추정가,구이·볶음·찜,직화닭발,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화철판닭갈비,13000,추정가,구이·볶음·찜,볶음요리,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,직화튤립국물닭발,22000,추정가,구이·볶음·찜,직화닭발,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,진상어묵탕,17000,추정가,국물요리,국·탕,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,치킨마요덮밥,10000,추정가,밥·죽류,덮밥,단품,식사,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R9191ba122616,계화정 왕십리점,서울 성동구 왕십리로 (상세주소 미확인),37.5632639,127.0341860,,허니버터갈릭감자튀김,10000,추정가,튀김·커틀릿,감자튀김,단품,사이드,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00,11:30-다음 날 02:00\r\n'
    'R296afe74b608,고기를 품다,"서울 성동구 왕십리로 291-2, 1층 (행당동)",37.5601737,127.0359269,한식,등겹살,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,14:00-22:00,휴무\r\n'
    'R296afe74b608,고기를 품다,"서울 성동구 왕십리로 291-2, 1층 (행당동)",37.5601737,127.0359269,한식,특목살,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,14:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,갈매기,19000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,계란찜,6000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,김치말이국수,7000,표시 가격 확인,국수류,국수,단품,식사,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,김치비빔국수,7000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,껍항정,19000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,꼬릿살,15000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,돼지한상,54000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,된장찌개(공깃밥 1개 제공),6000,표시 가격 확인,국물요리,찌개류,단품,식사,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,라면,6000,표시 가격 확인,국수류,라면,단품,식사,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,오겹살,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,왕고을한상,90000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'Rb84fb6cc0c85,고깃집왕고을,왕십리로24길 10-2 1층,37.5631541,127.0341253,돼지고기 구이/찜,특목살,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,16:00-22:00,휴무\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 7mm 냉삼,6900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 꽃목살 대패,7900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 꽃삼겹 대패,8900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 불대패,5900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 삼겹대패,4900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 오리대패,6900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 와규등심대패,11900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 천겹항정,6900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,고품격 한우++꽃등심 대패,12900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,돼지 한 판 세트(500g),37900,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,된장찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,모듬 한 판 세트(1kg),69500,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,보쌈정식 (점심),12000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,얼큰 순두부찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,오리 한판 세트(500g),38500,표시 가격 확인,구이·볶음·찜,오리요리,세트,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R06932867a650,고품격대패 왕십리본점,"서울 성동구 왕십리로 293, 1층 (행당동)",37.5602829,127.0358431,한식,제육정식 (점심),10000,표시 가격 확인,구이·볶음·찜,제육볶음,세트,식사,11:50-23:00,11:50-23:00,11:50-23:00,16:00-23:00,11:50-23:00,11:50-23:00,11:50-23:00\r\n'
    'R3d9bfaaf5b8b,고향냉면,서울 성동구 독서당로 301-93 .,37.5488131,127.0223950,냉면집,냉면,9000,표시 가격 확인,국수류,냉면,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,휴무\r\n'
    'R3d9bfaaf5b8b,고향냉면,서울 성동구 독서당로 301-93 .,37.5488131,127.0223950,냉면집,보리비빔밥,8000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,휴무\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,떡만두국,8000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,부대찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,소고기 덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,순두부찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,오징어 덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,제육볶음,9000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'R20d55bdcef26,고향식당,"서울 성동구 사근동길 63, 1층 (사근동)",37.5611636,127.0478854,한식,콩국수,9000,표시 가격 확인,국수류,국수,단품,식사,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00,07:00-21:00\r\n'
    'Rb8f50c6e6064,광동수산 산오징어,서울 성동구 행당로 127-1 (행당동) 1층,37.5584512,127.0330819,횟집,알밥,8000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rb8f50c6e6064,광동수산 산오징어,서울 성동구 행당로 127-1 (행당동) 1층,37.5584512,127.0330819,횟집,회덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,가리비찜,55000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,굴국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,굴매생이탕,20000,표시 가격 확인,국물요리,국·탕,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,굴밥,10000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,굴찜 대(3~4인),60000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,굴찜 소(2~3인),48000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,굴칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,멍게,20000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,바지락칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,산낙지연포탕 대,60000,표시 가격 확인,국물요리,국·탕,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,산낙지회,25000,표시 가격 확인,초밥·회,회·사시미,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,새소금구이 소 ( 600그램),38000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,새우소금구이 대(1킬로),55000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,생굴전,25000,표시 가격 확인,곁들임,전·부침,단품,사이드,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,생굴죽,10000,표시 가격 확인,밥·죽류,죽,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,아구찜 대(4~5인),55000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,아구찜중(2~3인),45000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,전복회,20000,표시 가격 확인,초밥·회,회·사시미,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,조개찜 대,72000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,조개찜 소,60000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,해물찜 대(4~5인),72000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,해물찜 중(2~3인),60000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,해물탕 대,60000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,해물탕소,50000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,해물파전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,해삼,20000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'Rf2127b4e4919,굴과찜사랑,"서울 성동구 행당로17길 23 (행당동, 행당동 298-50 지상1층)",37.5603327,127.0329104,횟집,활전복죽,15000,표시 가격 확인,밥·죽류,죽,단품,식사,휴무,15:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,14:00-23:30,12:00-22:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,1++ 한우암소한마 600g,140000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,1++ 한우암소한마리 400g,100000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,1++ 한우암소한마리 800g,180000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우 1++ 육사시미 250g,50000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 갈비본살 150g,40000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 갈비살 150g,40000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 등심 150g,40000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 살치살 150g,50000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 안창살 150g,50000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 육회 250g,50000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 제비추리 150g,40000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 치마살 150g,40000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R5c487672fec9,권영본의 용문집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 5층 501호 (마장동)",37.5715395,127.0400171,기타,한우1++ 토시살 150g,40000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,국내산 두부 [비건 샐러드],8900,표시 가격 확인,샐러드,샐러드,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,리코타&무스 [부드러운 샐러드],10900,표시 가격 확인,샐러드,샐러드,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,새우&버섯 [저칼로리 샐러드],8900,표시 가격 확인,샐러드,샐러드,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,생아보카도&새우 [식물성 샐러드],11900,표시 가격 확인,샐러드,샐러드,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,수제 닭가슴살 [고단백 샐러드],8900,표시 가격 확인,샐러드,샐러드,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,에그&베이컨 [가성비 샐러드],7900,표시 가격 확인,샐러드,샐러드,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,오리 훈제 [풍미 가득 샐러드],10900,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,육즙가득 이베리코 포케+곡물밥50g,12900,표시 가격 확인,샐러드,포케,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,직화불고기 [불향 가득 샐러드],10900,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,치킨텐더 [바삭한 샐러드],9900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,큐브소고기 포케+곡물밥50g,13900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,패스오더 전용 100원 아메리카노,100,표시 가격 확인,음료,커피,단품,음료,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rd6e30b0ea60e,그린비,"서울 성동구 고산자로 284, 성동샤르망 1층 122호 (행당동)",37.5649233,127.0368107,기타,훈제연어 포케+곡물밥50g,13900,표시 가격 확인,샐러드,포케,단품,식사,07:30-21:00,07:30-21:00,07:30-21:00,07:30-21:00,07:30-20:30,07:00-14:00,07:00-14:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,골뱅이추가,15000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,국물닭발,18000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,닭골뱅이탕(대),49000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,닭골뱅이탕(중),38000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,수제비사리,3000,표시 가격 확인,국수류,수제비,추가,추가옵션,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,야채닭똥집볶음,16000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,오뎅추가,3000,표시 가격 확인,분식,어묵,단품,사이드,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,유동골뱅이무침,23000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,참치김치전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,칼국수사리,3000,표시 가격 확인,국수류,칼국수,추가,추가옵션,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'Rb260d9e60589,금남닭골뱅이,"서울 성동구 독서당로 303-19 1,2층",37.5486826,127.0227944,백반/한정식,한우육회,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-22:00\r\n'
    'R7eb8212c5075,금녕곰탕,서울 성동구 왕십리로20길 11 1층,37.5629016,127.0350646,국/탕/찌개류,곰쌀국수,13000,표시 가격 확인,국수류,쌀국수,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R7eb8212c5075,금녕곰탕,서울 성동구 왕십리로20길 11 1층,37.5629016,127.0350646,국/탕/찌개류,곰탕,13000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R7eb8212c5075,금녕곰탕,서울 성동구 왕십리로20길 11 1층,37.5629016,127.0350646,국/탕/찌개류,명태회무침,14000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R7eb8212c5075,금녕곰탕,서울 성동구 왕십리로20길 11 1층,37.5629016,127.0350646,국/탕/찌개류,물수육(2인),39000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R7eb8212c5075,금녕곰탕,서울 성동구 왕십리로20길 11 1층,37.5629016,127.0350646,국/탕/찌개류,물수육(3인),57000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R7eb8212c5075,금녕곰탕,서울 성동구 왕십리로20길 11 1층,37.5629016,127.0350646,국/탕/찌개류,물수육(4인),73000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 꽃 목살 (180g),22000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 돼지 껍데기 (150g),12000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 등심꽃살 (180g),24000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 삼겹살 (180g),23000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 생 갈비 (200g),26000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 생고기 모둠한판 (700g),86000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,금돈 양념 本 갈비 (250g),26000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,양념꽃게무침,15000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,유기농 솥밥,4000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,이북식 손만두국,13000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 금돈 등심꽃살 한상,27000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 금돈 本 양념 갈비 한상,29000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 금돈 생 갈비 한상,29000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 한우 깍뚝 등심 한상,52000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 한우 등심 주물럭한상,52000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 한우 손 차돌 한상,36000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,점심 특선 한우1++ 참숯불고기 한상,52000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,토속 된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,트러플 치즈 계란,12000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한돈 김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우 뚝배기불고기,15000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우 육 사시미,38000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우 육회 물회,28000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우 육회 비빔밥,16000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우1++ 깍뚝 등심 (130g),49000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우1++ 등심 주물럭 (130g),49000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우1++ 등심 참숯불고기(130g),49000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우1++ 손 차돌박이 (130g),33000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,한우물회(1인식사용),15000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,함흥식 물냉면,12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7a02437617c4,금돈옥,"서울 성동구 고산자로 211, 1층 (행당동)",37.5590664,127.0337042,한식,함흥식 비빔냉면,12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R5105d7733b8b,금호돼지집,서울 성동구 금호로11길 3,37.5546424,127.0193063,돼지고기 구이/찜,돼지껍데기,8000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R5105d7733b8b,금호돼지집,서울 성동구 금호로11길 3,37.5546424,127.0193063,돼지고기 구이/찜,제주 목살 300g,36000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R5105d7733b8b,금호돼지집,서울 성동구 금호로11길 3,37.5546424,127.0193063,돼지고기 구이/찜,제주 삼겹살 300g,36000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R5105d7733b8b,금호돼지집,서울 성동구 금호로11길 3,37.5546424,127.0193063,돼지고기 구이/찜,제주 생갈비 300g,36000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R5105d7733b8b,금호돼지집,서울 성동구 금호로11길 3,37.5546424,127.0193063,돼지고기 구이/찜,제주생고기 김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R5b97828a2f58,기연각,"서울 성동구 용답중앙길 76, 1층 (용답동)",37.5630027,127.0536644,중국식,계란후라이,1500,표시 가격 확인,곁들임,곁들임,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R5b97828a2f58,기연각,"서울 성동구 용답중앙길 76, 1층 (용답동)",37.5630027,127.0536644,중국식,덴뿌라(고기튀김),23000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,가츠동(돈까스덮밥),8500,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,가츠소고기카레라이스,9500,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,고로케카레라이스,9000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,닭가슴살샐러드우동,10000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,데미그라스돈까스,11000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,데미그라스돈까스세트,15000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,데미그라스돈까스함박세트,13500,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,데미그라스함박스테이크,15500,표시 가격 확인,튀김·커틀릿,함박스테이크,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,로제파스타우동,10000,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,매콤카레우동,9000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,머슈룸데미그라스돈까스,13000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,명란크림우동,10000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,베이컨크림우동,11000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,불맛카레라이스,8500,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,불맛카레우동,9500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,붓카케우동,8500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,사케동,14000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,새우붓카케우동,11000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,샐러드우동,8000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,소고기매콤카레라이스,8000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,소고기우동스끼야끼,13000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,오야꼬동(치킨덮밥),8500,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,왕새우카레라이스,11000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,왕새우튀김우동,13000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,치쿠와우동,9500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,치킨카레라이스,9500,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,치킨카레우동,10000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R91de7c4464d8,긴자료코 한양대점,"서울 성동구 마조로 43, 1층 (행당동)",37.5615687,127.0413532,일식,토마토파스타우동,9000,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R97b915b12f39,길상참숯양갈비,행당로17길 19-1,37.5600730,127.0330487,돼지고기 구이/찜,가지튀김,16000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,휴무,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97b915b12f39,길상참숯양갈비,행당로17길 19-1,37.5600730,127.0330487,돼지고기 구이/찜,고급양갈비(250g),28000,표시 가격 확인,구이·볶음·찜,양고기구이,단품,식사,휴무,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97b915b12f39,길상참숯양갈비,행당로17길 19-1,37.5600730,127.0330487,돼지고기 구이/찜,마파두부,12000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97b915b12f39,길상참숯양갈비,행당로17길 19-1,37.5600730,127.0330487,돼지고기 구이/찜,양삼겹(180g),23000,표시 가격 확인,구이·볶음·찜,양고기구이,단품,식사,휴무,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97b915b12f39,길상참숯양갈비,행당로17길 19-1,37.5600730,127.0330487,돼지고기 구이/찜,항라새우,20000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,계란토마토,6000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,깐풍기,7000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,마라소고기,8000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,마라소고기탕면,7000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,볶음면,7000,표시 가격 확인,국수류,볶음면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,사천돼지고기,7000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,어향육슬,8000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,즈란양고기,8000,표시 가격 확인,구이·볶음·찜,양고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,파곱창,8000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,피망돼지고기,7000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,항라닭날개,8000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8e402a23eefa,길이식당(한양대점),"서울 성동구 사근동길 21-1, 1층 (사근동)",37.5609830,127.0433968,한식,항라육슬,8000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,계란말이김밥,4500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,고구마돈까스,10500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,군만두,5000,표시 가격 확인,분식,만두,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,김치김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,김치라면,6000,표시 가격 확인,국수류,라면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,김치우동,6500,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,꽁치김치찌개,10000,표시 가격 확인,국물요리,찌개류,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,돈까스,10000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,돈까스덮밥,11000,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,돌솥비빔밥,10000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,떡라면,6000,표시 가격 확인,국수류,라면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,뚝배기낙지불고기덮밥,11000,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,만두라면,6000,표시 가격 확인,국수류,라면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,매운오징어채김밥,4500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,멸치김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,모듬김밥,4500,표시 가격 확인,분식,김밥,모둠,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,모듬돈까스,10500,표시 가격 확인,튀김·커틀릿,돈가스,모둠,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,부대찌개,10500,표시 가격 확인,국물요리,찌개류,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,불고기덮밥,9500,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,사랑김밥,4000,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,사랑라면,5000,표시 가격 확인,국수류,라면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,사랑우동,6000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,새우볶음밥,10000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,새우튀김김밥,4500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,샐러드김밥,5500,표시 가격 확인,샐러드,샐러드,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,생선까스,10000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,소고기김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,야채비빔밥,9000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,얼큰잡채밥,9500,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,오뎅,6000,표시 가격 확인,분식,어묵,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,오뎅우동,6000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,오므라이스,10000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,오징어덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,오징어찌개,10000,표시 가격 확인,국물요리,찌개류,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,왕갈비탕,10000,표시 가격 확인,국물요리,갈비탕,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,육개장,10000,표시 가격 확인,국물요리,육개장,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,제육덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,주먹밥,3500,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,짬뽕라면,7000,표시 가격 확인,국수류,라면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,참치김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,참치덮밥,9500,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,치즈김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,치즈까스,10500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,치즈라면,6000,표시 가격 확인,국수류,라면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,치킨까스,10500,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,카레덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,카레돈까스,11000,표시 가격 확인,커리·스튜,커리,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,콩불덮밥,10500,표시 가격 확인,밥·죽류,덮밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,해물볶음우동,7500,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,햄야채볶음밥,9000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'R91728cb960db,김밥사랑 왕십리점,서울 성동구 왕십리로21길 6,37.5618119,127.0333890,김밥/만두/분식,황태해장국,9500,표시 가격 확인,국물요리,해장국,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,휴무,휴무\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,검정콩국수,9000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,들깨칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,떡만두국,9000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,바지락칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,보리밥,10000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,제육볶음,25000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,팥새알,10000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,팥칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'Rba70a5991c9f,꽁당보리밥팥죽,"서울 성동구 행당로17길 13-1, 1층 (행당동)",37.5594517,127.0332265,한식,해물파전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무,11:00-21:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,꿩냉채무침,40000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,꿩만두,11000,표시 가격 확인,분식,만두,단품,사이드,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,꿩샤브샤브1인분,35000,표시 가격 확인,국물요리,전골,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,꿩육회,40000,표시 가격 확인,초밥·회,회·사시미,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,꿩칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,꿩탕,60000,표시 가격 확인,국물요리,국·탕,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,돌오리구이 한마리,45000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,돌오리구이1인분,15000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,삼겹살,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,시골청국장,9000,표시 가격 확인,국물요리,찌개류,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,오리한방백숙,65000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R7295818ad389,꿩칼국수 행당1호점,서울 성동구 무학봉26길 9-1 1층,37.5616844,127.0330147,,오리훈제,60000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00,09:00-23:00\r\n'
    'R867b49f37d52,나그네파전,"서울 성동구 마조로1길 4 (행당동,(지상1층))",37.5583021,127.0401586,한식,해물파전,12000,표시 가격 확인,곁들임,전·부침,단품,사이드,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,[점심메뉴] 정통 닭갈비,12500,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,[점심메뉴] 치즈 찍먹 닭갈비,13500,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,[점심메뉴] 콩나물 닭갈비,12500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,정통 닭갈비,16000,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,춘천전통막국수,12000,표시 가격 확인,국수류,막국수,단품,식사,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,치즈 찍먹 닭갈비,18000,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8da937598fbf,나라구한닭 정통닭갈비(왕십리점),"서울 성동구 무학봉28길 19, 1층 (행당동)",37.5617077,127.0339597,한식,콩나물 닭갈비,17000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-23:00,11:30-23:00,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,갈릭 난,3500,표시 가격 확인,곁들임,난·빵,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,그린 샐러드,7000,표시 가격 확인,샐러드,샐러드,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,난,3000,표시 가격 확인,곁들임,난·빵,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,달 마크니,15000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,달 후라이드,15000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,램 브리야니,15000,표시 가격 확인,밥·죽류,브리야니,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,램 삭,17000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,믹스베지터블,15000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,버터 난,3500,표시 가격 확인,곁들임,난·빵,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,비프 도 피아자,17000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,비프코르마,17000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,사모사(2 Pieces),5000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,사프론 라이스,6000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,씨푸드 커리,17000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,야채 브리야니,13000,표시 가격 확인,밥·죽류,브리야니,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,어린이 세트,8500,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,인도맥주 킹피셔,8000,표시 가격 확인,음료,맥주,단품,음료,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치즈 난,6000,표시 가격 확인,곁들임,난·빵,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치킨 도 피아자,16000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치킨 마크니,16000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치킨 브리야니,14000,표시 가격 확인,밥·죽류,브리야니,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치킨 코르마,16000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치킨 탕그리,18000,표시 가격 확인,구이·볶음·찜,탄두리구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,치킨 티카 마살라,16000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,콘크림 스프,5000,표시 가격 확인,국물요리,수프,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,킹프론 탄두리,33000,표시 가격 확인,구이·볶음·찜,탄두리구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,탄두리 램,50000,표시 가격 확인,구이·볶음·찜,탄두리구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,탄두리 믹스 그릴,50000,표시 가격 확인,구이·볶음·찜,탄두리구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,탄두리 치킨 반마리(2pieces),14000,표시 가격 확인,구이·볶음·찜,탄두리구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,탄두리 치킨 한마리(4pieces),24000,표시 가격 확인,구이·볶음·찜,탄두리구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,토마토 스프,5000,표시 가격 확인,국물요리,수프,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,팔락 파니르,15000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,프론 빈달루,17000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,프론 칠리,17000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,A코스 Dinner Special,32000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,A코스 LUNCH SET,18000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,B코스 Dinner Special,37000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,B코스 LUNCH SET,23000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,C코스 Dinner Special,70000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1bb14dce7178,나마스떼 왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 지하1층 (행당동)",37.5612890,127.0383818,인도음식,C코스 LUNCH SET,30000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R89a4205a66a9,나으리,"서울 성동구 용답29길 15 (용답동,(지상1층))",37.5640255,127.0544767,한식,무뼈닭발 1인분,15000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R89a4205a66a9,나으리,"서울 성동구 용답29길 15 (용답동,(지상1층))",37.5640255,127.0544767,한식,콘치즈 추가,2000,표시 가격 확인,곁들임,치즈,추가,추가옵션,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R0f5fb2e6cc16,나으리 숯불닭발,서울 성동구 용답29길 15 1층,37.5640255,127.0544767,백반/한정식,무뼈닭발 1인분,15000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R0f5fb2e6cc16,나으리 숯불닭발,서울 성동구 용답29길 15 1층,37.5640255,127.0544767,백반/한정식,콘치즈 추가,2000,표시 가격 확인,곁들임,치즈,추가,추가옵션,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,매화수,6000,표시 가격 확인,음료,전통주,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,복분자,15000,표시 가격 확인,음료,전통주,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,볶음밥,2000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,주꾸미(350g),15000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,주먹밥,2000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,참이슬,5000,표시 가격 확인,음료,소주,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,청하,6000,표시 가격 확인,음료,전통주,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R53333ef27e0e,나정순할매쭈꾸미,서울 동대문구 무학로 144,37.5770819,127.0306982,주꾸미요리,하이트,4000,표시 가격 확인,음료,맥주,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R65c0a91c5cab,난망베이킹스튜디오,서울 동대문구 사가정로 41-1 (답십리동) 3층,37.5729777,127.0515625,카페,대만카스테라,3500,표시 가격 확인,디저트,케이크,단품,디저트,12:00-19:00,12:00-19:00,12:00-19:00,12:00-19:00,12:00-19:00,12:00-18:00,휴무\r\n'
    'R65c0a91c5cab,난망베이킹스튜디오,서울 동대문구 사가정로 41-1 (답십리동) 3층,37.5729777,127.0515625,카페,대만펑리수,3200,표시 가격 확인,미확인,미확인,단품,미확인,12:00-19:00,12:00-19:00,12:00-19:00,12:00-19:00,12:00-19:00,12:00-18:00,휴무\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,강된장쌈밥,12000,표시 가격 확인,밥·죽류,쌈밥,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,곰국,16000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,대구전,26000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,대방어 묵은지말이,24000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,대방어 회국수,16000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,돌문어간장국수,14000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,문어숙회,28000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,새우감자전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,전복들깨국수,16000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,투뿔육회 묵은지말이,22000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,한우보양전골,58000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,한우수육,38000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7e351a72062d,난포,서울 성동구 서울숲4길 18-8 1층,37.5470517,127.0426738,백반/한정식,한우육전,38000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,50년전통 남대문 고등어조림 공기밥+밑반찬+미역국,14000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,계란찜,4000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,계란후라이(1개),1000,표시 가격 확인,곁들임,계란,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,계란후라이(5개),5000,표시 가격 확인,곁들임,계란,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,낙지볶음(1인분)낙지220g+공기밥+밑반찬+직접담근김치 +소고기미역국,12000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,남대문 갈치골목 국내산 갈치조림 공기밥+밑반찬+미역국,14000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,사이다,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,오징어볶음(1인분)공기밥+밑반찬+직접담근김치 +소고기미역국,12000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,직화고등어 자반구이 공기밥+밑반찬+직접담근김치+소고기미역국,14000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,직화모듬생선구이(2종)밑반찬+미역국,14000,표시 가격 확인,구이·볶음·찜,생선구이,모둠,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,직화모듬생선구이(4종)밑반찬+미역국,31000,표시 가격 확인,구이·볶음·찜,생선구이,모둠,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,콜라,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R47c9835289e3,남대문 우리식당,"서울 성동구 왕십리로24길 12, 1층 (도선동)",37.5635325,127.0342952,백반/한정식,환타오렌지,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,녹두전,13000,표시 가격 확인,곁들임,전·부침,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,도가니,20000,표시 가격 확인,미확인,미확인,단품,미확인,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,메밀전병,8000,표시 가격 확인,곁들임,전·부침,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,모듬머리,34000,표시 가격 확인,구이·볶음·찜,보쌈·수육,모둠,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,보쌈 대,42000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,보쌈- 중,38000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,불족발,38000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,수육국밥,10000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,순대 대,16000,표시 가격 확인,분식,순대,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,순대 중,13000,표시 가격 확인,분식,순대,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,순대국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,술국- 대,38000,표시 가격 확인,국물요리,국밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,술국- 중,34000,표시 가격 확인,국물요리,국밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,옹심이만두,7000,표시 가격 확인,분식,만두,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rd20c83d67a06,남도순대국,"서울 성동구 마조로 9, 1층 (행당동)",37.5587191,127.0405680,한식,족발냉채,38000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,가지튀김볶음,14000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,고급양갈비,22000,표시 가격 확인,구이·볶음·찜,양고기구이,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,꿔바로우,17000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,마파두부,13000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,볶음밥,6000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,양꼬치,17000,표시 가격 확인,구이·볶음·찜,양고기구이,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'Rb01d0632d7cc,남매양꼬치,행당로17길 26 1층,37.5607025,127.0330774,중국집,온면,6000,표시 가격 확인,국수류,국수,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,휴무\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,감자전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,계란말이,13000,표시 가격 확인,곁들임,계란,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,고추전,23000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,김치전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,김치찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,깻잎전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,능계탕,18000,표시 가격 확인,국물요리,국·탕,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,동그랑땡,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,동태전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,두부김치,20000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,모둠전 6종,30000,표시 가격 확인,곁들임,전·부침,모둠,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,뼈없는닭발,20000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,뿔소라무침,30000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,새송이전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,새우전,25000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,생오겹살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,염소수육,38000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,염소탕,15000,표시 가격 확인,국물요리,국·탕,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,영양밥 추가,10000,표시 가격 확인,밥·죽류,흰밥,추가,추가옵션,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,옻닭,70000,표시 가격 확인,구이·볶음·찜,삼계탕·백숙,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,육전,25000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,제육볶음,20000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,한방능이백숙,80000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,해물파전,25000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R7611ba38a4bd,넝쿨마당행당,왕십리로21길 17 지하1층,37.5608930,127.0328453,한식,홍어무침,30000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,국산바다장어(100g),10000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,국산숯불산꼼장어(1인분),35000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,국산숯불산꼼장어스페셜 1-2인분,60000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,국산숯불산꼼장어스페셜 3-4인분,120000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,산꼼장어 샤브샤브(사전예약필수),85000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,산꼼장어 수육,35000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,산꼼장어 오마카세,65000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,소갈비살(150g),19000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,수제꼼장어군만두,12000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,와다,10000,표시 가격 확인,국물요리,전골,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,장어탕(전복),20000,표시 가격 확인,국물요리,국·탕,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,전복 버터구이,40000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R42007435361f,높은집종가산꼼장어,왕십리로21길 17,37.5608930,127.0328453,백반/한정식,전복라면,10000,표시 가격 확인,국수류,라면,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,구워먹는치즈,10000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,김치찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,꿀추가시 2개,500,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,날치알계란찜,5000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,냉동삼겹살 국내산 암돼지 최상급 180g,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,냉동삼겹살(180g),16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,도가니탕,15000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,된장술밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,된장찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,뚝배기불고기,11000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,물냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,비빔냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,셀프볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,소내장탕,13000,표시 가격 확인,국물요리,곱창탕,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,양념돼지껍데기,11000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,얼큰계란탕,5000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'R131481cb1299,능한이네,서울 성동구 왕십리로22길 10-1,37.5629016,127.0348722,돼지고기 구이/찜,해물순두부,9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00,11:00-다음 날 05:00\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,(V) 두부 버섯 참기름 샐러드,14000,표시 가격 확인,샐러드,샐러드,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,갈비찜과 밤퓨레,38000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,고사리 육개 볶음 국수,15000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,궁중 라구 떡볶이,19000,표시 가격 확인,분식,떡볶이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,들기름 성게 메밀 국수,21000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,루꼴라 오징어 김치전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,막걸리 아이스크림,7000,표시 가격 확인,디저트,아이스크림,단품,디저트,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,매생이 굴 라비올리,16000,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,매운 등갈비 찜 반상,23000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,묵은지 감태김밥,15000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,보리새우 미나리 전,15000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,붉은 원숭이 하이볼,13000,표시 가격 확인,음료,하이볼,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,산토리 생맥주,10000,표시 가격 확인,음료,맥주,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,서울밤 하이볼,11000,표시 가격 확인,음료,하이볼,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,스텔라 아르투아,10000,표시 가격 확인,음료,맥주,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,쌀 식혜,5000,표시 가격 확인,음료,전통음료,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,오렌지 에이드,6000,표시 가격 확인,음료,기타 음료,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,오미자 에이드,6000,표시 가격 확인,음료,에이드,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,제철 해산물 잣즙 냉채,23000,표시 가격 확인,샐러드,샐러드,단품,사이드,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,청귤 에이드,6000,표시 가격 확인,음료,에이드,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,크리스피 제육,33000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,트러플 피순대 구이,14000,표시 가격 확인,분식,순대,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,한방 찜닭 반상,23000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,한우 육회 김부각 타르트,15000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Rb2d6ade6384e,다반,"서울 성동구 서울숲4길 18-10, 1층 (성수동1가)",37.5469171,127.0426852,한식,호박식혜,5000,표시 가격 확인,음료,전통음료,단품,음료,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra320c87becb2,달님이네,서울 성동구 아차산로13길 31 (성수동2가) 1층,37.5458980,127.0615132,한식,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,06:30-17:00,06:30-17:00,06:30-17:00,06:30-17:00,06:30-17:00,휴무,휴무\r\n'
    'R509dbf59d031,담솥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,가지솥밥,12000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R509dbf59d031,담솥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,소고기숙주솥밥,13000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R509dbf59d031,담솥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,스테이크솥밥,18000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,냉면,9000,표시 가격 확인,국수류,냉면,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,떡국,10000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,떡만두국,10000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,만두국,10000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,버섯만두전골,15000,표시 가격 확인,분식,만두,단품,사이드,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,보쌈 대,45000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,보쌈 중,35000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,비빔국수,10000,표시 가격 확인,국수류,국수,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,비빔밥,10000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,생등심,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,수제돈까스,10000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,왕만두,9000,표시 가격 확인,분식,만두,단품,사이드,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,쭈꾸미,14000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,칼만두,10000,표시 가격 확인,국수류,칼국수,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,콩국수,11000,표시 가격 확인,국수류,국수,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Re0532fbccb52,담쟁이,"서울 성동구 왕십리로20길 11-1 (도선동,(1층))",37.5629926,127.0351482,한식,해물전,14000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,"50년대, 이북식만두국",12000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,"50년대,이북식만두반판",6000,표시 가격 확인,분식,만두,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,"50년대,이북식만두한판",12000,표시 가격 확인,분식,만두,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,"그시절, 이북식 전통냉면(물)",12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,"그시절, 이북식 전통냉면(비빔)",12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,대근 진국 갈비탕,18000,표시 가격 확인,국물요리,갈비탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,대근 특제 소갈비찜,67000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,돼지불고기비빔밥,10000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,마장동한우 육회냉면,15000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,마장동한우육회비빔밥,15000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,매콤불향 오징어볶음,26000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,명인선정 가브리보쌈,30000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,명인선정 가브리보쌈 (맛보기),8000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,바삭 파닭전,18000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,불향 오징어볶음비빔밥,11000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,불향가득 돼지불고기반판,8000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,불향가득 돼지불고기한판,15000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'Reb433333ada1,대근면옥,서울 성동구 행당로15길 1 1층,37.5582489,127.0316811,냉면,이북식 손만두전골,28000,표시 가격 확인,분식,만두,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,(한우)대창덮밥,12000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,갈비덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,곱도리탕,29000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,날치알 주먹밥,3000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,닭목살 간장 덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,닭목살 고추장 덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,대낙덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,대낙새전골,29000,표시 가격 확인,국물요리,전골,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,삼낙덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R824b622f0b67,대낚식당,"서울 성동구 연무장길 39-21, 1층 (성수동2가)",37.5434201,127.0539425,한식,항정살덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,대도 깍두기볶음밥,6000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,대도 한우곰탕(1Pack),5000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,대도 한우대패등심(120g),28000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,대도 한우육포(1Pack),8000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,대도 한우육회(150g),29000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,대도삼미(三味)모둠(450g),176000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,된장죽,6000,표시 가격 확인,밥·죽류,죽,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,부채살,62000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,업진살,62000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,치마살,62000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,파묵은지(60g) / *곁들임추천메뉴,2000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8acad7ca2fb9,대도식당 왕십리 본점,서울 성동구 무학로12길 3 1층,37.5673144,127.0306405,한식,한우생등심 오리지널컷(170g),49000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,갈비살,33000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,냉면,5000,표시 가격 확인,국수류,냉면,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,돼지갈비,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,등심,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,목살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,백반,10000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,삼겹살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,소갈비,40000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R4a67df6968d9,대성갈비,"서울 성동구 상원1길 26, 서울숲A타워 1층 101호,102호,103호 (성수동1가)",37.5461813,127.0486507,한식,차돌박이,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,가브리 덮밥,12000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,가브리 무침,9500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,가브리살 수육,11000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,감당부침개,7000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,감자만두,3000,표시 가격 확인,분식,만두,단품,사이드,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,남도떡갈비,7000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,더진국수육국밥,10000,표시 가격 확인,국수류,국수,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,맑은수육곰탕,11000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,머릿고기편육,9000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,명태회냉면,11000,표시 가격 확인,국수류,냉면,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,물냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,부대국밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,부산식수육국밥,11000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,비빔냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,뼈해장국,12000,표시 가격 확인,국물요리,해장국,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,사골해장라면,10000,표시 가격 확인,국수류,라면,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,소곱창전골,27000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,순대국,11000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,순대해장라면,9000,표시 가격 확인,국수류,라면,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,얼큰순대국,12000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,얼큰순대해장라면,10000,표시 가격 확인,국수류,라면,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,얼큰술국,25000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,전통 순대국,12000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,철판소곱창볶음,27000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,철판순대볶음,23000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R8b2ce4a58810,더진국 한양대점,"서울 성동구 마조로 7, 지상1층 (행당동)",37.5585930,127.0404404,한식,토종순대,8000,표시 가격 확인,분식,순대,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-21:00\r\n'
    'R080a6e132572,더킹돈 뚝섬점,서울 성동구 상원길 46,37.5481205,127.0484900,경양식,더킹 1인 돈까스,14000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R080a6e132572,더킹돈 뚝섬점,서울 성동구 상원길 46,37.5481205,127.0484900,경양식,더킹 2인 돈까스,19000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R080a6e132572,더킹돈 뚝섬점,서울 성동구 상원길 46,37.5481205,127.0484900,경양식,등심돈까스,11000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R080a6e132572,더킹돈 뚝섬점,서울 성동구 상원길 46,37.5481205,127.0484900,경양식,생선까스,12000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R080a6e132572,더킹돈 뚝섬점,서울 성동구 상원길 46,37.5481205,127.0484900,경양식,안심돈까스,14000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R080a6e132572,더킹돈 뚝섬점,서울 성동구 상원길 46,37.5481205,127.0484900,경양식,함박스테이크,11000,표시 가격 확인,튀김·커틀릿,함박스테이크,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R8493195cde75,동해루,서울 성동구 고산자로8길 26 .,37.5581438,127.0366469,중국식,자장면,6000,표시 가격 확인,국수류,짜장면,단품,식사,휴무,11:20-20:30,11:20-20:30,11:20-20:30,11:20-20:30,11:20-20:30,11:20-14:30\r\n'
    'R8493195cde75,동해루,서울 성동구 고산자로8길 26 .,37.5581438,127.0366469,중국식,짬뽕,7000,표시 가격 확인,국수류,짬뽕,단품,식사,휴무,11:20-20:30,11:20-20:30,11:20-20:30,11:20-20:30,11:20-20:30,11:20-14:30\r\n'
    'Rf7cfabaa509b,동해안백골뱅이,무학로2길 48,37.5623881,127.0346909,백반/한정식,골뱅이 무침(중),50000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00\r\n'
    'Rf7cfabaa509b,동해안백골뱅이,무학로2길 48,37.5623881,127.0346909,백반/한정식,과메기(해초),40000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00\r\n'
    'Rf7cfabaa509b,동해안백골뱅이,무학로2길 48,37.5623881,127.0346909,백반/한정식,백골뱅이(중),45000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00\r\n'
    'Rf7cfabaa509b,동해안백골뱅이,무학로2길 48,37.5623881,127.0346909,백반/한정식,백합칼국수,15000,표시 가격 확인,국수류,칼국수,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00\r\n'
    'Rf7cfabaa509b,동해안백골뱅이,무학로2길 48,37.5623881,127.0346909,백반/한정식,백합탕,35000,표시 가격 확인,국물요리,국·탕,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00\r\n'
    'Rf7cfabaa509b,동해안백골뱅이,무학로2길 48,37.5623881,127.0346909,백반/한정식,참문어숙회,50000,표시 가격 확인,초밥·회,회·사시미,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,갈비소바,13800,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,계란찜,3000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,냉면+제육,12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,돼지양념갈비,19000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,돼지양념구이,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,묵은지비빔면,7000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,물냉면,9000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,비빔냉면,9000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,삼겹살,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,소갈비살,18000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,양푼김치제육,10000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,육회,20000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,육회비빔밥,11000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,해물된장술밥,9000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'Rde3a450bcd80,돼지갈비 돼갈공명소바 답십리역 본점,서울 동대문구 고미술로 12 1층,37.5698236,127.0501753,,해물된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,17:00-22:00,17:00-22:00\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,가브리살(180g)등심에 덧붙어 있어 등심덧살이라고 불리며 한 마리당 200g에서 400g 정도 밖에 나오지 않는 고급 특수부위입니다.,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,계란찜,5000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,고수무침 고수를 드실 수 있는 분들은 주문하시면 후회하지 않으십니다.,8000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,김치찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,냉모밀,8000,표시 가격 확인,국수류,모밀·소바,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,냉제육(100g)앞다리살과 오겹의 살코기 부분을 수육처럼 삶아 차갑게 내어 특제 간장에 담군 파를 올려 먹는 메뉴입니다.,13000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,돼지껍데기(130g)돼지미식방에서 직접 만든 양념에 이틀간 숙성한 돼지껍데기 입니다.,10000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,라면,5000,표시 가격 확인,국수류,라면,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,"막고기주물럭(180g)막고기(오겹, 목살, 앞다리)를 주문과 동시에 간장베이스의 특제 소스에 주물러 나가는 양념주물럭입니다.",17000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,목살(180g)최고급 브랜드의 원육을 제공하고 있으며 드셔보시면 부드러움에서 다른 곳의 목살과 확연한 차이를 느끼실 수 있습니다.,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,알밥,8000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R617902fcb9ab,돼지미식방,"서울 성동구 왕십리로24나길 14, 1층 (도선동)",37.5639468,127.0349972,돼지고기 구이/찜,우리흑돈w 오겹살(180g)하이포크에서 우리나라 고유의 재래 흑돼지의 유전자를 교잡하여 만든 새로운 종으로 진한 육향과 탱탱한 식감이 특징입니다.,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,1인분 한돈 묵은 김치찜,17900,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,날치알 주먹밥,4900,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,두부김치찜 국산 돼지고기 500g,29900,표시 가격 확인,곁들임,반찬·무침,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,명인 무뼈국물닭발 세트,28900,표시 가격 확인,구이·볶음·찜,직화닭발,세트,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,명인 숯불무뼈닭발 세트,27900,표시 가격 확인,구이·볶음·찜,직화닭발,세트,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,명인 황금 계란찜 (황금레시피),4000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,모짜렐라치즈를 넣은 명품계란찜(강추),6000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,순두부김치찜 국산돼지고기 500g,29900,표시 가격 확인,국물요리,찌개류,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,참치 마요 날치알 주먹밥 high quality,7500,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,칠성사이다 500ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,칠성사이다 제로 500ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,쿨피스 450ml,1500,표시 가격 확인,음료,기타 음료,단품,음료,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,펩시 500ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,펩시 제로 라임 500ml,2500,표시 가격 확인,음료,탄산음료,단품,음료,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,프리미엄 김치찜 국산돼지고기 500g,27900,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R99ebca4c991e,두번먹는명인김치찜 청계천점,서울 동대문구 무학로20길 20 2층,37.5710436,127.0317314,,프리미엄 양폭탄 돼지김치찜 (한돈500g) 단품,25500,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,휴무\r\n'
    'R30c454fd94a7,들창코 본점,"서울 성동구 마장로 326, 지상1층 (마장동)",37.5674323,127.0454337,한식,Green Tea Powdered Samgyeopsal,12000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R30c454fd94a7,들창코 본점,"서울 성동구 마장로 326, 지상1층 (마장동)",37.5674323,127.0454337,한식,Hub Samgyeopsal,12000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R30c454fd94a7,들창코 본점,"서울 성동구 마장로 326, 지상1층 (마장동)",37.5674323,127.0454337,한식,Mok Samgyeopsal,12000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00,16:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,갈매기살 160g,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,계란찜,6000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,목살 200g,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,물(비)냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,삼겹살 200g,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,전투라면,8000,표시 가격 확인,국수류,라면,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R565ed4ea5327,땅코 참숯구이,"서울 성동구 행당로17길 26, 1층 (행당동)",37.5607025,127.0330774,한식,차돌된장술밥,8000,표시 가격 확인,국물요리,국밥,단품,식사,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:30-23:00,14:00-23:00,14:00-23:00\r\n'
    'R1847e8a4354e,또와,서울 성동구 성수일로12길 20 (성수동2가) 1층 101호,37.5482346,127.0533021,한식,삼겹살,15000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무,휴무\r\n'
    'R1847e8a4354e,또와,서울 성동구 성수일로12길 20 (성수동2가) 1층 101호,37.5482346,127.0533021,한식,점심뷔페,7000,표시 가격 확인,미확인,미확인,단품,미확인,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무,휴무\r\n'
    'R17ddc1a0f9d3,또와분식,서울 성동구 마장로35나길 40 (마장동) 101호,37.5671876,127.0432164,분식,떡볶이,5000,표시 가격 확인,분식,떡볶이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00\r\n'
    'R12bf757c5a4b,라도무쇠곱창,"서울 성동구 용답19길 3 (용답동,지상1층(용답길 74))",37.5627286,127.0514844,한식,물냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,휴무,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00\r\n'
    'R12bf757c5a4b,라도무쇠곱창,"서울 성동구 용답19길 3 (용답동,지상1층(용답길 74))",37.5627286,127.0514844,한식,볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,휴무,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00\r\n'
    'R12bf757c5a4b,라도무쇠곱창,"서울 성동구 용답19길 3 (용답동,지상1층(용답길 74))",37.5627286,127.0514844,한식,야채곱창,12000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,휴무,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00\r\n'
    'R12bf757c5a4b,라도무쇠곱창,"서울 성동구 용답19길 3 (용답동,지상1층(용답길 74))",37.5627286,127.0514844,한식,얼큰곱창전골,29000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,휴무,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00,15:30-다음 날 01:00\r\n'
    'R57069304ab89,라밥 왕십리역점,왕십리광장로 6,37.5609417,127.0360735,기타,,6500,대표가 추정(메뉴 미등록),,,,,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'R981f1efbab79,란칼국수,서울 성동구 성수일로8길 42 (성수동2가) 1층 104호,37.5453515,127.0550514,한식,칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,09:00-21:00,휴무\r\n'
    'Rb3f9373d19b6,려기도삭면,왕십리로21가길 1 1층,37.5610470,127.0330384,중국집,도삭면,8000,표시 가격 확인,국수류,도삭면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R0b6550c05df1,로데오 곱창,"서울 성동구 용답19길 8 (용답동,(지상1층))",37.5628995,127.0520543,한식,알곱창,13000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,휴무\r\n'
    'R0b6550c05df1,로데오 곱창,"서울 성동구 용답19길 8 (용답동,(지상1층))",37.5628995,127.0520543,한식,야채곱창,12000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,16:00-다음 날 01:00,휴무\r\n'
    'R7b316c34c54a,롤링파스타 한양대점,서울 성동구 마조로5길 8 (행당동),37.5601495,127.0402288,한식,로제크림 쉬림프 파스타,9500,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R7b316c34c54a,롤링파스타 한양대점,서울 성동구 마조로5길 8 (행당동),37.5601495,127.0402288,한식,베이컨크림리조또,9500,표시 가격 확인,밥·죽류,리조또,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R7b316c34c54a,롤링파스타 한양대점,서울 성동구 마조로5길 8 (행당동),37.5601495,127.0402288,한식,알리오 올리오,7900,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,말차 수플레,15500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,애니타임 브런치 수플레,19500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,얼그레이 수플레,15500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,제철과일 수플레,19500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,초당옥수수 수플레,16500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,초코 수플레,16500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,플레인 수플레,13500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,허니 수플레,15500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rec4c4e307e1d,루플레,"서울 성동구 마조로3길 3, 1층 (행당동)",37.5584514,127.0401401,한식,흑임자 수플레,16500,표시 가격 확인,디저트,수플레,단품,디저트,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,(계절)중국식냉면,16000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,고추잡채+꽃빵(4ps),32000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,고추잡채밥,15000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,깐풍기,28000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,동파육,50000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,마파연두부밥,12000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,멘보샤,32000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,블랙빈모둠해물요리,40000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,사천탕수육,26000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,삼선누룽지탕,48000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,양장피,28000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,어향동고,45000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,오향장육,40000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,유린기,28000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,유산슬,32000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,유산슬밥,17000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,자연송이소고기안심,58000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,전가복,58000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,중새우요리(칠리or크림or깐풍),32000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,짬뽕,12000,표시 가격 확인,국수류,짬뽕,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,차돌짬뽕,15000,표시 가격 확인,국수류,짬뽕,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,특선간짜장,15000,표시 가격 확인,국수류,짜장면,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,팔보채,38000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re55a910771b8,리송,"서울 성동구 왕십리광장로 17, 왕십리민자역사 3층 RE-03호 (행당동)",37.5612890,127.0383818,중국식,해물잡채밥,14000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,가마버터우동,10000,표시 가격 확인,국수류,우동,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,감자고로케 2pcs,3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,냉모밀,9500,표시 가격 확인,국수류,모밀·소바,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,네기치킨가라아게동,9900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,로스카츠(등심),12000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,붓카케우동,9500,표시 가격 확인,국수류,우동,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,아지타마고(계란장) 1ea,1500,표시 가격 확인,곁들임,계란,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,온우동,9000,표시 가격 확인,국수류,우동,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,왕새우튀김 1pcs,3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,차슈,2000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,츠케멘,9500,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,카레,3000,표시 가격 확인,커리·스튜,커리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,카츠브리오슈(카츠산도),8000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,파밥,3000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Re10ef38d4880,마메,"서울 성동구 마조로 27, 1층 101호 (행당동)",37.5602473,127.0409245,일식,히레카츠(안심),14000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,감자전&김치전 2P,7000,표시 가격 확인,곁들임,전·부침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,감자전&김치전 3P,10000,표시 가격 확인,곁들임,전·부침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,기본쌈세트,1000,표시 가격 확인,곁들임,쌈채소,세트,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕 통구이(대),47000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕 통구이(소),36000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕 통구이(중),42000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕떡볶이,5000,표시 가격 확인,분식,떡볶이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕통구이+쭈꾸미 세트(대),58000,표시 가격 확인,구이·볶음·찜,족발,세트,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕통구이+쭈꾸미 세트(소),47000,표시 가격 확인,구이·볶음·찜,족발,세트,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕통구이+쭈꾸미 세트(중),53000,표시 가격 확인,구이·볶음·찜,족발,세트,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,마왕특제소스(불족소스),1000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,미니족구이(소),29000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,보쌈구이(소),36000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,보쌈구이(중),42000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,보쌈구이+쭈꾸미 세트(소),47000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,보쌈구이+쭈꾸미 세트(중),53000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,사이다(1.25L),3000,표시 가격 확인,음료,탄산음료,단품,음료,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,사이다(500ml),2000,표시 가격 확인,음료,탄산음료,단품,음료,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,쭈꾸미볶음,12000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,천사세트(대),51000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,천사세트(중),47000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,콜라(1.25L),3000,표시 가격 확인,음료,탄산음료,단품,음료,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,콜라(500ml),2000,표시 가격 확인,음료,탄산음료,단품,음료,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,핫 블랙소스,1000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R647f51427e5e,마왕족발 왕십리점,행당로17길 23-1,37.5604335,127.0328451,족발/보쌈,핫마요소스,1000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,12:00-다음 날 01:00,12:00-다음 날 01:00\r\n'
    'R9345fcf52b26,마장동 할머니 갈비탕,"서울 성동구 청계천로10가길 10-9, 1층 (마장동)",37.5697443,127.0334243,한식,갈비탕,14000,표시 가격 확인,국물요리,갈비탕,단품,식사,10:00-14:00,10:00-14:00,10:00-14:00,10:00-14:00,10:00-14:00,10:00-14:00,휴무\r\n'
    'R52e9cc8ec7b9,마포갈매기,서울 성동구 용답29길 26 (용답동) 1층,37.5648498,127.0536509,한식,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,휴무\r\n'
    'R52e9cc8ec7b9,마포갈매기,서울 성동구 용답29길 26 (용답동) 1층,37.5648498,127.0536509,한식,비빔밥,8000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,계란말이백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,계란찜백반,8000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,굴비정식,11000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,김치볶음밥,8000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,낙지덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,낙지볶음(2인),24000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,내장탕,9000,표시 가격 확인,국물요리,곱창탕,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,닭도리탕 (대),35000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,닭도리탕 (소),30000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,돌솥비빔밥,8000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,동태찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,두부김치,11000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,떡국 (계절메뉴),9000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,떡라면,6000,표시 가격 확인,국수류,라면,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,떡만두국 (계절메뉴),9000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,라면,5000,표시 가격 확인,국수류,라면,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,만두국 (계절메뉴),9000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,만두라면,6000,표시 가격 확인,국수류,라면,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,물냉면 (계절메뉴),9000,표시 가격 확인,국수류,냉면,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,부대찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,비빔국수,8000,표시 가격 확인,국수류,국수,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,비빔냉면 (계절메뉴),9000,표시 가격 확인,국수류,냉면,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,비빔밥,8000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,삼겹살(1인),17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,서리태냉콩국수 (계절메뉴),11000,표시 가격 확인,국수류,국수,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,수제군만두(8개),5000,표시 가격 확인,분식,만두,단품,사이드,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,순두부찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,열무국수(계절메뉴),9000,표시 가격 확인,국물요리,국·탕,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,열무냉면 (계절메뉴),9000,표시 가격 확인,국수류,냉면,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,오삼볶음(2인),22000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,오징어덮밥,8000,표시 가격 확인,밥·죽류,덮밥,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,오징어볶음,11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,육개장,8000,표시 가격 확인,국물요리,육개장,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,잔치국수,7000,표시 가격 확인,국수류,국수,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,제육덮밥,8000,표시 가격 확인,밥·죽류,덮밥,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,제육볶음,11000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,쭈꾸미볶음,11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,쭈삼볶음(2인),22000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,청국장,8000,표시 가격 확인,국물요리,찌개류,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rce49a84865f9,만나식당,"서울 성동구 왕십리로21길 20, 1층 (행당동)",37.5607783,127.0324293,분식,칼국수,8000,표시 가격 확인,국수류,칼국수,단품,식사,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,07:00-22:00,09:00-22:00,휴무\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,돼지갈비 250g(국내산),16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,돼지생갈비살 250g(칠레산),16000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,돼지왕갈비 300g(캐나다산),22000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,된장찌게,5000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,"물냉면, 매콤비빔냉면",8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,소갈비살 180g(미국산),23000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,술밥된장찌게,5000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,폭탄계란찜,5000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,한돈 오겹살 180g(국내산),16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,항정살 180g(칠레산),16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'Rb459ad6dea4b,만대갈비,서울 성동구 왕십리로26길 6 (도선동),37.5636305,127.0328851,한식,"후식(물냉면, 비빔냉면)",3000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,12:00-22:30,12:00-22:30\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,(포장) 집밥같은 만두전골 (2인),22000,표시 가격 확인,분식,만두,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,(포장) 집밥같은 만두전골(3인),33000,표시 가격 확인,분식,만두,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,(포장)녹두반죽-3장 분량,14000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,(포장)만두만,16000,표시 가격 확인,분식,만두,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,(포장)아롱사태 수육만두전골(2인),30000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,(포장)아롱사태 수육만두전골(3인),43000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,녹두전(100% 녹두)1장,7000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,아롱사태 수육만두전골(2인),34000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,아롱사태 수육만두전골(3인),49000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,집밥같은 만두전골(2인),26000,표시 가격 확인,분식,만두,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4445c312b17b,만두전빵,서울 성동구 행당로15길 2-1 (행당동),37.5582970,127.0319097,한식,집밥같은 만두전골(3인),39000,표시 가격 확인,분식,만두,단품,사이드,11:30-21:00,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,간재미무침,35000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,간재미물회,35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,계란탕,12000,표시 가격 확인,국물요리,국·탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,고갈비,20000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,고추장찌개,20000,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,골뱅이무침,23000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,광어,38000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,광어 대,70000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,광어무침,41000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,광어물회,41000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,꽁치 김치찌개,23000,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,닭도리탕,25000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,돼지 김치찌개,23000,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,돼지불고기,23000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,두부김치,25000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,멍게,20000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,모듬 물회,65000,표시 가격 확인,초밥·회,회·사시미,모둠,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,반반튀김 (오징어&새우),20000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,백합탕,25000,표시 가격 확인,국물요리,국·탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,병어조림,50000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,뻔데기볶음,10000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,뻔데기탕,8000,표시 가격 확인,국물요리,국·탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,산낙지,30000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,산낙지물회,35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,새우튀김,20000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,서더리 매운탕,15000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,알탕,25000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,어묵탕,20000,표시 가격 확인,국물요리,국·탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,오삼불고기,35000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,오징어데침 무침,25000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,오징어볶음,25000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,오징어튀김,20000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,우럭,38000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,우럭무침,41000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,이면수+ 조기 구이,19000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,이면수구이,19000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,전복 버터구이,30000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,전복회,30000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,제육볶음,23000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,조기구이,19000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,참치 김치찌개,20000,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,통매운탕,25000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,통우럭구이,40000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,해물모듬,65000,표시 가격 확인,미확인,미확인,모둠,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R21c0d859f585,먹거리장터2,서울 성동구 행당로17길 16 (행당동),37.5598581,127.0335017,한식,해삼,25000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,휴무\r\n'
    'R58c25a1b616c,먹어봐 뒷고기,"서울 성동구 무학봉28길 21, 경남빌딩 1층 (행당동)",37.5615915,127.0341305,한식,계란찜,7000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,휴무\r\n'
    'R58c25a1b616c,먹어봐 뒷고기,"서울 성동구 무학봉28길 21, 경남빌딩 1층 (행당동)",37.5615915,127.0341305,한식,뒷고기,15000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,휴무\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,[재철별미] 신선한 제주 딱새우회,35900,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,1+보름달 차돌 사시미,31900,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,고기치즈떡볶이,19900,표시 가격 확인,분식,떡볶이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,나고야식미소모츠나베,24900,표시 가격 확인,국물요리,전골,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,단새우플래터,31900,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,단짠홀릭 요술 먹태,16900,표시 가격 확인,곁들임,곁들임,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,달콤쫀득수제마스카포네치즈곶감,12900,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,대창알리오올리오,16900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,돈코츠오뎅나베,23900,표시 가격 확인,분식,어묵,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,메이게츠 달달데이트세트,59900,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,메이게츠 시그니처세트,42900,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,메이게츠 하이볼,6900,표시 가격 확인,음료,하이볼,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,메이게츠 회식세트,54900,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,미도리하이볼,7900,표시 가격 확인,음료,하이볼,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,불맛가득야끼소바,13900,표시 가격 확인,국수류,볶음면,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,블루하와이 하이볼,7900,표시 가격 확인,음료,하이볼,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,산토리 하이볼,5900,표시 가격 확인,음료,하이볼,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,생맥주,4500,표시 가격 확인,음료,맥주,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,수제 유린카츠,17900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,수제 치킨난반가라아게,16900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,수제 콘치즈불닭가라아게,18900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,수제 흑마늘가라아게,17900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,숙성연어한판200g,22900,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,안퀴모호소마키,15900,표시 가격 확인,초밥·회,초밥,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,오키나와식 스키야끼,24900,표시 가격 확인,국물요리,전골,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,일본현지그맛그대로레몬사와!!,5900,표시 가격 확인,음료,칵테일,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,일식 계란말이,10900,표시 가격 확인,곁들임,계란,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,자몽 하이볼,7900,표시 가격 확인,음료,하이볼,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,카라이차돌탄탄나베,25900,표시 가격 확인,국물요리,전골,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,트리플베리아이스크림,8900,표시 가격 확인,디저트,아이스크림,단품,디저트,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,특급 무제한 사케 90분간,6900,표시 가격 확인,음료,사케,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,프리미엄삿포로 밀코,8000,표시 가격 확인,음료,맥주,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,프리미엄삿포로생맥주,8000,표시 가격 확인,음료,맥주,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Rfc7639146986,메이게츠 한양대,"서울 성동구 마조로7길 4-6 (행당동, 행당동 1-98)",37.5609986,127.0409429,한식,한우대창곱도리나베,25900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 01:00\r\n'
    'Ra6077e73ace5,멘푸라,서울 성동구 마조로3길 6 (행당동),37.5585779,127.0401175,한식,감자고로케(2P),2800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,휴무,11:00-20:30\r\n'
    'Ra6077e73ace5,멘푸라,서울 성동구 마조로3길 6 (행당동),37.5585779,127.0401175,한식,마제멘,8900,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,휴무,11:00-20:30\r\n'
    'Ra6077e73ace5,멘푸라,서울 성동구 마조로3길 6 (행당동),37.5585779,127.0401175,한식,치킨푸라이,3000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,휴무,11:00-20:30\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,1인세트 돈코츠라멘 + 불맛미니부타동,12900,표시 가격 확인,국수류,일본식 라멘,세트,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,2인 세트,29800,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,가라아게 5p,5000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,가라아게 카레덮밥,10500,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,감자고로케 4p,4000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,김치미니부타동,4500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,냉소바,8900,표시 가격 확인,국수류,모밀·소바,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,닭껍질교자 4p,4000,표시 가격 확인,분식,만두,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,더블체다카츠,11900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,데미등심카츠,11900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,데미히레카츠 (half),5500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,돈코츠라멘,8900,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,들기름 소바,8900,표시 가격 확인,국수류,모밀·소바,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,마라돈코츠라멘,9900,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,마라마제소바,10000,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,마라샤브쌀국수,10500,표시 가격 확인,국수류,쌀국수,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,마제소바,9500,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,마제소밥,9500,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,매운돈코츠라멘,9400,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,매운샤브쌀국수,10000,표시 가격 확인,국수류,쌀국수,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,매운파돈카츠,11500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,매콤카레카츠,13500,표시 가격 확인,커리·스튜,커리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,모짜치즈카츠,12500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,모짜치즈카츠 2p,5900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,미니카레밥,3800,표시 가격 확인,커리·스튜,커리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,반반카츠,11900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,불맛미니부타동,4000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,비빔소바,8500,표시 가격 확인,국수류,모밀·소바,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,사누끼어묵우동,8400,표시 가격 확인,국수류,우동,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,새우튀김 2p,4000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,샤브쌀국수,9500,표시 가격 확인,국수류,쌀국수,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,쇼유라멘,8900,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,수제등심카츠,10500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,얼큰김치우동,8900,표시 가격 확인,국수류,우동,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,에비돈코츠라멘,10900,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,에비카레덮밥,10500,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,짜조 2p,4000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,카레부타동,10500,표시 가격 확인,커리·스튜,커리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,카제소바,10000,표시 가격 확인,국수류,모밀·소바,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,큰김치부타동,9900,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,큰부타동,9500,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,판소바,8900,표시 가격 확인,국수류,모밀·소바,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R2c202d6fe331,면식당 서울한양대점,"서울 성동구 마조로 30, 101호 (행당동, 미진빌딩)",37.5603131,127.0415114,일식,히레카츠 (half),4900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R79d7740c6bfe,명가,서울 성동구 무학봉16길 1 (하왕십리동) 1층,37.5613619,127.0313741,한식,소불고기,10000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,갈릭디핑소스,300,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,꿀,300,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,단짠 베이컨 하와이안 피자 (L),25900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,단짠 베이컨 하와이안 피자 (R),21900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,리얼무스 고구마 피자 (L),25900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,리얼무스 고구마 피자 (R),21900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,마늘빵 (4ea),3000,표시 가격 확인,곁들임,빵,단품,사이드,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,몬스터 쉬림프 피자 (L),29900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,몬스터 쉬림프 피자 (R),25900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,방콕 쉬림프 피자 (L),30900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,방콕 쉬림프 피자 (R),26900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,버팔로 윙 (5ea),6000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,베이컨 소시지 폭탄 피자 (L),27900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,베이컨 소시지 폭탄 피자 (R),23900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,순살 통살 치킨 (2ea),5500,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,숯불갈비 피자 (L),30900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,숯불갈비 피자 (R),26900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,스프라이트 1.5L,2500,표시 가격 확인,음료,탄산음료,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,스프라이트 500ml,1500,표시 가격 확인,음료,탄산음료,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,아삭 베지터블 피자 (L),23900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,아삭 베지터블 피자 (R),19900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,옥꾸마 피자 (L),28900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,옥꾸마 피자 (R),24900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,올인 콤비네이션 피자 (L),25900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,올인 콤비네이션 피자 (R),21900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,왕십리 불곱창 피자 (L),30900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,원탑 소불고기 피자 (L),26900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,원탑 소불고기 피자 (R),22900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,웨지 포테이토 피자 (L),25900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,웨지 포테이토 피자 (R),21900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,제로콜라 500ml,1500,표시 가격 확인,음료,탄산음료,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,청양마요소스,700,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈가루,200,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈끝판왕 피자 (L),27900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈끝판왕 피자 (R),23900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈오븐 미트 도리아,7500,표시 가격 확인,밥·죽류,리조또,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈오븐 미트 파스타,7500,표시 가격 확인,파스타류,파스타,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈오븐 크림 도리아,8000,표시 가격 확인,밥·죽류,리조또,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈오븐 크림 파스타,8000,표시 가격 확인,파스타류,파스타,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈오븐 토마토 도리아,7500,표시 가격 확인,밥·죽류,리조또,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치즈오븐 토마토 파스타,7500,표시 가격 확인,파스타류,파스타,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치킨텐더 (3ea),3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치폴레전드 피자 (L),28900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,치폴레전드 피자 (R),24900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,콜라 1.25L,2500,표시 가격 확인,음료,탄산음료,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,콜라 500ml,1500,표시 가격 확인,음료,탄산음료,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,페퍼로니 피자 (L),23900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,페퍼로니 피자 (R),19900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,피클,500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,하이네켄 500ml 캔,4000,표시 가격 확인,음료,맥주,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,하이네켄제로 330ml 캔,4000,표시 가격 확인,음료,맥주,단품,음료,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,핫소스,100,표시 가격 확인,미확인,미확인,단품,미확인,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R7f3e33bf6309,명동피자 익스프레스 왕십리점,왕십리로21길 16 1층,37.5611813,127.0328228,피자,해쉬브라운 (2ea),3500,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,휴무,15:30-22:30,15:30-22:30,15:30-22:30,15:30-22:30,11:30-22:30,11:30-22:30\r\n'
    'R6e897616e874,명성식당,서울 동대문구 무학로 142-2 (용두동) 1층,37.5769853,127.0308033,백반/한정식,백반,7000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:00,휴무\r\n'
    'R6e897616e874,명성식당,서울 동대문구 무학로 142-2 (용두동) 1층,37.5769853,127.0308033,백반/한정식,청국장,8000,표시 가격 확인,국물요리,찌개류,단품,식사,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:00,휴무\r\n'
    'R6e897616e874,명성식당,서울 동대문구 무학로 142-2 (용두동) 1층,37.5769853,127.0308033,백반/한정식,콩나물국,7000,표시 가격 확인,국물요리,국·탕,단품,식사,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:30,06:30-21:00,휴무\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,"50년대, 이북식만두국",12000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,"50년대,이북식만두반판",6000,표시 가격 확인,분식,만두,단품,사이드,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,"50년대,이북식만두한판",12000,표시 가격 확인,분식,만두,단품,사이드,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,"그시절, 이북식 전통냉면(물)",12000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,"그시절, 이북식 전통냉면(비빔)",12000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,대근 진국 갈비탕,18000,표시 가격 확인,국물요리,갈비탕,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,대근 특제 소갈비찜,67000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,돼지불고기비빔밥,10000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,마장동한우 육회냉면,15000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,마장동한우육회비빔밥,15000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,매콤불향 오징어볶음,26000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,명인선정 가브리보쌈,30000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,명인선정 가브리보쌈 (맛보기),8000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,바삭 파닭전,18000,표시 가격 확인,곁들임,전·부침,단품,사이드,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,불향 오징어볶음비빔밥,11000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,불향가득 돼지불고기반판,8000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,불향가득 돼지불고기한판,15000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R30ce59cba93a,명인김은숙 대근면옥 왕십리본점,"서울 성동구 행당로15길 1, 1층 (행당동)",37.5582489,127.0316811,한식,이북식 손만두전골,28000,표시 가격 확인,분식,만두,단품,사이드,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-21:30,11:00-21:30\r\n'
    'R51133639c9cc,모음(moeum),"서울 성동구 살곶이8길 22, 1층 (행당동)",37.5563145,127.0429786,한식,고등어 오일 스파게티니,19000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'R51133639c9cc,모음(moeum),"서울 성동구 살곶이8길 22, 1층 (행당동)",37.5563145,127.0429786,한식,봉골레,19000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'R51133639c9cc,모음(moeum),"서울 성동구 살곶이8길 22, 1층 (행당동)",37.5563145,127.0429786,한식,이베리코 갈비살 탈리아텔레,25000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,12:00-22:00,휴무\r\n'
    'R3f3d8b990182,문츠바베큐,"서울 성동구 상원1길 22, 굿타워 1층 101호 (성수동1가)",37.5456523,127.0485443,패밀리레스트랑,문츠 브리스킷 플래터,86000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'R3f3d8b990182,문츠바베큐,"서울 성동구 상원1길 22, 굿타워 1층 101호 (성수동1가)",37.5456523,127.0485443,패밀리레스트랑,문츠 비프립 플래터,120000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'R3f3d8b990182,문츠바베큐,"서울 성동구 상원1길 22, 굿타워 1층 101호 (성수동1가)",37.5456523,127.0485443,패밀리레스트랑,베이컨립,42000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'R3f3d8b990182,문츠바베큐,"서울 성동구 상원1길 22, 굿타워 1층 101호 (성수동1가)",37.5456523,127.0485443,패밀리레스트랑,비프립 단품,86000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'R3f3d8b990182,문츠바베큐,"서울 성동구 상원1길 22, 굿타워 1층 101호 (성수동1가)",37.5456523,127.0485443,패밀리레스트랑,포르게타,36000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,가정식 스테이크,18800,표시 가격 확인,구이·볶음·찜,스테이크,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,고기듬뿍 매운 하노이 쌀국수,10800,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,고기듬뿍 진한 소고기 쌀국수,10500,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,곱대 마제소바,14500,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,곱창 대창 덮밥(한정메뉴),14900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,곱창 스테이크(한정메뉴),22700,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,곱창&대창 덮밥 한상차림,16800,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,단새우 스테이크 덮밥 한상차림(S),17300,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,대창 덮밥,13500,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,대창 스테이크,21700,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,대창 큐브 스테이크 덮밥(대큐),15900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,리얼망고쥬스,5500,표시 가격 확인,음료,주스,단품,음료,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,마리네이드 토마토 사라다,7300,표시 가격 확인,샐러드,샐러드,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,명란 크림 파스타,13500,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,미도인 곱창 떡볶이 (한정메뉴),13900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,미도인 곱창 라멘(한정메뉴),13200,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,미도인 떡볶이,9900,표시 가격 확인,분식,떡볶이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,미도인 마제소바,10500,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,미도인 스카치 에그 (한정메뉴),5500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,미도인 스테이크 덮밥,11800,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,민물장어&스테이크한상(한정메뉴),17800,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,바질크림 새우파스타,15500,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,새우 로제 생면 파스타,14500,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,소막창 덮밥,12300,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,스폐셜 스테이크 한상차림,16000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,우니 스테이크 덮밥 한상차림(S),20800,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,찹쌀치즈볼,4900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,철판 소막창 볶음우동,14800,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,청두 사천 탄탄멘,9500,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,큐브 스테이크 덮밥,12800,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,큐브 스테이크 덮밥 한상차림,14900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,트리플 스테이크,19800,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,화산 불백 덮밥,9800,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R3fee2ef42cae,미도인 왕십리역사,서울 성동구 왕십리광장로 17 왕십리 민자역사 4층 4-6호,37.5612890,127.0383818,경양식,히츠마부시,27800,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rf55f4eb6a4d9,미분당 신당점,서울 성동구 왕십리로 410 센트라스 J동 B102호 (하왕십리동),37.5655624,127.0240882,베트남음식,양지쌀국수,11500,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rf55f4eb6a4d9,미분당 신당점,서울 성동구 왕십리로 410 센트라스 J동 B102호 (하왕십리동),37.5655624,127.0240882,베트남음식,차돌쌀국수,10000,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rf55f4eb6a4d9,미분당 신당점,서울 성동구 왕십리로 410 센트라스 J동 B102호 (하왕십리동),37.5655624,127.0240882,베트남음식,차돌양지쌀국수,12500,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rf55f4eb6a4d9,미분당 신당점,서울 성동구 왕십리로 410 센트라스 J동 B102호 (하왕십리동),37.5655624,127.0240882,베트남음식,차돌양지힘줄 쌀국수,13000,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rf55f4eb6a4d9,미분당 신당점,서울 성동구 왕십리로 410 센트라스 J동 B102호 (하왕십리동),37.5655624,127.0240882,베트남음식,힘줄쌀국수,14000,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,가니미소 대게딱지장,25000,표시 가격 확인,곁들임,곁들임,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,딱새우회,30000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,미쁘동,18000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,미쁘소바,17000,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,방어회 덮밥,23000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,성게알소바,18000,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,스지탕정식,18000,표시 가격 확인,국물요리,국·탕,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,연어모듬회,30000,표시 가격 확인,초밥·회,회·사시미,모둠,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,연어소바,17000,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,연어알연어동,19000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,우니도로동,30000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R5eef7eb929bc,미쁘동,서울 성동구 서울숲2길 11-12 2층,37.5475514,127.0401268,일식 회/초밥,토마토연어국수,22000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Ra846506fafb4,미소김밥,서울 성동구 무수막길 92,37.5539257,127.0201410,김밥/만두/분식,정식1 (돈까스 + 원조김밥 + 쫄면),9500,표시 가격 확인,여러 요리 구성,세부유형 미정,세트,식사,휴무,07:00-20:30,07:00-20:30,07:00-20:30,07:00-20:30,07:00-20:30,07:00-20:30\r\n'
    'Ra846506fafb4,미소김밥,서울 성동구 무수막길 92,37.5539257,127.0201410,김밥/만두/분식,정식2 (돈가스 + 참치김밥 + 우동),10500,표시 가격 확인,여러 요리 구성,세부유형 미정,세트,식사,휴무,07:00-20:30,07:00-20:30,07:00-20:30,07:00-20:30,07:00-20:30,07:00-20:30\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,갈비튀김교자,6000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,교자탕수,6000,표시 가격 확인,분식,만두,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,돈목살수비드냉채(오향장육),9000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,레몬 크림새우,8000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,마라양볶음,8000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,마파두부,7000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,매운새우완탕,7000,표시 가격 확인,국물요리,국·탕,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,명물 야끼교자,5000,표시 가격 확인,분식,만두,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,명물 야끼소바,9000,표시 가격 확인,국수류,볶음면,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,사이다,4000,표시 가격 확인,음료,탄산음료,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,사천갈릭새우,9000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,사천바지락볶음,9000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,사천식 오이목이냉채,5000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,스부타,8000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,어향가지,7000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,제로콜라,4000,표시 가격 확인,음료,탄산음료,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,차항(볶음밥),5000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,치킨 가라아게,7000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,치킨 난반,9000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,칠리새우,8000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,카라이 야끼소바,9000,표시 가격 확인,국수류,볶음면,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,콜라,4000,표시 가격 확인,음료,탄산음료,단품,음료,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,포테포 사라다,6000,표시 가격 확인,샐러드,샐러드,단품,사이드,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,항정살 미소볶음,9000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,흑후추 테바사키,7000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'R23c4dc3ad6aa,미스타교자 용답역점,"서울 성동구 용답19길 10, 1층 (용답동)",37.5629640,127.0521245,한식,히야시 토마토,3000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,17:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,갈비찜 대,72000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,갈비찜 소,52000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,갈비찜 중,62000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,고기가득 찐만두,15000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,꼬리곰탕,30000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,꼬리수육전골 대,88000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,꼬리수육전골 중,66000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,누룽지사리,5000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,당면사리,6000,표시 가격 확인,국수류,당면,추가,추가옵션,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,도가니탕,17000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,돈까스안주,15000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,뚝배기불고기,10000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,만두사리,5000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,비빔냉면,10000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,삼겹김치전,19000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,설렁탕,10000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,소머리곰탕,14000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,소면사리,5000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,손만두국,10000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,수육 대,70000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,수육 소,43000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,수육 중,55000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,수제돈까스,10000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,아삭이고추튀김,25000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,욱즙팡팡 튀김만두,15000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,특설렁탕,13000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Re442aab66673,민정식당 서울숲직영점,"서울 성동구 상원길 51, 1,2층 (성수동1가)",37.5486660,127.0480457,한식,해물파전,19000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rbf037e7186e7,민준이네,"서울 성동구 왕십리로 313, 1층 (행당동)",37.5616195,127.0344269,한식,양푼김치찌개 (점심),9000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbf037e7186e7,민준이네,"서울 성동구 왕십리로 313, 1층 (행당동)",37.5616195,127.0344269,한식,우리돼지 양념돼지갈비 (1인분),19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbf037e7186e7,민준이네,"서울 성동구 왕십리로 313, 1층 (행당동)",37.5616195,127.0344269,한식,제육볶음 (점심),9000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,고기덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,고기덮밥 세트,13900,표시 가격 확인,밥·죽류,덮밥,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,군만두,9000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,녹두전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,들깨칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,들깨칼국수 세트,14900,표시 가격 확인,국수류,칼국수,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,들깨칼국수 수육 한상차림,15900,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,미역국,10000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,미역국 세트,13900,표시 가격 확인,국물요리,국·탕,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,삼겹살 수육,26000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,손만두,9000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,육전,15000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,육전비빔칼국수,14000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,육전비빔칼국수 세트,15900,표시 가격 확인,국수류,칼국수,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,칼국수 세트,13900,표시 가격 확인,국수류,칼국수,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,칼국수 수육한상차림,13900,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,칼만둣국,13000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,칼만둣국 세트,15900,표시 가격 확인,국물요리,떡국·만둣국,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,콩국수,12000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R39be88c931e1,밀본,"서울 성동구 아차산로7길 4, 1층 (성수동2가)",37.5455642,127.0543971,한식,콩국수 세트,14900,표시 가격 확인,국수류,국수,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,갈비탕,15000,표시 가격 확인,국물요리,갈비탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,도가니탕,18000,표시 가격 확인,국물요리,국·탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,동죽조개 칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,떡만둣국,10000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,만두곰탕,10000,표시 가격 확인,분식,만두,단품,사이드,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,맑은나주곰탕,10000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,사골곰탕,10000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,소내장탕,13000,표시 가격 확인,국물요리,곱창탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,소머리곰탕(한우),14000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,순대국,10000,표시 가격 확인,국물요리,국밥,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,얼큰 칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,얼큰순대국,10000,표시 가격 확인,국물요리,국밥,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,얼큰우거지곰탕,11000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'R96f68d23d0a9,바우네나주곰탕 왕십리역점,왕십리로21길 10,37.5615957,127.0331852,국/탕/찌개류,제육볶음,10000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30,09:30-21:30\r\n'
    'Rc931b6c780d9,발이스타 왕십리본점,"서울 성동구 왕십리로 318, 1~2층 (도선동)",37.5622081,127.0345132,한식,콘마요 파닭바베큐,28000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,16:00-다음 날 02:00,16:00-다음 날 02:00,16:00-다음 날 02:00,16:00-다음 날 02:00,16:00-다음 날 02:00,16:00-다음 날 02:00,16:00-다음 날 01:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,게살라볶이,6500,표시 가격 확인,분식,떡볶이,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,게살치즈라볶이,7500,표시 가격 확인,분식,떡볶이,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,고추김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,김치 김밥,5800,표시 가격 확인,곁들임,반찬·무침,단품,사이드,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,김치제육덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,단짠진미채김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,돈까스 김밥,5800,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,땡초진미채김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,떡볶이 세트(방.꼬 + 쉐프 떡볶이),8000,표시 가격 확인,분식,떡볶이,세트,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,라면 세트(라면 + 방.꼬),7500,표시 가격 확인,국수류,라면,세트,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,방배김밥,4800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,부산밀면(비빔or물),7500,표시 가격 확인,국수류,냉면,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,비빔국수,6000,표시 가격 확인,국수류,국수,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,새우튀김김밥,5800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,소고기 김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,쉐프떡볶이,4500,표시 가격 확인,분식,떡볶이,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,쉐프치즈떡볶이,5500,표시 가격 확인,분식,떡볶이,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,어묵(3개),3000,표시 가격 확인,분식,어묵,단품,사이드,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,우엉김밥,4800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,잔치국수,6000,표시 가격 확인,국수류,국수,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,제육구이김밥,5800,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,제육덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,참치김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,치즈김밥,5800,표시 가격 확인,분식,김밥,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'Ra32f2dd46ed4,방배김밥 왕십리역점,서울 성동구 왕십리광장로 6 2층,37.5609417,127.0360735,김밥/만두/분식,튀김우동,7000,표시 가격 확인,국수류,우동,단품,식사,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00,09:00-20:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,그릴드치즈버거,9500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,그릴드치즈버거 세트,13200,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,더블그릴드치즈버거,12400,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,더블그릴드치즈버거 세트,16100,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,더블클래식치즈버거,9400,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,더블클래식치즈버거 세트,13100,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,더블트러플치즈버거,9500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,더블트러플치즈버거 세트,13200,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,버거리버거,7300,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,버거리버거 세트,11000,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,버팔로윙&스틱,5200,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,비프앤쉬림프버거,9100,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,비프앤쉬림프버거 세트,12800,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,새우링,4200,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,생와사비 더블알새우버거,9000,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,생와사비 더블알새우버거 세트,12700,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,생와사비 새우버거,6000,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,생와사비 새우버거 세트,9700,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,생와사비 알새우소고기버거,9500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,생와사비 알새우소고기버거 세트,13200,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,소불고기버거,5400,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,소불고기버거 세트,9100,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,쉬림프버거,5800,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,쉬림프버거 세트,9500,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,양념감자(멕시칸칠리),3500,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,양념감자(사워크림어니언),3500,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,양념너겟(멕시칸칠리),3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,양념너겟(사워크림어니언),3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,어니언링,3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,에그베이컨버거,8200,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,에그베이컨버거 세트,11900,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,에그불고기버거,6600,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,에그불고기버거 세트,10300,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,츄러스,3500,표시 가격 확인,디저트,구움과자,단품,디저트,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,치즈스틱,2700,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,치킨너겟,3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,코울슬로,2500,표시 가격 확인,샐러드,샐러드,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,콘샐러드,2500,표시 가격 확인,샐러드,샐러드,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,크리스피치킨버거,7200,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,크리스피치킨버거 세트,10900,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,크리스피치킨텐더,3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,클래식치즈버거,6500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,클래식치즈버거 세트,10200,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,클래식치킨버거,6800,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,클래식치킨버거 세트,10500,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,텍사스바베큐버거,4900,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,텍사스바베큐버거 세트,8600,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,통모짜치즈스틱,3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,통모짜치즈스틱(NEW),3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,트러플베이컨해쉬버거,8700,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,트러플베이컨해쉬버거 세트,12400,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,트러플치즈버거,6600,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,트러플치즈버거 세트,10300,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,프레쉬버거,5300,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,프레쉬버거 세트,9000,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,프렌치프라이(L),3000,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,하와이안버거,7700,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,하와이안버거 세트,11400,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,핫치폴레버거,7500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,핫치폴레버거 세트,11200,표시 가격 확인,피자·버거·샌드위치,버거,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R79ebdee8fb3a,버거리 한양대점,서울 성동구 왕십리로 231 신원1빌딩 1층,37.5573493,127.0415923,버거,해쉬브라운,2500,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rbbc3c2ffb36a,버섯집,"서울 성동구 왕십리로5길 9-10, 1층 (성수동1가)",37.5466930,127.0441345,한식,눈꽃버섯탕,12000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rbbc3c2ffb36a,버섯집,"서울 성동구 왕십리로5길 9-10, 1층 (성수동1가)",37.5466930,127.0441345,한식,들깨버섯탕,11000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rbbc3c2ffb36a,버섯집,"서울 성동구 왕십리로5길 9-10, 1층 (성수동1가)",37.5466930,127.0441345,한식,맑은버섯탕,10000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rbbc3c2ffb36a,버섯집,"서울 성동구 왕십리로5길 9-10, 1층 (성수동1가)",37.5466930,127.0441345,한식,버섯사리추가,8000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rbbc3c2ffb36a,버섯집,"서울 성동구 왕십리로5길 9-10, 1층 (성수동1가)",37.5466930,127.0441345,한식,버섯생불고기전골,17000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rbbc3c2ffb36a,버섯집,"서울 성동구 왕십리로5길 9-10, 1층 (성수동1가)",37.5466930,127.0441345,한식,얼큰버섯탕,10000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R11442d48cea1,베네세레-왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 B-04호 (행당동)",37.5612890,127.0383818,기타,마르게리따 피자,17000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R11442d48cea1,베네세레-왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 B-04호 (행당동)",37.5612890,127.0383818,기타,비스마르크 피자,18000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R11442d48cea1,베네세레-왕십리점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 B-04호 (행당동)",37.5612890,127.0383818,기타,토마토 페스카토레,17000,표시 가격 확인,파스타류,파스타,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R617e820d2141,베네토,서울 성동구 마조로 3길 7-1 1층,37.5584691,127.0396216,패스트푸드,알리오올리오 스파게티,9900,표시 가격 확인,파스타류,파스타,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,12:00-23:00,휴무\r\n'
    'R617e820d2141,베네토,서울 성동구 마조로 3길 7-1 1층,37.5584691,127.0396216,패스트푸드,페퍼로니 피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,12:00-23:00,휴무\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,고향집스페셜 2인 (400g),120000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,고향집스페셜 3인 (540g),150000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,고향집스페셜 4인 (720g),190000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,스페셜데이_럭셔리 (540g),180000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,스페셜데이_로얄 (720g),230000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,스페셜데이_커플 (400g),140000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 꽃살치살 (300g),120000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 꽃새우살 (300g),120000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 꽃업진살 (300g),80000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 꽃치마살 (300g),70000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 모둠 1 (800g),170000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 모둠 2 (700g),140000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 모둠 3 (550g),110000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 모둠 대판 (1kg),200000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 부채살 (300g),60000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 샤토브리앙 (300g),110000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 안창살 (300g),100000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 육사시미 (300g),60000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 제비추리 (300g),60000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 차돌박이 (300g),60000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 치마살 (300g),60000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우 토시살 (300g),70000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우갈비살 (300g),80000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우등심 (300g),90000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우안심 (300g),90000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'Rf101a7d77879,변산반도 고향집,서울 성동구 살곶이길 18 마장축산물시장 공영주차장 2층 3호,37.5715395,127.0400171,,한우육회 (300g),60000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40,11:00-23:40\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,*겨울시즌시작* 숙성대방어(소),53000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,*겨울시즌시작* 숙성대방어(중),73000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,*겨울한정* 숙성대방어(대),93000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,광어 (소),35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,광어(대),59000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,광어(중),47000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,광어+우럭 (소),37000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,깻잎막회 (소),35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,도다리(소),38000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,모둠회(중),48000,표시 가격 확인,초밥·회,회·사시미,모둠,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,모듬초밥,13000,표시 가격 확인,초밥·회,초밥,모둠,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,모듬회(대),60000,표시 가격 확인,초밥·회,회·사시미,모둠,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,모듬회(소),36000,표시 가격 확인,초밥·회,회·사시미,모둠,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,별 ★★★★넷 세트(스페셜)5~6인,148000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,별 ★★★셋 세트(패밀리)4인추천,108000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,별★★둘 세트(더블)3인 추천,75000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,새우오징어튀김,20000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,생새우회 + (머리버터구이),29000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,생선초밥,13000,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,시마아지+벤자리돔(소),50000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,여름한정) 벤자리돔(소),50000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,여름한정) 벤자리돔+광어(소),45000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,여름한정) 시마아지 + 광어 (소),45000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,여름한정)줄무늬전갱이(시마아지)(소),50000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,연어 (소),35000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,연어+광어 (소),35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,연어+광어+우럭 (중),49000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,연어+우럭 (소),37000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,연어초밥,14000,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,우럭 (소),38000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,우럭(대),62000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,우럭(중),50000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,줄돔(중),65000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R0781e680a281,별바다,서울 성동구 마장로 328 (마장동),37.5675001,127.0455692,일식,참돔(중),55000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'R10ef3bab8340,본떡볶이,서울 성동구 용답중앙길 42 (용답동) 1층,37.5650852,127.0509119,한식,떡볶이,4000,표시 가격 확인,분식,떡볶이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R156a5eeca470,본앤브레드컴퍼니,"서울 성동구 마장로42길 1, 새림약국 지하1,1,2,3층 (마장동)",37.5668041,127.0445212,식육(숯불구이),(Room) Dinner Tasting Menu,250000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,12:00-22:15,휴무,12:00-22:15,12:00-22:15,12:00-22:15,12:00-22:15,12:00-22:15\r\n'
    'R156a5eeca470,본앤브레드컴퍼니,"서울 성동구 마장로42길 1, 새림약국 지하1,1,2,3층 (마장동)",37.5668041,127.0445212,식육(숯불구이),Signature Tasting Menu,380000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,12:00-22:15,휴무,12:00-22:15,12:00-22:15,12:00-22:15,12:00-22:15,12:00-22:15\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,7가지야채죽,10000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,낙지김치비빔밥,12500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,낙지김치죽(해장),12500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,단호박죽,10500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,동지팥죽,10000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,본죽장조림버터비빔밥,12500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,불고기낙지비빔밥 (불낙),13500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,삼계전복죽(보양),17500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,삼계죽(보양),13500,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,소불고기버섯뚝배기,12000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,쇠고기야채죽,12500,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,신짬뽕비빔밥,12500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,신짬뽕죽(해장),12500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,전복내장죽,15000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,전복내장해물비빔밥,15500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,전복죽(보양),13000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,참깨계란새우쮺,12500,표시 가격 확인,미확인,미확인,단품,미확인,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,참치야채비빔밥,11500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,추어죽,15000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Re90a603981dc,본죽&비빔밥 왕십리뉴타운점,"서울 성동구 무학로 33, 156동 1층 106호 (하왕십리동, 텐즈힐)",37.5685657,127.0279661,한식,푸팟퐁커리게살쮺,17000,표시 가격 확인,커리·스튜,커리,단품,식사,09:30-21:00,09:30-21:00,09:30-21:00,09:30-21:00,09:30-20:30,09:30-15:30,09:30-15:30\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,7가지야채죽,9500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,낙지김치죽 (해장),11000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,본죽 단팥죽(달달한 새알팥죽),10500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,본죽 동지팥죽(담백한 밥알팥죽),10000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,새우죽,9500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,쇠고기미역죽,11000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,쇠고기버섯죽,11000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,전복죽 (보양),13000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,차돌해물짬뽕뚝배기,13000,표시 가격 확인,국수류,짬뽕,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,참치야채죽,10500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,통영굴버섯죽,10000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,특새우죽 (새우2배),12500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,특통영굴버섯죽 (굴2배),13000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,특해물죽 (해물2배),14500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,푸팟퐁커리게살쮺(반반쮺)(카레),16500,표시 가격 확인,커리·스튜,커리,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,해물죽,11500,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,홍게품은죽 (진) (홍게다리살2배),19000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,홍게품은죽 (특) (홍게다리살3배),24000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'Rb89329c99b07,본죽&비빔밥(행당동점),"서울 성동구 행당로 101, 1층 (행당동)",37.5579305,127.0301331,한식,흰죽,6000,표시 가격 확인,밥·죽류,죽,단품,식사,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-20:30,09:30-15:30,휴무\r\n'
    'R81010c455777,본추추어탕 용두직영점,서울 동대문구 한빛로 74 1층,37.5780980,127.0306913,,돼지갈비,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R81010c455777,본추추어탕 용두직영점,서울 동대문구 한빛로 74 1층,37.5780980,127.0306913,,새우추어탕(매운맛/순한맛),12000,표시 가격 확인,국물요리,추어탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R81010c455777,본추추어탕 용두직영점,서울 동대문구 한빛로 74 1층,37.5780980,127.0306913,,얼큰추어탕,10000,표시 가격 확인,국물요리,추어탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R81010c455777,본추추어탕 용두직영점,서울 동대문구 한빛로 74 1층,37.5780980,127.0306913,,추어탕,10000,표시 가격 확인,국물요리,추어탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R81010c455777,본추추어탕 용두직영점,서울 동대문구 한빛로 74 1층,37.5780980,127.0306913,,추어튀김(소),9000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R81010c455777,본추추어탕 용두직영점,서울 동대문구 한빛로 74 1층,37.5780980,127.0306913,,통추어탕(매운맛/순한맛),12000,표시 가격 확인,국물요리,추어탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,가자미구이,12000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,갈치구이,13000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,갈치구이 반상,16900,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,감자채전,9000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,강된장보리밥,12900,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,고사리 갈치조림 한상,17900,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,고사리 제육볶음 반상,16900,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,고사리제육볶음,15000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,돼지갈비찜,25000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,맛보기보쌈,12000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,벌교 꼬막무침,10000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,벌교 꼬막보리밥,12900,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이 2인세트,42000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이 3인세트,66000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이 4인세트,88000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이 5인세트,120000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이든든상,15900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이진심상,16900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이한상A,13900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,봄이한상B (보쌈),14900,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,외할머니 비빔국수,8000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,육회보리밥,13900,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,직화 쭈꾸미보쌈,28000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,직화 쭈꾸미볶음,18000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,직화 쭈꾸미볶음 반상,17900,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,쭈꾸미 해물파전,18000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,치즈떡갈비스테이크,12000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Raa13a87179c0,봄이보리밥 왕십리역점,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,한식,황금고등어구이,13000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R9ab6253ad61d,봉평옹심이메밀칼국수,서울 성동구 용답19길 11-1 (용답동),37.5633568,127.0521605,한식,비빔메밀국수,10000,표시 가격 확인,국수류,막국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-20:00\r\n'
    'R9ab6253ad61d,봉평옹심이메밀칼국수,서울 성동구 용답19길 11-1 (용답동),37.5633568,127.0521605,한식,옹심이 메밀칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-20:00\r\n'
    'R9ab6253ad61d,봉평옹심이메밀칼국수,서울 성동구 용답19길 11-1 (용답동),37.5633568,127.0521605,한식,옹심이만,12000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-20:00\r\n'
    'R9ab6253ad61d,봉평옹심이메밀칼국수,서울 성동구 용답19길 11-1 (용답동),37.5633568,127.0521605,한식,호박채전,12000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-20:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,갈비만두,6500,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,굴림만두,6500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,냉모밀,8500,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,떡국,7500,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,마늘새끼튀김만두,7000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,마늘튀김만두,7000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,만두 떡볶이,8000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,만두곰탕,9500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,만두육개장,9500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,만두짬뽕밥,9500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,매운국물쫄면,8000,표시 가격 확인,국수류,쫄면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,멸치칼국수+굴림만두5개,10500,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,멸치칼국수+튀김만두2개,11500,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,명물찐만두,6000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,모둠만두,14000,표시 가격 확인,분식,만두,모둠,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌국시,7000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌국시+튀김만두2개,10500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌국시+튀김새끼만두6개,10500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌냉만둣국,7500,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌만둣국,9000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌만둣굿+갈비만두4개,11500,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌만둣굿+굴림만두5개,11500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌멸치칼국수,8000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌비빔국시,8000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌얼큰만둣국,9500,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌얼큰칼국수,8500,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌얼큰칼만두,10500,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌칼만두,10000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌큰모둠만두,17000,표시 가격 확인,분식,만두,모둠,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌피냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌피냉면+튀김만두2개,11500,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,북촌피냉면+튀김새끼만두6개,11000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,블랙만두,6000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,블랙튀김만두,6000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,비빔국시+튀김만두,11500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,비빔냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,비빔냉면+튀김만두,11500,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,비빔냉면+튀김새끼만두,11000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,비빔만두,7500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,새끼만두,6000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,새우만두,6500,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,새우튀김만두,6500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,얼큰칼국수+굴림만두,11000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,얼큰칼국수+튀김만두,12000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,열무국시,8000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,열무냉면,8500,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,쫄면,7500,표시 가격 확인,국수류,쫄면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,쫄면+튀김만두2개,11500,표시 가격 확인,국수류,쫄면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,튀김만두,6000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,튀김새끼만두,6000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R14ed9bbc8a5c,북촌손만두 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 2층 이티-01호 (행당동)",37.5612890,127.0383818,한식,피쫄면,8500,표시 가격 확인,국수류,쫄면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R28a304a4b0ce,분99(Bun99),"서울 성동구 마조로 20, 1층 (행당동)",37.5589378,127.0409856,일식,분짜,11500,표시 가격 확인,국수류,국수,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R28a304a4b0ce,분99(Bun99),"서울 성동구 마조로 20, 1층 (행당동)",37.5589378,127.0409856,일식,양지 쌀국수,10000,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,2인엽기떡볶이,9000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,감자채튀김,2500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,계란 (2개),1500,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,계란야채죽,5000,표시 가격 확인,밥·죽류,죽,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,계란찜,2000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,국물닭발세트,19500,표시 가격 확인,구이·볶음·찜,직화닭발,세트,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,국물무뼈닭발,17000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,국물통뼈닭발,16000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,김말이 (3개),2000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,꿔바로우(5개),5900,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,당면사리,2000,표시 가격 확인,국수류,당면,추가,추가옵션,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,대파,1000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,떡추가,1000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,로제떡볶이,16000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,로제밀키트(오리지널/착한맛),11000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,마라떡볶이,16000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,마라로제떡볶이,18000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,만두 (4개),2000,표시 가격 확인,분식,만두,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,메추리알 (5개),1000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,모둠튀김,2000,표시 가격 확인,튀김·커틀릿,튀김,모둠,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,모짜치즈,3000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,바삭치즈만두 (7개),2000,표시 가격 확인,분식,만두,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,베스트세트,20000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,베이컨,3000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,분모자,2500,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,순대 (300g),3000,표시 가격 확인,분식,순대,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,숯불닭발세트,18500,표시 가격 확인,구이·볶음·찜,직화닭발,세트,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,숯불무뼈닭발,16000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,숯불통뼈닭발,15000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,스페셜세트,25000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,실속세트,17500,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,야채튀김 (1개),1000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,양배추,1000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽기닭볶음탕,24000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽기떡볶이,14000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽기밀키트(오리지널/착한맛),9000,표시 가격 확인,분식,떡볶이,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽기시즈닝(버터갈릭맛),300,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽기핫불소스,700,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽도그,2000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,엽봉(5개),5000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,오뎅 추가,1000,표시 가격 확인,분식,어묵,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,오뎅튀김(15개),2000,표시 가격 확인,분식,어묵,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,우동사리,2000,표시 가격 확인,국수류,우동,추가,추가옵션,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,우삼겹,3000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,주먹김밥(셀프),2000,표시 가격 확인,분식,김밥,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,중국당면,2500,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,참치마요밥,3500,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,콘마요,2500,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,통유부(4개),1000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,퐁당치즈만두 (7개),2000,표시 가격 확인,분식,만두,단품,사이드,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'R4c37ccfb681f,불닭발 땡초 동대문 엽기떡볶이 한양대점,"서울 성동구 마조로 15-6, 1층 (행당동)",37.5592565,127.0404355,한식,햄,1000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30,10:00-22:30\r\n'
    'Rca6f32cdd80f,브라운팟,무학봉28길 16,37.5616496,127.0336153,카페,,5000,대표가 추정(메뉴 미등록),,,,,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30\r\n'
    'Ra3c7122c9ee6,블랙빈스,"서울 성동구 고산자로 130, 1층 (응봉동)",37.5516796,127.0355274,기타,녹차 라떼,6000,표시 가격 확인,음료,라테,단품,음료,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00\r\n'
    'Ra3c7122c9ee6,블랙빈스,"서울 성동구 고산자로 130, 1층 (응봉동)",37.5516796,127.0355274,기타,밀크티 라떼,6000,표시 가격 확인,음료,라테,단품,음료,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00\r\n'
    'Ra3c7122c9ee6,블랙빈스,"서울 성동구 고산자로 130, 1층 (응봉동)",37.5516796,127.0355274,기타,아메리카노,4500,표시 가격 확인,음료,커피,단품,음료,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00\r\n'
    'Ra3c7122c9ee6,블랙빈스,"서울 성동구 고산자로 130, 1층 (응봉동)",37.5516796,127.0355274,기타,초콜릿 라떼,6000,표시 가격 확인,음료,라테,단품,음료,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00\r\n'
    'Ra3c7122c9ee6,블랙빈스,"서울 성동구 고산자로 130, 1층 (응봉동)",37.5516796,127.0355274,기타,카페모카,5900,표시 가격 확인,음료,커피,단품,음료,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00,08:30-22:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),[평일점심특선] 극가성비 육회비빔밥,8900,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),[평일점심특선]고사리오징어불고기,10900,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),가마솥 돼지구이 반판(420g),42500,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),가마솥 돼지구이한판(840g),74500,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),가마솥 선별반판(400g),39500,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),가마솥 선별한판(800g),69500,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),가마솥 시그니처 모듬세트(900g),92900,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),가마솥 시그니처 커플세트(500g),52900,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),고기 듬뿍 김치찌개,6900,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),공기밥(안성쌀),500,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),마무리 볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),미나리 한접시,3500,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),미숫가루하이볼,2900,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),삼죽육회 반접시(100g),11900,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),삼죽하이볼,4900,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),새콤달콤 비빔국수,6500,표시 가격 확인,국수류,국수,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),슬러시 김치말이국수,6500,표시 가격 확인,국수류,국수,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),시골 할머니 청국장,6900,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),안성맞춤[가오리무침 반접시100g],12900,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),옛날 인절미 아이스크림,2500,표시 가격 확인,디저트,아이스크림,단품,디저트,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),요거트 하이볼,2900,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),짐빔하이볼,8000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),짚불 빗살삼겹 (150g),10900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),짚불 빗살항정(150g),11900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),짚불 육즙목살(150g),9900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),짚불 특고기(150g),9900,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),짚불훈연 갈비본살(160g),22000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),체다 계란찜,3500,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'Ra8ec9c12ecb0,삼죽돈,"서울 성동구 무학봉16길 15, 1층 (행당동)",37.5603917,127.0326431,식육(숯불구이),치즈추가,3000,표시 가격 확인,곁들임,치즈,추가,추가옵션,11:30-22:30,11:30-22:30,11:30-22:30,11:30-22:30,11:30-24:00,12:00-24:00,12:00-24:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,"(비건,저당) 단호박 노릇두부 샐러디",9200,표시 가격 확인,샐러드,샐러드,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,"(비건,저당) 청양간장 두부 비빔밥",9200,표시 가격 확인,밥·죽류,비빔밥,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,(저당) 고추장제육 간장메밀 누들볼,11400,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,(저당) 그라브락스 연어 샐러디,11900,표시 가격 확인,샐러드,샐러드,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,(저당) 그라브락스 연어 파스타누들볼,11900,표시 가격 확인,파스타류,파스타,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,(저당) 그라브락스 연어 포케볼,11400,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,(저당) 치킨 콥 샐러디,9200,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,[프로틴] 바질 레몬연어 샌드위치,9700,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,[프로틴] 바질크림 치킨 샌드위치,7900,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,[프로틴] 비프에그마요 샌드위치,7600,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,[프로틴] 치킨 반미 샌드위치,7600,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,[프로틴] 프레시 잠봉 샌드위치,8900,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,고소우삼겹 곡물랩,8400,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,고추장 제육 포케볼,9900,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,고추장제육 들기름 파스타 누들볼,11600,표시 가격 확인,파스타류,파스타,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,그라브락스 연어 랩,9200,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,단호박 크림스프,4900,표시 가격 확인,국물요리,수프,단품,사이드,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,닭다리살 MAX 프로틴 박스,12900,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,로스트닭다리살 랩,8900,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,로스트닭다리살 샐러디,9900,표시 가격 확인,샐러드,샐러드,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,로스트닭다리살 샐러디 세트,12400,표시 가격 확인,샐러드,샐러드,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,맥시칸 랩 세트,9700,표시 가격 확인,피자·버거·샌드위치,랩·타코,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,메밀면볼,5000,표시 가격 확인,국수류,냉면,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,멕시칸 랩,7200,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,바베큐닭다리살 곡물랩,8400,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,바베큐닭다리살 포케볼,9900,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,바질치킨 랩,7900,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,바질크림 잠봉 랩,8200,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,불고기 반미 샌드위치,7900,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,불고기 반미 샌드위치 세트,10400,표시 가격 확인,피자·버거·샌드위치,샌드위치,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,소고기 비빔메밀 누들볼,8200,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,스파이시 쉬림프 샌드위치,9500,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,스프라이트제로 355ml,2400,표시 가격 확인,음료,탄산음료,단품,음료,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,시저치킨 랩,7200,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,아메리카노 (HOT/ICE),2700,표시 가격 확인,음료,커피,단품,음료,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,양송이 크림스프,4500,표시 가격 확인,국물요리,수프,단품,사이드,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,에그마요 랩,5500,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,우삼겹 메밀 누들볼,9300,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,우삼겹 메밀 누들볼 세트,11800,표시 가격 확인,샐러드,포케,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,우삼겹 포케볼,9400,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,우삼겹 MAX 프로틴 박스,12900,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,착즙 주스(그린밀싹),4500,표시 가격 확인,음료,주스,단품,음료,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,착즙 주스(레드클렌즈),4500,표시 가격 확인,음료,주스,단품,음료,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,착즙 주스(오렌지당근),4500,표시 가격 확인,음료,주스,단품,음료,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,채소볼,4500,표시 가격 확인,미확인,미확인,단품,미확인,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,치킨 MAX 프로틴 파스타,12900,표시 가격 확인,파스타류,파스타,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,치킨토마토 스튜,4900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,칠리베이컨 곡물랩,7400,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,칠리베이컨 포케볼,8900,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,칠리베이컨 포케볼 세트,11400,표시 가격 확인,샐러드,포케,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,카사바칩,1900,표시 가격 확인,곁들임,곁들임,단품,사이드,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,코크제로 355ml,2400,표시 가격 확인,음료,탄산음료,단품,음료,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,클래식 치킨 샌드위치,6500,표시 가격 확인,피자·버거·샌드위치,샌드위치,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,타코 쉬림프 랩,9200,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,타코 쉬림프 샐러디,10900,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,탄단지 샐러디,8600,표시 가격 확인,샐러드,샐러드,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,파스타볼,5000,표시 가격 확인,파스타류,파스타,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,포케볼,4500,표시 가격 확인,샐러드,포케,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R1833b66aa626,샐러디 한양대점,서울 성동구 마조로 24 1층,37.5598225,127.0413009,그 외 기타 간이 음식점,포테이토 크림스프,4500,표시 가격 확인,국물요리,수프,단품,사이드,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,10:00-23:00,10:00-23:00\r\n'
    'R6df8e2fd681c,샤브야키 성동왕십리,"서울 성동구 마조로 66, 102동 2층 205,206,207,208,209호 (마장동, 라봄 성동)",37.5636206,127.0419749,한식,미취학 아동 (36개월~7세),6900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R6df8e2fd681c,샤브야키 성동왕십리,"서울 성동구 마조로 66, 102동 2층 205,206,207,208,209호 (마장동, 라봄 성동)",37.5636206,127.0419749,한식,자숙 통문어,13000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R6df8e2fd681c,샤브야키 성동왕십리,"서울 성동구 마조로 66, 102동 2층 205,206,207,208,209호 (마장동, 라봄 성동)",37.5636206,127.0419749,한식,주말·공휴일 (성인),30900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R6df8e2fd681c,샤브야키 성동왕십리,"서울 성동구 마조로 66, 102동 2층 205,206,207,208,209호 (마장동, 라봄 성동)",37.5636206,127.0419749,한식,초등학생,13900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R6df8e2fd681c,샤브야키 성동왕십리,"서울 성동구 마조로 66, 102동 2층 205,206,207,208,209호 (마장동, 라봄 성동)",37.5636206,127.0419749,한식,평일 디너 (성인),29900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R6df8e2fd681c,샤브야키 성동왕십리,"서울 성동구 마조로 66, 102동 2층 205,206,207,208,209호 (마장동, 라봄 성동)",37.5636206,127.0419749,한식,평일 런치 (성인),24900,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R48e080e458a4,서리재 왕십리역점,서울 성동구 왕십리광장로 17 (상세주소 미표시),37.5612890,127.0383818,한식,돼지고기 김치찌개 솥밥 정식,14900,표시 가격 확인,밥·죽류,솥밥,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R48e080e458a4,서리재 왕십리역점,서울 성동구 왕십리광장로 17 (상세주소 미표시),37.5612890,127.0383818,한식,직화 제육 2인 차림,49900,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R48e080e458a4,서리재 왕십리역점,서울 성동구 왕십리광장로 17 (상세주소 미표시),37.5612890,127.0383818,한식,직화 제육 정찬,23500,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R48e080e458a4,서리재 왕십리역점,서울 성동구 왕십리광장로 17 (상세주소 미표시),37.5612890,127.0383818,한식,차돌 된장찌개 솥밥 정식,13900,표시 가격 확인,밥·죽류,솥밥,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R48e080e458a4,서리재 왕십리역점,서울 성동구 왕십리광장로 17 (상세주소 미표시),37.5612890,127.0383818,한식,차돌 영양부추 덮밥,15000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R8c8a465bc9d9,서울숲 오삼불고기,"서울 성동구 왕십리로8길 8 (성수동1가,(지상1층))",37.5463244,127.0456224,한식,부대찌개,11000,표시 가격 확인,국물요리,찌개류,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R8c8a465bc9d9,서울숲 오삼불고기,"서울 성동구 왕십리로8길 8 (성수동1가,(지상1층))",37.5463244,127.0456224,한식,오삼불고기,11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R8c8a465bc9d9,서울숲 오삼불고기,"서울 성동구 왕십리로8길 8 (성수동1가,(지상1층))",37.5463244,127.0456224,한식,오징어무채쌈 대,35000,표시 가격 확인,미확인,미확인,단품,미확인,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R8c8a465bc9d9,서울숲 오삼불고기,"서울 성동구 왕십리로8길 8 (성수동1가,(지상1층))",37.5463244,127.0456224,한식,오징어볶음,10000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R8c8a465bc9d9,서울숲 오삼불고기,"서울 성동구 왕십리로8길 8 (성수동1가,(지상1층))",37.5463244,127.0456224,한식,제육볶음,10000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,감자와버터,3500,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,돌판짜파게티,7000,표시 가격 확인,국수류,라면,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,떡쌈,4000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,로스삼겹,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,명란계란탕,7000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,모둠(삼겹+천겹+등심덧살) 400g,38000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,묵사발,4000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,물냉면,6500,표시 가격 확인,국수류,냉면,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,비빔냉면,6500,표시 가격 확인,국수류,냉면,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,삼겹,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,생미나리,5500,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,석암볶음밥,6000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,천겹,16000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,치즈구이,6000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,치즈된장술밥,7500,표시 가격 확인,국물요리,국밥,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,한우차돌박이(1++),25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'Rbc3e3f090eef,석암 생소금구이 왕십리점,"서울 성동구 왕십리로 311-1, 1층 (행당동)",37.5616358,127.0345954,한식,해물된장찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"갈치,곤드레솥밥",16000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"고등어,김메밀",16000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"단호박 치즈 케이크,녹차소금",9500,표시 가격 확인,디저트,케이크,단품,디저트,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"돌문어,제철 샐러드",17000,표시 가격 확인,샐러드,샐러드,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"맥적 닭구이,표고 와사비",17000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"수육,청어알 무김치,시금장",23000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"알감자 튀김,청양고추 마요",12000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"육회 묵은지말이,고추장 마요",17000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R56970b07bd31,선채 성수,"서울 성동구 아차산로1길 11, 지하1층 (성수동1가)",37.5490856,127.0457552,한식,"전복,칼비빔",16000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,감자전,17000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,두부김치전골,28000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,들기름막국수,12000,표시 가격 확인,국수류,막국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,들깨메밀수제비,12000,표시 가격 확인,국수류,수제비,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,들깨메밀칼국수,12000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,메밀왕만두,8000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,메밀전병,8000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,물막국수,12000,표시 가격 확인,국수류,막국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,버섯만두전골,28000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,보쌈(대),50000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,보쌈(소),30000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,비빔막국수,12000,표시 가격 확인,국수류,막국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,성수묵사발,12000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,성수한상,15000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,수육(반접시/한접시),8000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,얼큰수제비,12000,표시 가격 확인,국수류,수제비,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,쟁반막국수,35000,표시 가격 확인,국수류,막국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,추가막국수,6000,표시 가격 확인,국수류,막국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'R5383dd7fe985,성수막국수,"서울 성동구 아차산로 3, 성수동 아파트형 공장 두앤캔 하우스 1층 101,102호 (성수동1가)",37.5483591,127.0453959,한식,해물부추전,18000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,휴무\r\n'
    'Rb2605ec2a551,성수식당,서울 성동구 성수일로12길 19-1 (성수동2가) 1층,37.5484247,127.0534354,한식,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rb2605ec2a551,성수식당,서울 성동구 성수일로12길 19-1 (성수동2가) 1층,37.5484247,127.0534354,한식,삼겹살,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,들기름 골동면,11000,표시 가격 확인,국수류,국수,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,들기름 골동반,11000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,등심 돈까스 1pcs,4000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,로제 치킨새우 돈까스,14800,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,매콤 바질로제 돈까스,12000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,반숙 계란장,1500,표시 가격 확인,곁들임,계란,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,새우튀김,3000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,수제 가지 만두,12000,표시 가격 확인,분식,만두,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,에그 스프레드 치킨 덮밥,11000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,완당 (만두만 있어요),12000,표시 가격 확인,분식,만두,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,점장 추천 세트,36000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,줄서는 식당 2인,32500,표시 가격 확인,미확인,미확인,단품,미확인,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,줄서는 식당 3인 세트,48000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,직화 불고기,3500,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,직화불고기 덮밥,11000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,직화불고기 미니 덮밥,4000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,청양 에그 스프레드 돈까스,12000,표시 가격 확인,국물요리,수프,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,청양 에그 스프레드 치킨새우 돈까스,14800,표시 가격 확인,국물요리,수프,단품,사이드,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,촉촉 바삭 돈까스,10500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,커플을 위한 2인 세트,29000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,통닭다리 치킨,5000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc2009560f195,성수완당 왕십리역직영점,"서울 성동구 왕십리광장로 17, 비트플렉스 3층 기-01호 (행당동)",37.5612890,127.0383818,한식,해장 완당면,12000,표시 가격 확인,국수류,국수,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,막걸리,5000,표시 가격 확인,음료,막걸리,단품,음료,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,맥주,5000,표시 가격 확인,음료,맥주,단품,음료,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,소주,5000,표시 가격 확인,음료,소주,단품,음료,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,음료수,3000,표시 가격 확인,음료,기타 음료,단품,음료,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,족발 (대)3인분,50000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,족발 (중)2~3인분,45000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,족발 (특대)4인분,55000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R14620822f81b,성수족발,서울 성동구 아차산로7길 7 (성수동2가),37.5460229,127.0542140,한식,청하,7000,표시 가격 확인,음료,전통주,단품,음료,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40,12:00-21:40\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,양장피(소),30000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,양장피(중),40000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,짜장면,8000,표시 가격 확인,국수류,짜장면,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,차돌짬뽕,13000,표시 가격 확인,국수류,짬뽕,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,탕수육(소),25000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,탕수육(중),32000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R19b7f6c843d7,성수차이나,"서울 성동구 아차산로7길 20, 지상1층 (성수동2가)",37.5468146,127.0549599,일식,해물짬뽕,10000,표시 가격 확인,국수류,짬뽕,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무,10:30-20:30\r\n'
    'R4a3cc9adf757,세진식당,서울 동대문구 서울시립대로 61-4 (전농동),37.5768265,127.0477728,백반/한정식,된장찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-15:00,휴무\r\n'
    'R4a3cc9adf757,세진식당,서울 동대문구 서울시립대로 61-4 (전농동),37.5768265,127.0477728,백반/한정식,백반,6000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-15:00,휴무\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,광어 초밥(10pcs),24000,표시 가격 확인,초밥·회,초밥,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,냉모밀,9000,표시 가격 확인,국수류,모밀·소바,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,모둠초밥(11pcs),16000,표시 가격 확인,초밥·회,초밥,모둠,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,사시미 소(1~2인),29000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,사시미 중(2~3인),43000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,연어 초밥(10pcs),22000,표시 가격 확인,초밥·회,초밥,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,연어회덮밥,15000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,특 모둠초밥A(12pcs),18000,표시 가격 확인,초밥·회,초밥,모둠,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,특 모둠초밥B(12pcs),29000,표시 가격 확인,초밥·회,초밥,모둠,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'R706e5adb0469,세키신 왕십리점,무학봉26길 10-1 1층,37.5615868,127.0328116,초밥·롤,회덮밥,12000,표시 가격 확인,밥·죽류,덮밥,단품,식사,17:30-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00,12:00-23:00\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,구마볶 구이 마무리 볶음밥 입니다,5000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,"모둠구이 550g 신선한 채소들과, 곱창, 막창, 대창, 염통 까지 한 번에 즐기실 수 있습니다",59000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,모둠,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,차돌 된장 술밥 밥이 말아져서 나오는 술밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,치즈 불닭볶음면 불닭볶음면과 치즈의 만남,5000,표시 가격 확인,국수류,라면,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,"한우곱창 하루하루 제일 신선한 곱창을 공수해와 정성스럽게 손질과, 초벌 후 맛있게 구워드립니다.",28000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,한우대창 두툼하고 뚱뚱한 대창이 고소함이 일품입니다.,28000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,한우막창 노릇노릇하게 구워서 담백하니 소주안주로 최고입니다.,28000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rff9af18143ef,소꿉놀이,서울 성동구 무학봉28길 3 (하왕십리동),37.5624699,127.0324838,백반/한정식,한우염통 마장동에서 제일 신선한 염통을 선별해 옵니다.,17000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Rba40250b63ad,소나무,서울 성동구 행당로17길 20,37.5601183,127.0332872,한식,된장찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30\r\n'
    'Rba40250b63ad,소나무,서울 성동구 행당로17길 20,37.5601183,127.0332872,한식,물냉면,6000,표시 가격 확인,국수류,냉면,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30\r\n'
    'Rba40250b63ad,소나무,서울 성동구 행당로17길 20,37.5601183,127.0332872,한식,비빔냉면,6000,표시 가격 확인,국수류,냉면,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30\r\n'
    'Rba40250b63ad,소나무,서울 성동구 행당로17길 20,37.5601183,127.0332872,한식,한우소갈비살1인분,21000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,고춧가루제육볶음,13800,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,발효청 스파클링(오미자/사과),5000,표시 가격 확인,음료,기타 음료,단품,음료,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,발효청 음료(오미자/사과),4000,표시 가격 확인,음료,기타 음료,단품,음료,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,산나물밥,9800,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,소녀방앗간 꽃핀수육,17800,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,우렁된장찌개,13800,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,장아찌불고기밥,12800,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,제로사이다,2000,표시 가격 확인,음료,탄산음료,단품,음료,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,참명란비빔밥,12800,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,찹쌀고추장찌개,13800,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R0ae63dcb8539,소녀방앗간 서울숲시작점,서울 성동구 왕십리로5길 9-16 (성수동1가),37.5465615,127.0441247,한식,채소영양밥,10800,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30,11:30-20:30\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,땡초참치김밥,6000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,떡라면,5000,표시 가격 확인,국수류,라면,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,떡볶이,5500,표시 가격 확인,분식,떡볶이,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,라면,4500,표시 가격 확인,국수류,라면,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,라볶이,6000,표시 가격 확인,분식,떡볶이,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,멸추김밥,6000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,비빔국수,7500,표시 가격 확인,국수류,국수,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,소담김밥,4000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,어린이김밥,3000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,와사비참치김밥,5000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,잔치국수,7000,표시 가격 확인,국수류,국수,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,제육덮밥,9000,표시 가격 확인,밥·죽류,덮밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,진미채김밥,6000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,짜계치(짜파게티계란치즈),6000,표시 가격 확인,국수류,라면,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,참치김밥,5000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,참치키토김밥,6000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,치즈김밥,5000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,치즈떡볶이,6000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,치즈라면,5000,표시 가격 확인,국수류,라면,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,치즈라볶이,6500,표시 가격 확인,분식,떡볶이,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,치즈키토김밥,6000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,키토김밥,5000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,피칸테마요김밥,7000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,피칸테오리진김밥,8000,표시 가격 확인,분식,김밥,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'R8e37cce3579d,소담이네김밥&국수,"서울 성동구 천호대로 286 1층, 소담이네김밥",37.5665284,127.0524663,,해장라면,6500,표시 가격 확인,국수류,라면,단품,식사,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,07:00-14:00,휴무\r\n'
    'Rf75c96c81983,소림마라 동대문점,서울 동대문구 황물로 90 보광타워 1층 105-1호,37.5703555,127.0501245,마라탕/훠궈,꿔바로우(찹쌀탕수육),18000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-21:40\r\n'
    'Rf75c96c81983,소림마라 동대문점,서울 동대문구 황물로 90 보광타워 1층 105-1호,37.5703555,127.0501245,마라탕/훠궈,마라샹궈 기본1인분,15000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-21:40\r\n'
    'Rf75c96c81983,소림마라 동대문점,서울 동대문구 황물로 90 보광타워 1층 105-1호,37.5703555,127.0501245,마라탕/훠궈,마라샹궈100g,3000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-21:40\r\n'
    'Rf75c96c81983,소림마라 동대문점,서울 동대문구 황물로 90 보광타워 1층 105-1호,37.5703555,127.0501245,마라탕/훠궈,마라탕 기본1인분,8000,표시 가격 확인,국물요리,마라탕,단품,식사,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-21:40\r\n'
    'Rf75c96c81983,소림마라 동대문점,서울 동대문구 황물로 90 보광타워 1층 105-1호,37.5703555,127.0501245,마라탕/훠궈,마라탕100g,1650,표시 가격 확인,국물요리,마라탕,단품,식사,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-21:40\r\n'
    'Rf75c96c81983,소림마라 동대문점,서울 동대문구 황물로 90 보광타워 1층 105-1호,37.5703555,127.0501245,마라탕/훠궈,볶음밥,6000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-다음 날 03:00,10:30-21:40\r\n'
    'R8c5c5111cf2c,소문난순대국왕족발,서울 성동구 용답25라길 23 (용답동) 1층,37.5635166,127.0530048,한식,미니족,5000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R8c5c5111cf2c,소문난순대국왕족발,서울 성동구 용답25라길 23 (용답동) 1층,37.5635166,127.0530048,한식,순대국,8000,표시 가격 확인,국물요리,국밥,단품,식사,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,규동,13800,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,낫또소바 (냉),17800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,니싱소바 (냉•온),21800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,니쿠소바 (온),18800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,스페셜텐동,13800,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,에비텐소바(냉•온),18800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,온천계란,1000,표시 가격 확인,곁들임,계란,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,와카메소바(냉•온),15800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,자루소바(냉),13800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,카모 세이로,21800,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,카케소바(냉•온),13800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,텐뿌라,12800,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'R5a39434e9886,소바마에 니고,"서울 성동구 연무장길 39-15, 지1층 (성수동2가)",37.5433354,127.0539017,일식,토로로소바(냉•온),17800,표시 가격 확인,국수류,모밀·소바,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-20:30,11:00-20:30\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,고기왕만두 반접시,4000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,고기왕만두 한접시,8000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,냉면 사리추가,6000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,달꽃 보쌈 반접시,10000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,달꽃 보쌈 한접시,20000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,명태회냉면,12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,명태회냉면 매운맛,12000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,명태회무침 포장,16000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,소고기국밥,10000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,소칼,12000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,아바이순대 반접시,10000,표시 가격 확인,분식,순대,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,아바이순대 한접시,20000,표시 가격 확인,분식,순대,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,알고니 칼국수,14000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,알고니탕,14000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,육밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,육칼,12000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,함흥물냉면,10000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,함흥물냉면 매운맛,10000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rcd41e7e7a616,속초면옥 달꽃식당,"서울 성동구 왕십리로 410, L동 1층 121,122호 (하왕십리동, 센트라스)",37.5661105,127.0276465,한식,해장냉면,11000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,갈치솥밥,17900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,꼬막 솥밥,14900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,도미관자솥밥,17900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,돼지고사리솥밥,14900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,민물장어솥밥,25900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,스테이크솥밥,17900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,유린기,8500,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,전복솥밥,17900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,직접 만든 새우튀김,7900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R404f9d9e40af,솔솥 성수점,"서울 성동구 연무장5가길 24, 1층 (성수동2가)",37.5432098,127.0553023,기타,치즈닭갈비솥밥,14900,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R49fb89a06baa,송하정,서울 성동구 무학봉길 77 (하왕십리동) .,37.5613079,127.0310798,한식,백반,9000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,휴무,휴무\r\n'
    'R1417159136e6,수상한막창,"서울 성동구 무학봉28길 20, 1층 (행당동)",37.5614532,127.0339165,한식,된장밥,5000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,15:00-23:30,16:00-23:30\r\n'
    'R1417159136e6,수상한막창,"서울 성동구 무학봉28길 20, 1층 (행당동)",37.5614532,127.0339165,한식,된장찌개,5000,표시 가격 확인,국물요리,찌개류,단품,식사,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,15:00-23:30,16:00-23:30\r\n'
    'R1417159136e6,수상한막창,"서울 성동구 무학봉28길 20, 1층 (행당동)",37.5614532,127.0339165,한식,소금구이 300g,18000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,15:00-23:30,16:00-23:30\r\n'
    'R1417159136e6,수상한막창,"서울 성동구 무학봉28길 20, 1층 (행당동)",37.5614532,127.0339165,한식,양념구이 300g,18000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,15:00-23:30,16:00-23:30\r\n'
    'R1417159136e6,수상한막창,"서울 성동구 무학봉28길 20, 1층 (행당동)",37.5614532,127.0339165,한식,주먹밥,3000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,16:00-23:30,15:00-23:30,16:00-23:30\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,[signature] 유자미나리세비체,26000,표시 가격 확인,초밥·회,회·사시미,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,1++한우감태육회,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,리얼수박셔벗,13000,표시 가격 확인,디저트,아이스크림,단품,디저트,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,마라크림새우,25000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,명태회들기름메밀면,20000,표시 가격 확인,국수류,냉면,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,소울타코 (미나리타코),18000,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,오리가슴살,32000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,일품관자송이선,25000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,전통주 스페셜 하이볼,10000,표시 가격 확인,음료,하이볼,단품,음료,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,제주고사리비빔면,22000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,차돌된장술밥,20000,표시 가격 확인,국물요리,국밥,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,청어알두부플래터,15000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,토하젓 미나리 오일파스타,22000,표시 가격 확인,파스타류,파스타,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,항정구이와 여수칠게장,32000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,해물고추장크림옹심이,26000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,해물누룽지탕,28000,표시 가격 확인,국물요리,국·탕,단품,식사,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'R00671bf42a7f,수울래 금호,서울 성동구 독서당로 326 3층,37.5479157,127.0251226,한식,흑초가지새우,25000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,17:00-23:30,휴무\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,공기밥,2000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,닭갈비 철판 볶음밥(오늘은 2인 Size Up),21900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,닭갈비 철판 볶음밥(오늘은 기본),14900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,"막국수(물, 비빔)",10000,표시 가격 확인,국수류,막국수,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,매생이 손수제비(오늘은 기본),14400,표시 가격 확인,국수류,수제비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,매운 손수제비(오늘은 기본),13400,표시 가격 확인,국수류,수제비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,매운 손수제비(오늘은 Size Up),15400,표시 가격 확인,국수류,수제비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,세트 메뉴 ( 닭갈비 + 수제비 )(1인 세트 (300g)),26300,표시 가격 확인,구이·볶음·찜,닭갈비,세트,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,세트 메뉴 ( 닭갈비 + 수제비 )(2인 세트 (600g)),41300,표시 가격 확인,구이·볶음·찜,닭갈비,세트,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,세트 메뉴 ( 닭갈비 + 수제비 )(3인 세트 (900g)),56300,표시 가격 확인,구이·볶음·찜,닭갈비,세트,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,세트 메뉴 ( 닭갈비 + 수제비 )(4인 세트 (1.2kg)),71300,표시 가격 확인,구이·볶음·찜,닭갈비,세트,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,"소세지+오뎅+떡+고구마+면사리(쫄면,라면,우동 택1)",11900,표시 가격 확인,분식,어묵,단품,사이드,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,야채 비빔밥,10000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,양념 닭갈비 2인 (600g),29900,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,양념 닭갈비 3인 (900g),44900,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,양념 닭갈비 4인 (1.2kg),59900,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,"자연치즈100+고구마+면사리(쫄면,라면,우동 택1)",9400,표시 가격 확인,국수류,쫄면,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,"자연치즈100+떡+면사리(쫄면,라면,우동 택1)",8900,표시 가격 확인,국수류,쫄면,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,칼국수,7500,표시 가격 확인,국수류,칼국수,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,항아리 손수제비(오늘은 기본),12400,표시 가격 확인,국수류,수제비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'Rb43b33bd6efd,수제비먹는닭갈비,"서울 성동구 왕십리로21다길 39, 1층 (행당동)",37.5596373,127.0332377,한식,항아리 손수제비(오늘은 Size Up),14400,표시 가격 확인,국수류,수제비,단품,식사,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50,11:00-23:50\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,28년 전통 양념 철판 닭갈비,13000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,김치전,5000,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,낙지와 닭갈비,16000,표시 가격 확인,구이·볶음·찜,닭갈비,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,물막국수(칡),7000,표시 가격 확인,국수류,막국수,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,비빔막국수(칡),7500,표시 가격 확인,국수류,막국수,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,서리태콩국수,10000,표시 가격 확인,국수류,국수,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,수제비,7500,표시 가격 확인,국수류,수제비,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,웰빙야채비빔밥,7500,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,통큰 낙지 철판볶음밥,10000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,통큰 닭 철판볶음밥,9000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,항아리 손 수제비,7500,표시 가격 확인,국수류,수제비,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,항아리 손 수제비 특,9500,표시 가격 확인,국수류,수제비,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,항아리미역칼국수,7500,표시 가격 확인,국수류,칼국수,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,항아리미역칼국수 특,9500,표시 가격 확인,국수류,칼국수,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,"항아리손수제비(매운, 미역, 얼큰)",8500,표시 가격 확인,국수류,수제비,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R408592a0556e,수제비먹는닭갈비,왕십리로21길 14 1층,37.5612676,127.0329186,한식,"항아리손수제비(매운, 미역, 얼큰) 특",10000,표시 가격 확인,국수류,수제비,단품,식사,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30,10:00-23:30\r\n'
    'R7d9f43a08d9e,수현횟집,무학봉28길 12,37.5618750,127.0331966,일식 회/초밥,복 매운탕,15000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R7d9f43a08d9e,수현횟집,무학봉28길 12,37.5618750,127.0331966,일식 회/초밥,산우럭탕,24000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R7d9f43a08d9e,수현횟집,무학봉28길 12,37.5618750,127.0331966,일식 회/초밥,생대구탕,12000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R7d9f43a08d9e,수현횟집,무학봉28길 12,37.5618750,127.0331966,일식 회/초밥,알탕,8000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,"40년 닭볶음탕(大)양념 쏙쏙 밴 닭볶음탕, 엄마가 해주던 그맛!",45000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,감자탕(小)둘이 먹으면 배부른 양!,30000,표시 가격 확인,국물요리,감자탕,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,감자탕(中)우리가 상상하던 그 감자탕!,40000,표시 가격 확인,국물요리,감자탕,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,고기 많은 감자탕(大)깻잎 들깨향 가득한 여러분이 찾던 뼈다귀 감자탕!,45000,표시 가격 확인,국물요리,감자탕,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,닭볶음탕(小)두명이 즐기기 넉넉해요!,30000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,"닭볶음탕(中)큼직한 닭에, 푹 익은 감자까지!",40000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,볶음밥 필수코스! 새콤달콤 시그니쳐 볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R97c076335691,순천식당,서울 성동구 왕십리로24길 9-1 (도선동),37.5634335,127.0340915,백반/한정식,"한돈 냉동삼겹살 (200g)최고등급 한돈, 두툼한 냉삼은 맛이 다릅니다!",18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00,15:00-24:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,곤부지메 대광어스시(2pc),5200,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,곤부지메 엔가와스시(2pc),6900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,근본이 살아있는 본품스시(10pc),19900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,기특함 가득찬 특품스시(12pc),25900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,단짠 다마고스시(2pc),2900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,달콤큼직 홍새우스시(2pc),5900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,닭다리살가라아게,12900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,뜨끈즉석우동,8000,표시 가격 확인,국수류,우동,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,바르게 담은 정품 스시(10pc),15900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,본품스시(10pc)와 특제모밀정식,22800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,본품스시와 13품 술상,35800,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,본품스시와 3품요리 정식,25800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,살얼음냉모밀,8000,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,수제 간장새우스시(2pc),6900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,신선 유부스시(2pc),1900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,양품스시(8pc)와 특제모밀정식,18800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,양품스시와 13품 술상,31800,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,양품스시와 3품요리 정식,21800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,작품스시(13pc)와 특제모밀정식,38800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,작품스시와 13품 술상,51800,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,작품스시와 3품요리 정식,41800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,저온숙성 참돔스시(2pc),6900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,제철우니군함스시 (1oc),7900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,특대형 초새우스시(2pc),4900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,특품스시(12pc)와 특제모밀정식,28800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,특품스시와 13품 술상,41800,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,특품스시와 3품요리 정식,31800,표시 가격 확인,초밥·회,초밥,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,한점 한점 예술인 작품스시(13pc),35900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,S급 참치 속살스시(2pc),5900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R1c23906ad234,스시도쿠,서울 성동구 왕십리광장로 6 1층 스시도쿠 왕십리본점,37.5609417,127.0360735,일식,S급 혼마구로 뱃살스시(1pc),6900,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,‘뜨끈가득’ 라이브 차돌스키야키,24900,표시 가격 확인,국물요리,전골,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,’별품‘ 세계일 제철 우니 삼합,24900,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,‘시그니처 스시와 차돌스키야키 특정식,35900,표시 가격 확인,초밥·회,초밥,세트,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,‘즉석팔팔’ 라이브 차돌우동,14900,표시 가격 확인,국수류,우동,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,320 참치 타르타르,29900,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,곤부지메 연어스시 3pc,7900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,그란도 후토마끼 (8pc),32800,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,그란도 후토마키 (4pc),16900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,눈꽃치즈고로케(2pc),5900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,다마고카츠(2pc),5900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,더성수 사시미,69000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,묵은지 곤부지메 대광어스시 3pc,7900,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,본품(本品)스시,19900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,산토리 하이볼,8000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,살얼음 판모밀 (5~10월 한정),12900,표시 가격 확인,국수류,모밀·소바,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,생생활스시 3pc,8900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,생생활스시 5pc,12900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,셰프 특제 육회 타르타르,18900,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,순도 100% 닭다리살 가라아게,12900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,스시와 차돌우동 정식,25900,표시 가격 확인,초밥·회,초밥,세트,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,스페셜 모둠덴뿌라,15900,표시 가격 확인,튀김·커틀릿,튀김,모둠,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,야마하이(山廃) ‘레알’ 유자 하이볼,9000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,업그레이드 스시와 스페셜덴뿌라 특정식,42900,표시 가격 확인,초밥·회,초밥,세트,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,엔트리 스시와 차돌스키야키 정식,29900,표시 가격 확인,초밥·회,초밥,세트,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,왕새우덴뿌라(2pc),5900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,우니스시 1pc,6900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,절품(絕品)스시,32900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,주방장 수제 타코와사비,11900,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,진저 더블 레몬 하이볼,9000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,짐빔 하이볼,8000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,천연벌꿀 스모키얼그레이 하이볼,9000,표시 가격 확인,음료,하이볼,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,특품(特品)스시,25900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,하이엔드 스시와 6품(六品) 코스요리,69000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,핸드메이드 간장새우장 (7pc),12900,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf8c4c4d602a9,스시도쿠 더성수,서울 성동구 서울숲6길 13 Forest Seongsu 1층,37.5477165,127.0430659,일식,S급 혼마구로 뱃살스시 1pc,5900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,[시그니처] 특품카미동,23900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,[인기메뉴] 아부리동,21900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,나마生혼마구로동,31900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,다마고카츠 4pc,5900,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,본품 카미동,17900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,사케동,21900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,오도로우니동,34900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,우니듬뿍동,40900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,작품 카미동,32900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,절품 카미동,43900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R8b44140fbfe7,스시도쿠 카미동 왕십리본점,서울 성동구 왕십리로 295 왕십리가축병원 2층,37.5604413,127.0357361,일식,후토마끼 4pc,17900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,디너 테이블오마카세(기),140000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,디너 테이블오마카세(동),110000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,디너 테이블오마카세(이),85000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,런치 테이블오마카세특선(카),65000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,런치(무),45000,표시 가격 확인,여러 요리 구성,미확인,단품,식사 추정,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,사시미 포장,130000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,"스시,사시미포장(19p)",100000,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,스시포장(15p),65000,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,스페셜디너(사전예약시),190000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'Re52bbbf81c2f,스시무카,"서울 성동구 왕십리로 410, L동 1층 103호 (하왕십리동, 센트라스)",37.5661105,127.0276465,일식,VIP런치코스,95000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,누룽지,6000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,돼집껍데기,8000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,매운등갈비 매운등갈비,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,양념등갈비 양념등갈비,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,용궁라면,11000,표시 가격 확인,국수류,라면,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,잔치국수,7000,표시 가격 확인,국수류,국수,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,"최루탄주먹밥 7,000~",12000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R2d0057c44098,시간이 맛 있는집 청춘예찬,서울 성동구 무학로2길 36 (도선동),37.5631453,127.0335028,돼지고기 구이/찜,홍합탕,16000,표시 가격 확인,국물요리,국·탕,단품,식사,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,16:00-23:55,휴무\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,가쯔돈우동세트,12000,표시 가격 확인,국수류,우동,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,규동우동세트,12500,표시 가격 확인,밥·죽류,덮밥,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,김치가쯔우동세트,12500,표시 가격 확인,곁들임,반찬·무침,세트,사이드,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,나가사끼 짬뽕,13000,표시 가격 확인,국수류,짬뽕,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,냉모밀정식,18000,표시 가격 확인,국수류,모밀·소바,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,돈까스 우동정식,15000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,돈까스카레덮밥세트,12000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,돈코츠라멘,12000,표시 가격 확인,국수류,일본식 라멘,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,비빔모밀,12000,표시 가격 확인,국수류,모밀·소바,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,신기소 정식,18000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,어린이돈까스세트,10000,표시 가격 확인,튀김·커틀릿,돈가스,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,치킨마요 덮밥우동세트,12000,표시 가격 확인,튀김·커틀릿,치킨,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,함박까스 우동세트,16000,표시 가격 확인,튀김·커틀릿,함박스테이크,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R6c51bb7de756,신기소플러스,"서울 성동구 왕십리광장로 17, 왕십리민자역사 4층 (행당동)",37.5612890,127.0383818,일식,회덮밥 우동세트,12000,표시 가격 확인,밥·죽류,덮밥,세트,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R67aa54b8d4a1,신라방마라탕 장안동맛집,서울 동대문구 천호대로79길 66 (장안동),37.5649880,127.0645309,,마라상궈,16000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R67aa54b8d4a1,신라방마라탕 장안동맛집,서울 동대문구 천호대로79길 66 (장안동),37.5649880,127.0645309,,마라탕,7000,표시 가격 확인,국물요리,마라탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R3fed28b04f05,신설보쌈앤칼국수,서울 동대문구 하정로3길 10 (신설동) 1층,37.5755990,127.0259803,한식,바지락칼국수,8000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R3fed28b04f05,신설보쌈앤칼국수,서울 동대문구 하정로3길 10 (신설동) 1층,37.5755990,127.0259803,한식,보쌈정식,10000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,계란 후라이,3000,표시 가격 확인,곁들임,곁들임,단품,사이드,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,계란찜,7000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,냉면(후식),5000,표시 가격 확인,국수류,냉면,단품,식사,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,대인,8000,표시 가격 확인,미확인,미확인,단품,미확인,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,돼지 고기 고추장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,된장찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,버섯구이,5000,표시 가격 확인,미확인,미확인,단품,미확인,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,소인(10세 이하),4000,표시 가격 확인,미확인,미확인,단품,미확인,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7c4926b371f8,신숯불탁탁 마장점,서울 성동구 고산자로24길 12-1 3층,37.5697868,127.0389084,,육회무침비,5000,표시 가격 확인,초밥·회,회·사시미,단품,식사,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10,10:30-22:10\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,단호박 타르트,8700,표시 가격 확인,디저트,구움과자,단품,디저트,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,디카페인 고구마 티라미수,8700,표시 가격 확인,디저트,케이크,단품,디저트,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,쑥 인절미 갸또,8500,표시 가격 확인,디저트,케이크,단품,디저트,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,아이스 아메리카노,5500,표시 가격 확인,음료,커피,단품,음료,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,인절미 브라우니,6000,표시 가격 확인,디저트,케이크,단품,디저트,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,초코라떼,6500,표시 가격 확인,음료,라테,단품,음료,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R7166bfa2061d,심재응봉,"서울 성동구 고산자로2길 4-1, 1층 1호 (응봉동)",37.5519623,127.0357251,기타,카페라떼,6000,표시 가격 확인,음료,커피,단품,음료,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,국물,200,표시 가격 확인,미확인,미확인,단품,미확인,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,국물떡볶이,7000,표시 가격 확인,분식,떡볶이,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,냉모밀,9000,표시 가격 확인,국수류,모밀·소바,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,돈까스김밥,6000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,돈까스냉모밀,11000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,돈까스물냉면,10500,표시 가격 확인,국수류,냉면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,돈까스비빔냉면,10500,표시 가격 확인,국수류,냉면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,멸치고추김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,묵은지참치김밥,6000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,물냉면,8500,표시 가격 확인,국수류,냉면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,비빔냉면,8500,표시 가격 확인,국수류,냉면,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,비빔쫄우동,8500,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,샐러드김밥,5000,표시 가격 확인,샐러드,샐러드,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,소고기김밥,6000,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,소세지계란김밥,6000,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,순대만,4500,표시 가격 확인,분식,순대,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,순두부짬뽕우동,9000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,싸다김밥,4500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,어묵우동,8000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,어묵탕,7500,표시 가격 확인,국물요리,국·탕,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,얼큰장우동,9000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,제육김밥,6000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,즉석우동,7000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,참치김밥,5500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,치즈김밥,5000,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,치즈떡볶이,8500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,킹스팸김밥,6000,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,통새우김밥,6000,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,통통계란김밥,6500,표시 가격 확인,분식,김밥,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf0c6ba15d049,싸다김밥 왕십리역점,"서울 성동구 왕십리로 320, 1층 102,103호 (도선동)",37.5623438,127.0343698,분식,해물볶음우동,9000,표시 가격 확인,국수류,우동,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rfd9888f38c74,아담만두 더 랩,왕십리광장로 10,37.5610370,127.0363767,분식,,7000,대표가 추정(메뉴 미등록),,,,,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,고구마약과아이스,8900,표시 가격 확인,미확인,미확인,단품,미확인,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,고추육회불빔면,18900,표시 가격 확인,초밥·회,회·사시미,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,깍두기빙수,8900,표시 가격 확인,곁들임,반찬·무침,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,꽈리고추닭목살볶음,11900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,누룽지아귀간육회,22900,표시 가격 확인,초밥·회,회·사시미,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,닭고기육전,10900,표시 가격 확인,곁들임,전·부침,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,닭목살유린기,13900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,닭발기름떡볶이,14900,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,들깨닭전골(한정판매),23900,표시 가격 확인,국물요리,전골,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,뚝배기치즈불닭,23900,표시 가격 확인,미확인,미확인,단품,미확인,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,묵은지볶음밥,6900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,복분자수박하이볼,6900,표시 가격 확인,음료,하이볼,단품,음료,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,뽀글치즈감자전,18900,표시 가격 확인,곁들임,전·부침,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,산나물닭냉채,9900,표시 가격 확인,곁들임,반찬·무침,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,소곱창순두부전골,23900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,알곤이짬뽕탕,21900,표시 가격 확인,국물요리,국·탕,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,야관문하이볼,5900,표시 가격 확인,음료,하이볼,단품,음료,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,양파닭,14900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,얼큰해장닭전골,23900,표시 가격 확인,국물요리,전골,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,오미자담금주하이볼,5900,표시 가격 확인,음료,하이볼,단품,음료,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,오미자담금주하이볼병(4잔분량),19900,표시 가격 확인,음료,하이볼,단품,음료,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,원조닭전골,22900,표시 가격 확인,국물요리,전골,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,진흙오리전병(한정판매),19900,표시 가격 확인,곁들임,전·부침,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,통돼지후라이드,18900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,통통떡마늘똥집,11900,표시 가격 확인,미확인,미확인,단품,미확인,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,포슬포슬감자떡,12900,표시 가격 확인,미확인,미확인,단품,미확인,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,흑마늘닭튀김,13900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Redd29aeb9630,아리계곡 왕십리 한양대점,"서울 성동구 마조로3가길 14, 1~2층 (행당동)",37.5591337,127.0401208,호프/통닭,흑임자오이무침,7900,표시 가격 확인,곁들임,반찬·무침,단품,사이드,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 02:00,16:30-다음 날 03:00,16:30-다음 날 03:00,16:30-다음 날 02:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,김밥,4500,표시 가격 확인,분식,김밥,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,떡볶이,5500,추정가,분식,떡볶이,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,떡볶이튀김범벅,11000,추정가,분식,떡볶이,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,라면,4000,표시 가격 확인,국수류,라면,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,모듬튀김,5000,표시 가격 확인,튀김·커틀릿,튀김,모둠,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,모듬튀김(5개),6000,표시 가격 확인,튀김·커틀릿,튀김,모둠,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,물떡,3000,표시 가격 확인,분식,어묵,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,새우튀김,5000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,순대,5500,추정가,분식,순대,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,오뎅,4000,표시 가격 확인,분식,어묵,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,우동,4000,표시 가격 확인,국수류,우동,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,쫄면,7000,표시 가격 확인,국수류,쫄면,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Red005d7c5405,악어떡볶이,"서울 성동구 마조로 33 (행당동, 행당동 3-27 지상1층)",37.5606346,127.0410889,분식,튀김양념,5500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Re133c32677ed,알촌 한양대점,서울 성동구 마조로3길 6 (행당동),37.5585779,127.0401175,한식,그릴드치킨알밥(약간매운맛),7300,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'Re133c32677ed,알촌 한양대점,서울 성동구 마조로3길 6 (행당동),37.5585779,127.0401175,한식,순한알밥,6400,추정가,밥·죽류,알밥·영양밥,단품,식사,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'Re133c32677ed,알촌 한양대점,서울 성동구 마조로3길 6 (행당동),37.5585779,127.0401175,한식,약매알밥,6700,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'R8bc123f8f675,암암리,"서울 성동구 서울숲2길 13, 1층 (성수동1가)",37.5473070,127.0401986,한식,매운 고추 찐 만두 (포장가능),15000,표시 가격 확인,분식,만두,단품,사이드,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,휴무\r\n'
    'R8bc123f8f675,암암리,"서울 성동구 서울숲2길 13, 1층 (성수동1가)",37.5473070,127.0401986,한식,암암리 고추 양념장 (온라인구매가능),10000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,휴무\r\n'
    'R8bc123f8f675,암암리,"서울 성동구 서울숲2길 13, 1층 (성수동1가)",37.5473070,127.0401986,한식,암암리 매운 고추 만둣국,15000,표시 가격 확인,국물요리,떡국·만둣국,단품,식사,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,11:30-14:30,휴무\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,가마타마 우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,가케우동,7500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,냉 우동,7500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,붓가케우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,쇼유 우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,앙카케 타마고토지우동,9500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,와카메우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,자루우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,츠키미우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'R40f1cc458af1,야마타니우동,서울 성동구 왕십리로21길 11 1층,37.5613084,127.0331460,일식 면 요리,카레우동,8500,표시 가격 확인,국수류,우동,단품,식사,휴무,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-19:45,11:40-15:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,(급냉)삼겹살 문경약돌돼지,14800,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,(생) 삼겹살 문경약돌돼지,15800,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,껍데기,8800,표시 가격 확인,미확인,미확인,단품,미확인,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,마약 야채 돼지찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,마약 야채 돼지찌개 전골(2~3인분),25000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,물냉! 비냉!,7000,표시 가격 확인,국수류,냉면,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,불판 짜파게티,5000,표시 가격 확인,국수류,라면,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,차돌박이,15800,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,특목살 문경약돌돼지,15800,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Raff7c675a8b7,양평회관,"서울 성동구 마조로7길 16, 1,2층 (행당동)",37.5609025,127.0395686,기타,항정살 문경약돌돼지,17800,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00\r\n'
    'Rff27b5671e7e,어수정부대찌개,서울 동대문구 무학로34길 4-1,37.5762310,127.0309513,백반/한정식,돼지머리고기 대,15000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rff27b5671e7e,어수정부대찌개,서울 동대문구 무학로34길 4-1,37.5762310,127.0309513,백반/한정식,돼지머리고기 소,10000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rff27b5671e7e,어수정부대찌개,서울 동대문구 무학로34길 4-1,37.5762310,127.0309513,백반/한정식,부대찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rff27b5671e7e,어수정부대찌개,서울 동대문구 무학로34길 4-1,37.5762310,127.0309513,백반/한정식,부대찌개 포장,6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'Rff27b5671e7e,어수정부대찌개,서울 동대문구 무학로34길 4-1,37.5762310,127.0309513,백반/한정식,순대국,6000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R0c325566f5ab,엄마김치찌개,서울 성동구 용답중앙15길 1 (용답동) 1층,37.5639311,127.0528466,한식,문어숙회,20000,표시 가격 확인,초밥·회,회·사시미,단품,식사,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00,08:00-22:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,새우버거,10500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,스위트포테이토 프라이즈,7000,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,A.D.B burger,9900,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,Bacon burger,9500,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,Cheese fries,6500,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,Classic burger,7000,표시 가격 확인,피자·버거·샌드위치,버거,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,French fries,4500,표시 가격 확인,튀김·커틀릿,감자튀김,단품,사이드,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1c70df02d9b,에프터드링크버거,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,기타,Soda,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rc1a45a0004f0,엘닷,서울 성동구 성수일로12길 20 (성수동2가) 1층 103호,37.5482346,127.0533021,일식 회/초밥,모듬초밥(10pcs),8900,표시 가격 확인,초밥·회,초밥,모둠,식사,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,휴무,휴무\r\n'
    'Rc1a45a0004f0,엘닷,서울 성동구 성수일로12길 20 (성수동2가) 1층 103호,37.5482346,127.0533021,일식 회/초밥,모듬초밥(8pcs),6900,표시 가격 확인,초밥·회,초밥,모둠,식사,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,11:00-15:00,휴무,휴무\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,가라아게 카레,13000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,가츠덮밥,12000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,경양식 돈가츠,13000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,규카레,13000,표시 가격 확인,커리·스튜,커리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,김치돈가츠나베,14000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,돈가츠 카레,13000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,리코타치즈샐러드,12000,표시 가격 확인,샐러드,샐러드,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,명란바질오일파스타,17000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,믹스 덮밥,14000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,부채살크림파스타,18000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,부타덮밥,13000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,새우튀김덮밥,13000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,생연어덮밥,16000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,소고기치즈가츠정식,16000,표시 가격 확인,구이·볶음·찜,소고기구이,세트,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,안심가츠정식,14000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,알쌈리조또,17000,표시 가격 확인,밥·죽류,리조또,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,야끼니쿠동,14000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,차돌중화크림파스타,18000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rb08a67ee312a,연남토마 성수점,"서울 성동구 서울숲4길 16-20, 1,2층 (성수동1가)",37.5470056,127.0423615,기타,치즈가츠정식,15000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'Rfdc47c5b678c,연평도 목포 산꽃게,"서울 성동구 마장로 323 (마장동,(지상1,2층))",37.5676997,127.0448590,한식,간장게장 少,65000,표시 가격 확인,미확인,미확인,단품,미확인,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rfdc47c5b678c,연평도 목포 산꽃게,"서울 성동구 마장로 323 (마장동,(지상1,2층))",37.5676997,127.0448590,한식,꽃게백숙 中,100000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rfdc47c5b678c,연평도 목포 산꽃게,"서울 성동구 마장로 323 (마장동,(지상1,2층))",37.5676997,127.0448590,한식,꽃게양념찜 中,100000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rfdc47c5b678c,연평도 목포 산꽃게,"서울 성동구 마장로 323 (마장동,(지상1,2층))",37.5676997,127.0448590,한식,꽃게탕 中,100000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Re14b8598aec9,옛고을닭한마리,서울 동대문구 사가정로 7 (답십리동) 1층,37.5706001,127.0490140,닭/오리고기 구이/찜,닭곰탕,9000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Re14b8598aec9,옛고을닭한마리,서울 동대문구 사가정로 7 (답십리동) 1층,37.5706001,127.0490140,닭/오리고기 구이/찜,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,감자탕 대,45000,표시 가격 확인,국물요리,감자탕,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,감자탕 소,35000,표시 가격 확인,국물요리,감자탕,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,감자탕 중,40000,표시 가격 확인,국물요리,감자탕,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,감자탕볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,내장탕,10000,표시 가격 확인,국물요리,곱창탕,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,당면,5000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,라면,2000,표시 가격 확인,국수류,라면,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,막걸리,4000,표시 가격 확인,음료,막걸리,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,맥주,5000,표시 가격 확인,음료,맥주,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,뼈 추가,25000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,뼈해장국,10000,표시 가격 확인,국물요리,해장국,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,소주,5000,표시 가격 확인,음료,소주,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,수제비,5000,표시 가격 확인,국수류,수제비,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,우거지·야채,5000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,육개장,10000,표시 가격 확인,국물요리,육개장,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,음료,2000,표시 가격 확인,음료,기타 음료,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,청하,7000,표시 가격 확인,음료,전통주,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,치즈추가,3000,표시 가격 확인,곁들임,치즈,추가,추가옵션,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rdeeda2b3fe70,옛골감자탕,"서울 성동구 아차산로7길 18, 1층 (성수동2가)",37.5466853,127.0548976,한식,칼국수,5000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R47e60c3985db,오늘보쌈칼국수,"서울 성동구 행당로17길 24-1, 1층 (행당동)",37.5605494,127.0331509,식육(숯불구이),가브리살 보쌈(대),53000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R47e60c3985db,오늘보쌈칼국수,"서울 성동구 행당로17길 24-1, 1층 (행당동)",37.5605494,127.0331509,식육(숯불구이),가브리살 보쌈(소),39000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R47e60c3985db,오늘보쌈칼국수,"서울 성동구 행당로17길 24-1, 1층 (행당동)",37.5605494,127.0331509,식육(숯불구이),가브리살 보쌈(중),46000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R47e60c3985db,오늘보쌈칼국수,"서울 성동구 행당로17길 24-1, 1층 (행당동)",37.5605494,127.0331509,식육(숯불구이),일품☆통문어 보쌈,68000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R47e60c3985db,오늘보쌈칼국수,"서울 성동구 행당로17길 24-1, 1층 (행당동)",37.5605494,127.0331509,식육(숯불구이),칼국수 정식,10000,표시 가격 확인,국수류,칼국수,세트,식사,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00,11:20-22:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,개불,20000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,계절모듬 대,70000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,계절모듬 소,45000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,계절모듬 중,55000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,광어 대,40000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,광어 중,30000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,광어/우럭 대,60000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,광어/우럭 중,45000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,광어/우럭/연어,50000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,농어,60000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,농어 2/1,30000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,대방어 대,90000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,대방어 중,70000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,도다리세꼬시,35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,도다리회,40000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,도미,60000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,도미 2/1,30000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,멍게,15000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,산낙지,25000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,아나고 세꼬시,35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,연어 대,50000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,연어 소,30000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,연어 중,40000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,우럭 소,40000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,우럭 중,60000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,전복 3미,20000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,특대광어 1.5키로이상,80000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6e61383cd96a,오늘회식,"서울 성동구 용답중앙길 72, 1층 (용답동)",37.5631691,127.0534451,한식,해삼,25000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,매콤 돈카츠,14000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,부산어묵우동,10500,표시 가격 확인,국수류,우동,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,비빔 돈카츠,13500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,수제 생돈카츠,11500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,스페셜A세트,18500,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,스페셜B세트,15500,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,에비카츠우동,10500,표시 가격 확인,국수류,우동,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,자가제면 냉소바,10000,표시 가격 확인,국수류,모밀·소바,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,자가제면 비빔소바,10000,표시 가격 확인,국수류,모밀·소바,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,체다크림치즈 돈카츠,14000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R026b9b9fab1f,오레노카츠,"서울 성동구 성수일로 47, 1층 102호 (성수동1가)",37.5447538,127.0495208,중국식,카레 돈카츠,13500,표시 가격 확인,커리·스튜,커리,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,돔베고기(도마),27000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,돔베고기(반도마),15000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,보말죽,12000,표시 가격 확인,밥·죽류,죽,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,보말칼국수,12000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,얼큰고기국수,9000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,전복죽,12000,표시 가격 확인,밥·죽류,죽,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,제주도 고기국수,9000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,제주도 비빔국수,9000,표시 가격 확인,국수류,국수,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra5ab156c4bf3,오소록,"서울 성동구 왕십리로 294-2, 대도빌딩 1층 (행당동)",37.5604654,127.0363186,분식,창도름순대,14000,표시 가격 확인,분식,순대,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,드립 커피,4500,표시 가격 확인,음료,커피,단품,음료,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,루꼴라 에 모르타델라,23000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,마르게리타,19000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,무화과 고르곤졸라,20500,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,살시챠 에 살라미,22000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,살시챠 에 풍기,23800,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,카프레제 부라타 샐러드,19500,표시 가격 확인,샐러드,샐러드,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,트러플 버섯 뇨끼,19500,표시 가격 확인,파스타류,뇨끼,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,포르치니 크레마,21500,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R9f9174cbaebd,올리몰리,"서울 성동구 고산자로8길 20, 1층 (행당동)",37.5579647,127.0360264,경양식,화덕빵,7500,표시 가격 확인,곁들임,빵,단품,사이드,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R2b9b3fb48e41,옵션2(OPTION2),"서울 성동구 마조로 11, 2층 (행당동)",37.5590101,127.0406716,기타,베이컨 토마토 도리아,9500,표시 가격 확인,밥·죽류,리조또,단품,식사,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30,10:00-21:30\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,닭내장탕 대,40000,표시 가격 확인,국물요리,곱창탕,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,닭내장탕 소,25000,표시 가격 확인,국물요리,곱창탕,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,닭내장탕 중,30000,표시 가격 확인,국물요리,곱창탕,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,막걸리,4000,표시 가격 확인,음료,막걸리,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,맥주,5000,표시 가격 확인,음료,맥주,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,소주,5000,표시 가격 확인,음료,소주,단품,음료,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R9bda9cda3529,왕십리 닭내장탕집,서울 성동구 무학로2길 11,37.5643182,127.0312881,국/탕/찌개류,오징어무침,20000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,(복사본)국내산 장어 기본 덮밥 한마 국내산 장어 한마리를 숯불에 구운 덮밥입니다.,19900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,간장양념구이 숯불에 구운 국내산 장어와 특제소스의 장어구이.,19900,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,국내산 장어 기본 덮밥 한마리 국내산 장어 한마리를 숯불에 구운 덮밥입니다.,19900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,닭가슴살,4300,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,닭날개,3600,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,닭다리살,4300,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,닭똥집,2100,표시 가격 확인,미확인,미확인,단품,미확인,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,소금장어구이,19900,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,야키토리 4종세트,11900,표시 가격 확인,구이·볶음·찜,꼬치구이,세트,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,염통,2100,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,장어덮밥,19900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,혼술 기린 맥주세트,16900,표시 가격 확인,음료,맥주,단품,음료,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R5f939335c001,왕십리 장어덮밥 야키토리,"서울 성동구 왕십리로24길 13, 1층 (도선동)",37.5636423,127.0340988,일식,혼술 사케(도쿠리) 세트,16900,표시 가격 확인,음료,사케,단품,음료,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,골뱅이무침,25000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,구룡포과메기,30000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,꼬막,15000,표시 가격 확인,미확인,미확인,단품,미확인,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,문어숙회,28000,표시 가격 확인,초밥·회,회·사시미,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,불쭈꾸미 (대),32000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,불쭈꾸미 (중),22000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,오뎅꼬치 3개 [골뱅이 추가메뉴],2000,표시 가격 확인,분식,어묵,단품,사이드,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,왕새우구이,15000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,칼국수 [골뱅이 드신 후],5000,표시 가격 확인,국수류,칼국수,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,통골뱅이 (대),49000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,통골뱅이 (중),37000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R1aa88a5155d6,왕십리 통골뱅이,"서울 성동구 왕십리로24길 7-1, 지상1층 (도선동)",37.5632296,127.0338607,한식,통닭발,22000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 01:00,15:00-다음 날 02:00,13:00-다음 날 02:00,15:00-다음 날 01:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,고추장삼겹살,7222,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,고추장삼겹오징어,7222,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,관자+표고+명란,15000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,급냉꽃삼겹살(국내산 암돼지),8333,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,냉동박사 한상 대(4인),95000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,냉동박사 한상 소(2인),55000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,냉동박사 한상 중(3인),75000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,된장찌개누룽지술밥,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,듀록 흑돼지대패삼겹살,5500,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,물냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,버섯채소모듬,5000,표시 가격 확인,미확인,미확인,모둠,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,오삼부대로스,33000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,왕십리부대찌개,11000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,우삼겹,6000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,주원산냉동오리,6500,표시 가격 확인,미확인,미확인,단품,미확인,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,진꽃생삼겹살(국내산 암돼지),8888,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,청어알비빔밥,6000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,특수한 부대찌개,13000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'R6d3d99af2e81,왕십리냉박사,왕십리로21길 17 1층,37.5608930,127.0328453,백반/한정식,회비빔냉면,10000,표시 가격 확인,국수류,냉면,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'Rc1eeec3b4ac0,왕십리사이공,"서울 성동구 고산자로12길 2, 1층 (행당동)",37.5602094,127.0347491,한식,매운쌀국수,11000,표시 가격 확인,국수류,쌀국수,단품,식사,휴무,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'Rc1eeec3b4ac0,왕십리사이공,"서울 성동구 고산자로12길 2, 1층 (행당동)",37.5602094,127.0347491,한식,분짜,16000,표시 가격 확인,국수류,국수,단품,식사,휴무,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'Rc1eeec3b4ac0,왕십리사이공,"서울 성동구 고산자로12길 2, 1층 (행당동)",37.5602094,127.0347491,한식,퍼 가,10500,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,감자 한접시,5000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,버섯모듬,10000,표시 가격 확인,미확인,미확인,모둠,미확인,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,버섯모듬 3-4가지버섯제공(상시종류변경),10000,표시 가격 확인,미확인,미확인,모둠,미확인,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,부추(한접시),3000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,생오리 로스구이,49000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,양념주물럭,50000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,영양볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,오리 반마리,35000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,"오리 반마리 로스,양념,훈제 동일가격",35000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,"오리, 토종닭 한방백숙(능이버섯)",78000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,"오리, 토종닭 한방백숙(능이버섯)1시간 전 예약 필수",78000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,"오리, 토종닭 한방백숙(버섯천국)",78000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,"오리, 토종닭 한방백숙(버섯천국)1시간 전 예약 필수",78000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,오리뚝배기해장국(점심식사),10000,표시 가격 확인,국물요리,해장국,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,오리뚝배기해장국(점심식사)평일 오후 3시까지 주문가능 (주말/공휴일 주문불가),10000,표시 가격 확인,국물요리,해장국,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,오리반반 한마리,53000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,오리훈제,55000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,통마늘(한접시),3000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,후식 열무냉면/계절,7000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rfdac94eb9350,왕십리오리사냥,서울 성동구 행당로17길 7,37.5590638,127.0334217,닭/오리고기 구이/찜,후식 열무냉면/계절 여름 계절 메뉴,7000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30,11:30-21:30\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,2인세트 510g,49000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,3-4인 프리미엄세트 690g,63000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,꽃게된장찌개 (공기밥1개포함),6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,냉면(물/비빔/불닭),6000,표시 가격 확인,국수류,냉면,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,뚝배기계란찜,4000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,마무리볶음밥(계란후라이포함),3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,맥주,5000,표시 가격 확인,음료,맥주,단품,음료,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,소주,4000,표시 가격 확인,음료,소주,단품,음료,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,이베리코 흑돼지 목살 180g,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,통가브리살 150g,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,통목살 180g,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,통삼겹살 180g,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,통항정살 150g,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,평일점심)김치찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,평일점심)돼지김치찜,8000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rc9190a0105d6,왕십리육감식당,"서울 성동구 무학봉28길 1, 1층 (하왕십리동)",37.5625411,127.0323735,한식,평일점심)제육볶음,7000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:40-23:00,11:40-23:00\r\n'
    'Rfc7c22a5ebec,왕십리한고기,왕십리로20길 8 1층,37.5624781,127.0350531,돼지고기 구이/찜,김치 짜글이,6000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rfc7c22a5ebec,왕십리한고기,왕십리로20길 8 1층,37.5624781,127.0350531,돼지고기 구이/찜,한우등심1++(180g),38000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rfc7c22a5ebec,왕십리한고기,왕십리로20길 8 1층,37.5624781,127.0350531,돼지고기 구이/찜,한우모듬갈비살 1++(180g),36000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rfc7c22a5ebec,왕십리한고기,왕십리로20길 8 1층,37.5624781,127.0350531,돼지고기 구이/찜,한우육회(200g),20000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rfc7c22a5ebec,왕십리한고기,왕십리로20길 8 1층,37.5624781,127.0350531,돼지고기 구이/찜,한우차돌박이 1++(180g),28000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R1ce9cab3531b,용두동 복집,서울 동대문구 왕산로16길 9 1층,37.5774607,127.0313376,,복매운탕,27000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R1ce9cab3531b,용두동 복집,서울 동대문구 왕산로16길 9 1층,37.5774607,127.0313376,,복불고기,29000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R1ce9cab3531b,용두동 복집,서울 동대문구 왕산로16길 9 1층,37.5774607,127.0313376,,복사시미 1kg,180000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R1ce9cab3531b,용두동 복집,서울 동대문구 왕산로16길 9 1층,37.5774607,127.0313376,,복지리,27000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'R1ce9cab3531b,용두동 복집,서울 동대문구 왕산로16길 9 1층,37.5774607,127.0313376,,복튀김,35000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,갈치조림(1인),16000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,김치찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,낙지볶음(1인),14000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,동태탕(1인),10000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,동태해물탕(대),52000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,동태해물탕(중),42000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,생대구탕(1인),14000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,소머리국밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,소머리수육,48000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,순대국,9000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,쌈밥정식(1인),10000,표시 가격 확인,밥·죽류,쌈밥,세트,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,오징어볶음(1인),14000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'Re5c970c0185e,용두똥돼지,서울 동대문구 무학로 114 1층,37.5743634,127.0308702,돼지고기 구이/찜,청국장,9000,표시 가격 확인,국물요리,찌개류,단품,식사,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00,10:00-24:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,가마타마,9000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,가조쿠우동,9000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,라유멘,10000,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,면추가,2500,표시 가격 확인,국수류,면,추가,추가옵션,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,붓가케우동,7500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,새우튀김,2500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,육육우동,9500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,치즈육육우동,11500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,치즈카레우동,12000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,카레우동,10000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R33418cfe24c6,우동 가조쿠 본점,서울 성동구 왕십리로 215-1 1층,37.5567679,127.0429100,일식 면 요리,키츠네우동,8000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6594e5ef7f29,우동가조쿠,"서울 성동구 왕십리로 215-1 (행당동,지상2층)",37.5567679,127.0429100,한식,가조쿠우동,9000,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6594e5ef7f29,우동가조쿠,"서울 성동구 왕십리로 215-1 (행당동,지상2층)",37.5567679,127.0429100,한식,붓가케우동,7500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6594e5ef7f29,우동가조쿠,"서울 성동구 왕십리로 215-1 (행당동,지상2층)",37.5567679,127.0429100,한식,육육우동,9500,표시 가격 확인,국수류,우동,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R71d942b546a3,우리한식부페식당,서울 동대문구 장한로5길 41 (장안동) 1층,37.5650247,127.0641322,뷔페,한식뷔페(점심),7000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,11:00-14:30,휴무,휴무\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,고기만두(4개),6000,표시 가격 확인,분식,만두,단품,사이드,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,냉제육 미나리무침,12000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,소주 잔술(100ml),2000,표시 가격 확인,음료,소주,단품,음료,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,유태 얼큰국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,유태 얼큰국밥(특),13000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,유태국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,유태국밥(특),13000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R99741a83aa30,유태,"서울 성동구 마조로9길 3, 1층 (행당동)",37.5611301,127.0409703,일식,한산소곡주 잔술(100ml),3000,표시 가격 확인,음료,전통주,단품,음료,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:00,11:00-20:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,1+ 녹돈모둠세트(5인분),94000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,1+ 녹돈세트(3인분),58000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,1++ 홍두깨 한우육회(120g),17000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,1++ 홍두깨 한우육회(200g),28000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,168시간숙성 1+ 녹돈꽃목살,18500,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,168시간숙성 1+ 녹돈삼겹살,18500,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,72시간 과일숙성양념돼지갈비(250g),20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,녹돈 1+ 가브리살,18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,먹물함흥물냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,먹물함흥물냉면(곱),11000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,먹물함흥비빔냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,먹물함흥비빔냉면(곱),11000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,모듬야채구이,4000,표시 가격 확인,미확인,미확인,모둠,미확인,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,배추시래기 한우술국,15000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,양념벌집돼지껍데기,10000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,저녁 녹돈 김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,저녁 녹돈 된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 1++ 한우국밥,13000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 1++ 홍두깨육회비빔밥,13000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 김치찌개+제육 세트(2인이상),14000,표시 가격 확인,국물요리,찌개류,세트,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 녹돈 김치찌개,10000,표시 가격 확인,국물요리,찌개류,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 녹돈 된장찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 도토리묵밥,8000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 먹물함흥물냉면,11000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 먹물함흥물냉면 곱빼기,15000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 먹물함흥비빔냉면,11000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 먹물함흥비빔냉면 곱빼기,15000,표시 가격 확인,국수류,냉면,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,점심 불향제육볶음,12000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,천포생갈비(200g),23000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R750a2c7d8404,육다정수제갈비 왕십리로21가길 3 1층,왕십리로21가길 3 1층,37.5609920,127.0331509,돼지고기 구이/찜,폭탄 계란찜,6000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,깻잎전,16000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,모둠전,30000,표시 가격 확인,곁들임,전·부침,모둠,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,물막국수정식,16500,표시 가격 확인,국수류,막국수,세트,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,미소곰탕,11000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,미소곰탕정식,17000,표시 가격 확인,국물요리,곰탕·설렁탕,세트,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,비빔막국수정식,16500,표시 가격 확인,국수류,막국수,세트,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,생굴파전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소고기국밥,10000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소고기국밥정식,16500,표시 가격 확인,구이·볶음·찜,소고기구이,세트,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소고기육전(대),26000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소고기육전(소),16000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소곱창전골(대),55000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소곱창전골(중),39000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소불고기전골(2인),30000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소불고기전골(3인),45000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,소불고기전골(4인),60000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,수제 소고기완자,16000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,오징어초무침,20000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,육전 완자 반반,22000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,육전국밥,11000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,육전모둠전골(대),55000,표시 가격 확인,곁들임,전·부침,모둠,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,육전모둠전골(중),39000,표시 가격 확인,곁들임,전·부침,모둠,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,육전물막국수,11000,표시 가격 확인,국수류,막국수,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,육전비빔막국수,11000,표시 가격 확인,국수류,막국수,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,차돌곱창볶음(3~4인분),45000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,치즈해물김치전,22000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,치즈해물파전,22000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,함박곰탕,14000,표시 가격 확인,튀김·커틀릿,함박스테이크,단품,식사,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,해물김치전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'R8993599611d0,육전국밥 성수역점,"서울 성동구 아차산로 88 (성수동2가,(지상1층))",37.5449888,127.0537831,일식,해물파전,20000,표시 가격 확인,곁들임,전·부침,단품,사이드,08:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-22:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,떡갈비,5000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,사이다,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,최상급 한우1++ 육사시미(160g),24000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,최상급 한우1++ 육회(150g),23000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,콜라,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,통문어숙회,24000,표시 가격 확인,초밥·회,회·사시미,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,한우육회+사시미 세트,39000,표시 가격 확인,구이·볶음·찜,소고기구이,세트,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rd25539578fb8,육회한인생,무학봉16길 28,37.5596404,127.0335125,돼지고기 구이/찜,한우육회+통문어숙회 세트,39000,표시 가격 확인,구이·볶음·찜,소고기구이,세트,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,감자탕 2人 감자탕,30000,표시 가격 확인,국물요리,감자탕,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,감자탕 3人,40000,표시 가격 확인,국물요리,감자탕,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,감자탕 4人,50000,표시 가격 확인,국물요리,감자탕,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,김치전골 3人,32000,표시 가격 확인,곁들임,전·부침,단품,사이드,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,김치전골 4人,40000,표시 가격 확인,곁들임,전·부침,단품,사이드,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,김치전골 소,25000,표시 가격 확인,곁들임,전·부침,단품,사이드,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,닭도리탕 2人 닭도리탕,30000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,닭도리탕 3人,40000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,닭도리탕 4人,50000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,닭한마리 2人 닭한마리,30000,표시 가격 확인,국물요리,닭한마리,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,닭한마리 3人,40000,표시 가격 확인,국물요리,닭한마리,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,닭한마리 4人,51000,표시 가격 확인,국물요리,닭한마리,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,뼈해장국,10000,표시 가격 확인,국물요리,해장국,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,삼겹살,16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,오리로스,30000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,옻닭 에약,70000,표시 가격 확인,구이·볶음·찜,삼계탕·백숙,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,옻오리 예약,70000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,한방백숙 예약,70000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'R9bdafcf2bd39,은좌식당,서울 성동구 왕십리로 330 (도선동),37.5629291,127.0332763,백반/한정식,한방오리 예약,70000,표시 가격 확인,구이·볶음·찜,오리요리,단품,식사,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00,16:00-다음 날 04:00\r\n'
    'Raf0d7c013ea6,응봉참숯생소금구이,"서울 성동구 독서당로62길 4 (응봉동,(지상2층))",37.5496683,127.0311180,한식,돼지갈비 (180g),18000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Raf0d7c013ea6,응봉참숯생소금구이,"서울 성동구 독서당로62길 4 (응봉동,(지상2층))",37.5496683,127.0311180,한식,물냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Raf0d7c013ea6,응봉참숯생소금구이,"서울 성동구 독서당로62길 4 (응봉동,(지상2층))",37.5496683,127.0311180,한식,생삼겹살 (150g),14000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Re5f670874917,이돈집,서울 성동구 마조로3길 7 (행당동),37.5584999,127.0399270,분식,근고기 김치찌개 2인,14000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Re5f670874917,이돈집,서울 성동구 마조로3길 7 (행당동),37.5584999,127.0399270,분식,라면사리,1000,표시 가격 확인,국수류,라면,추가,추가옵션,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Re5f670874917,이돈집,서울 성동구 마조로3길 7 (행당동),37.5584999,127.0399270,분식,반근세트,18000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Re5f670874917,이돈집,서울 성동구 마조로3길 7 (행당동),37.5584999,127.0399270,분식,직화제육,6000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,간사이 오뎅 나베,23000,표시 가격 확인,분식,어묵,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,감태에 싸서 먹는 명란구이.,15000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,게살 크림 고로케,16000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,계절 살얼음 물회,35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,"눈꽃치즈 스파이시 관자,새우",24000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,대방어 사시미,29000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,레몬마요 크림새우,21000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,메로구이,35000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,볼락 간장 튀김,35000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,사천식 나가사끼 짬뽕 나베,25000,표시 가격 확인,국수류,짬뽕,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,새콤달콤 칠리새우,23000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,세가지맛 석화 플래터,25000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,수제카츠 김치 나베 (극강의 시원함),25000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,시메사바,35000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,"연어회, 참치회, 낫또 샐러드.",25000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,오늘의 회무침(조기 품절 될수있음),25000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,오징어 다리 튀김과 쥐포,17000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,"우니, 단새우, 관자",38000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,우니(계절 한정),35000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,이로 5종 사시미,25000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,일본식 왕새우 덴푸라(10개),24000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,초당 옥수수 튀김,13000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,치킨 가라아게,20000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,크림치즈 모찌리도후,13000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,타래야끼 닭다리살 꼬치(6P),20000,표시 가격 확인,구이·볶음·찜,꼬치구이,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,통영 굴 미소나베(계절한정),25000,표시 가격 확인,국물요리,전골,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,통영산 굴 초회(계절한정),15000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,특대 고등어 구이,20000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,프리미엄 굴짬뽕(계절 한정),29000,표시 가격 확인,국수류,짬뽕,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,프리미엄 활 시메사바,39000,표시 가격 확인,초밥·회,회·사시미,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,해물 토마토 해장 파스타,28000,표시 가격 확인,파스타류,파스타,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Rcb92973c9685,이로 이자카야,"서울 성동구 왕십리로 284-1, 1층 (행당동)",37.5598935,127.0367761,일식,화이트 와인 조개찜,27000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00,17:00-다음 날 01:00\r\n'
    'Re6cfc9d1c6ed,이자와 왕십리엔터식스점,"서울 성동구 왕십리광장로 17, 4층 (행당동, 왕십리민자역사)",37.5612890,127.0383818,일식,규카츠,18000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Re6cfc9d1c6ed,이자와 왕십리엔터식스점,"서울 성동구 왕십리광장로 17, 4층 (행당동, 왕십리민자역사)",37.5612890,127.0383818,일식,마제소바,12000,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Re6cfc9d1c6ed,이자와 왕십리엔터식스점,"서울 성동구 왕십리광장로 17, 4층 (행당동, 왕십리민자역사)",37.5612890,127.0383818,일식,치킨난반,14000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R59da574c6acc,인더매스,"서울 성동구 마장로 270, 1층 (마장동)",37.5654005,127.0397761,기타,이탈리안 치아바타 샌드위치 + 커피세트,13500,표시 가격 확인,피자·버거·샌드위치,샌드위치,세트,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00\r\n'
    'R59da574c6acc,인더매스,"서울 성동구 마장로 270, 1층 (마장동)",37.5654005,127.0397761,기타,콜비잭 치즈 샌드위치 + 커피세트,14500,표시 가격 확인,피자·버거·샌드위치,샌드위치,세트,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00\r\n'
    'R59da574c6acc,인더매스,"서울 성동구 마장로 270, 1층 (마장동)",37.5654005,127.0397761,기타,토마토 스프 단품,5900,표시 가격 확인,국물요리,수프,단품,사이드,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00\r\n'
    'Ra9536fd1b331,인바이트,"서울 성동구 용답중앙27길 7 (용답동, METROCITY) 1층 101호",37.5632275,127.0547024,기타,오늘의커피,2500,표시 가격 확인,음료,커피,단품,음료,08:00-20:00,08:00-20:00,08:00-20:00,08:00-20:00,08:00-20:00,10:00-17:00,10:00-17:00\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,김치찌개(공기밥포함),8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,물냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,비빔냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우 육회,25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우갈비탕,19000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우고기국수,6000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우된장찌개(공기밥포함),8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우라면,6000,표시 가격 확인,국수류,라면,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우사시미,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R489f8bf51507,인생한우,"서울 성동구 마장로31길 43, 지상2층 (마장동, 522-2)",37.5701658,127.0382165,한식,한우육회비빔밥,12000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40,11:00-21:40\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,가지(고기+새우)튀김,45000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,갑오징어해물쟁반짜장,12000,표시 가격 확인,국수류,짜장면,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,고추유린기,40000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,굴짬뽕(겨울),15000,표시 가격 확인,국수류,짬뽕,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,동파육,55000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,삼선간짜장,12000,표시 가격 확인,국수류,짜장면,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,시그니처 어향동고,45000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,옛날볶음밥+후라이,12000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,옛날짬뽕(밥),13000,표시 가격 확인,국수류,짬뽕,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,원조 육즙탕수육,35000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,전가복 스페셜,90000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,팔보채,58000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,하얀짬뽕(밥),14000,표시 가격 확인,국수류,짬뽕,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R4bf2f864d7c0,일일향,"서울 성동구 왕십리로 345, 1층 (하왕십리동)",37.5632069,127.0315545,중국식,해물잡탕밥,18000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R193122c9e40f,일품백송칼국수,"서울 성동구 고산자로12길 1-1, 지상1층 (행당동)",37.5602314,127.0350535,한식,닭칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R193122c9e40f,일품백송칼국수,"서울 성동구 고산자로12길 1-1, 지상1층 (행당동)",37.5602314,127.0350535,한식,설렁탕,12000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R193122c9e40f,일품백송칼국수,"서울 성동구 고산자로12길 1-1, 지상1층 (행당동)",37.5602314,127.0350535,한식,손만두,10000,표시 가격 확인,분식,만두,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,돼지 머리고기,15000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,소머리국밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,소머리국밥 특,13000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,소수육,40000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,순대 + 머리고기,17000,표시 가격 확인,분식,순대,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,순대국,10000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,순대국 특,12000,표시 가격 확인,국물요리,국밥,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,순대전골,35000,표시 가격 확인,분식,순대,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R01e862281be1,장성식당,"서울 성동구 용답길 157-31, 1층 (용답동)",37.5630369,127.0548821,한식,야채순대,15000,표시 가격 확인,분식,순대,단품,식사,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,10:00-20:00,휴무,휴무\r\n'
    'R115550a47cb4,장수보리밥,서울 중구 퇴계로85길 12-4 (황학동),37.5662749,127.0198648,백반/한정식,고기쌈밥,12000,표시 가격 확인,밥·죽류,쌈밥,단품,식사,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,휴무\r\n'
    'R115550a47cb4,장수보리밥,서울 중구 퇴계로85길 12-4 (황학동),37.5662749,127.0198648,백반/한정식,보리밥,8000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,휴무\r\n'
    'R115550a47cb4,장수보리밥,서울 중구 퇴계로85길 12-4 (황학동),37.5662749,127.0198648,백반/한정식,야채쌈밥,9000,표시 가격 확인,밥·죽류,쌈밥,단품,식사,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,11:00-19:00,휴무\r\n'
    'R16608aeaa080,장안순대국밥,서울 동대문구 고미술로 81,37.5655980,127.0559350,,돼지국밥,9000,표시 가격 확인,국물요리,국밥,단품,식사,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-22:00,05:00-22:00\r\n'
    'R16608aeaa080,장안순대국밥,서울 동대문구 고미술로 81,37.5655980,127.0559350,,돼지김치찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-22:00,05:00-22:00\r\n'
    'R16608aeaa080,장안순대국밥,서울 동대문구 고미술로 81,37.5655980,127.0559350,,모둠정식,13000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-22:00,05:00-22:00\r\n'
    'R16608aeaa080,장안순대국밥,서울 동대문구 고미술로 81,37.5655980,127.0559350,,보쌈정식,13000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-22:00,05:00-22:00\r\n'
    'R16608aeaa080,장안순대국밥,서울 동대문구 고미술로 81,37.5655980,127.0559350,,사골우거지국밥,10000,표시 가격 확인,국물요리,국·탕,단품,식사,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-22:00,05:00-22:00\r\n'
    'R16608aeaa080,장안순대국밥,서울 동대문구 고미술로 81,37.5655980,127.0559350,,순대국밥,9000,표시 가격 확인,국물요리,국밥,단품,식사,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-23:00,05:00-22:00,05:00-22:00\r\n'
    'R278cbec1579c,장원닭한마리,서울 동대문구 전농로 36 답십리포레스트스퀘어 1층 107호,37.5678677,127.0574043,닭/오리고기 구이/찜,닭도리탕 (대),43000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,15:00-22:30\r\n'
    'R278cbec1579c,장원닭한마리,서울 동대문구 전농로 36 답십리포레스트스퀘어 1층 107호,37.5678677,127.0574043,닭/오리고기 구이/찜,닭도리탕 (중),33000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,15:00-22:30\r\n'
    'R278cbec1579c,장원닭한마리,서울 동대문구 전농로 36 답십리포레스트스퀘어 1층 107호,37.5678677,127.0574043,닭/오리고기 구이/찜,닭한마리 (대),40000,표시 가격 확인,국물요리,닭한마리,단품,식사,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,15:00-22:30\r\n'
    'R278cbec1579c,장원닭한마리,서울 동대문구 전농로 36 답십리포레스트스퀘어 1층 107호,37.5678677,127.0574043,닭/오리고기 구이/찜,닭한마리 (중),30000,표시 가격 확인,국물요리,닭한마리,단품,식사,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,16:00-22:30,15:00-22:30\r\n'
    'Rf2a3eee82f46,재민식당,서울 동대문구 무학로27길 11 (용두동) 1층,37.5744808,127.0298514,백반/한정식,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,08:00-20:00,08:00-20:00,08:00-20:00,08:00-20:00,08:00-20:00,08:00-15:00,휴무\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,감자탕 2~3인분 포장,15000,표시 가격 확인,국물요리,감자탕,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,감자탕 3~4인분 포장,20000,표시 가격 확인,국물요리,감자탕,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,감자탕 大,40000,표시 가격 확인,국물요리,감자탕,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,감자탕 小,30000,표시 가격 확인,국물요리,감자탕,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,감자탕 中,35000,표시 가격 확인,국물요리,감자탕,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,뼈찜 2~3인분 포장,27000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,뼈찜 3~4인분 포장,32000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,뼈찜(대),44000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,뼈찜(소),34000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,뼈찜(중),39000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,뼈해장국,10000,표시 가격 확인,국물요리,해장국,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,얼큰 국밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,오리지날 국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'Rf77cc20414b4,재수돈감자탕 왕십리점,왕십리로21길 13 1층,37.5611903,127.0330605,국/탕/찌개류,특뼈해장국,13000,표시 가격 확인,국물요리,해장국,단품,식사,휴무,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00,10:00-22:00\r\n'
    'R57d241315535,전과함께,"서울 성동구 무학봉15길 28-1, 1층 (하왕십리동)",37.5635909,127.0303483,한식,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,15:00-22:00,휴무\r\n'
    'R482c140765f9,전과함께(막걸리한잔),서울 성동구 무학봉15길 28-1 (하왕십리동) 1층,37.5635909,127.0303483,,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,12:00-22:00,15:00-22:00,휴무\r\n'
    'R5ac3a405c454,전봇대집,"서울 성동구 살곶이길 18, 성동안심상가 마장청계점 3층 302호 (마장동)",37.5715395,127.0400171,기타,한우 모듬 (700g),110000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30,11:00-22:30\r\n'
    'Rf7184ae3d659,전주식당,서울 성동구 무학봉15가길 1 (하왕십리동) .,37.5614972,127.0308649,한식,백반,8000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30\r\n'
    'Rf7184ae3d659,전주식당,서울 성동구 무학봉15가길 1 (하왕십리동) .,37.5614972,127.0308649,한식,생선구이,10000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30,08:00-21:30\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,가마솥100cm통순대 100cm 통순대가 들어간 압도적인 비쥬얼! 전골과 김치전까지 꽁짜!,49000,표시 가격 확인,분식,순대,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,가마솥모둠수육 가마솥수육과 가마솥전골 2가지 메뉴를 즐길 수 있는 정남옥 시그니처 메뉴,42900,표시 가격 확인,구이·볶음·찜,보쌈·수육,모둠,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,가마솥순대곱창전골,37900,표시 가격 확인,분식,순대,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,가마솥우삼겹수육 우삼겹수육과 가마솥전골 2가지를 한 번에 즐길 수 있는 정남옥 시그니처 메뉴,46900,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,고기폭탄순대국 고기와 내장이 2배가 들어가 양도 단백질도 풍부합니다!,15900,표시 가격 확인,국물요리,국밥,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,녹두전,8900,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,들기름돼지육전,17900,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,떡갈비(2p),7000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,미나리 사골순대국 진한 사골 육수에 미나리의 향긋함이 더 해졌습니다.,10900,표시 가격 확인,국물요리,국밥,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,미나리 얼큰순대국 얼큰한 국물과 향긋한 미나리가 만나 더욱 시원하게 만듭니다,11900,표시 가격 확인,국물요리,국밥,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,뼈해장국 부드러운 등뼈와 사골 육수로 끓여내어 국물의 무게감이 다릅니다,11900,표시 가격 확인,국물요리,해장국,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,사골순대국 가마솥으로 끓여낸 깊고 진한 사골육수가 일품입니다.,9900,표시 가격 확인,국물요리,국밥,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,섞어모둠수육 수육·순대·내장을 골고루 섞어 다양한 식감을 한 번에 즐기는 모둠 메뉴입니다.,32000,표시 가격 확인,구이·볶음·찜,보쌈·수육,모둠,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,순대곱창볶음(맛보기),9900,표시 가격 확인,분식,순대,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,술국,21900,표시 가격 확인,국물요리,국밥,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,신림순대곱창볶음,25900,표시 가격 확인,분식,순대,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,양푼이미나리감자탕,35900,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,얼큰사골순대국 20시간 이상 항아리 숙성된 비법 양념장으로 끓였습니다. 강추!!,10900,표시 가격 확인,국물요리,국밥,단품,식사,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,청양매콤부추전,13900,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R74df11be1449,정남옥순대국 왕십리점,"서울 성동구 왕십리로 331, 1층 (하왕십리동)",37.5625972,127.0327852,한식,치즈김치전,8900,표시 가격 확인,곁들임,전·부침,단품,사이드,10:00-다음 날 10:00,10:00-22:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00,10:00-다음 날 10:00\r\n'
    'R24f1fe56bedf,정담초밥 신금호 직영점,서울 성동구 금호로 162 1층,37.5553035,127.0201472,초밥·롤,2인 세트 B,38000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R24f1fe56bedf,정담초밥 신금호 직영점,서울 성동구 금호로 162 1층,37.5553035,127.0201472,초밥·롤,모둠초밥 (10p),12000,표시 가격 확인,초밥·회,초밥,모둠,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R24f1fe56bedf,정담초밥 신금호 직영점,서울 성동구 금호로 162 1층,37.5553035,127.0201472,초밥·롤,특초밥 (12p),16000,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf5c2ca353926,정통남원추어탕,왕십리로21가길 1,37.5610470,127.0330384,국/탕/찌개류,얼큰한 통추어탕,10000,표시 가격 확인,국물요리,추어탕,단품,식사,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,휴무\r\n'
    'Rf5c2ca353926,정통남원추어탕,왕십리로21가길 1,37.5610470,127.0330384,국/탕/찌개류,추어탕,9000,표시 가격 확인,국물요리,추어탕,단품,식사,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,휴무\r\n'
    'Rf5c2ca353926,정통남원추어탕,왕십리로21가길 1,37.5610470,127.0330384,국/탕/찌개류,추어튀김,15000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,11:15-21:00,휴무\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),김치찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),돼지불백,9000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),된장찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),순두부찌개,9000,표시 가격 확인,국물요리,찌개류,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),오제볶음,10000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),오징어볶음,11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),"오징어볶음 (2인분 이상 주문 가능, 1인분)",11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),제육볶음,9000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),쭈꾸미볶음,11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),"쭈꾸미볶음 (2인분 이상 주문 가능, 1인분)",11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'Rc88aafda74e9,제제집 한양대점,"서울 성동구 마조로3길 4, 1층 (행당동)",37.5585231,127.0402949,김밥(도시락),쭈제볶음,10000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,가스파쵸 냉파스타 (여름한정메뉴),21000,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,강소백 하이볼,9500,표시 가격 확인,음료,하이볼,단품,음료,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,라구파스타,22900,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,마라크림 새우 리조또,28500,표시 가격 확인,밥·죽류,리조또,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,마파 양갈비,39500,표시 가격 확인,구이·볶음·찜,양고기구이,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,맥주,8000,표시 가격 확인,음료,맥주,단품,음료,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,바삭한 쯔란새우,23000,표시 가격 확인,미확인,미확인,단품,미확인,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,바질새우춘권,15900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,발사믹 꿔바로우,24000,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,볼로냐 두부,19500,표시 가격 확인,미확인,미확인,단품,미확인,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,봉골레 파스타,21000,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,봉골레울면,19500,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,새우 크림리조또,21000,표시 가격 확인,밥·죽류,리조또,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,어향가지와 부라타치즈 튀김,19500,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,오이탕탕이,11900,표시 가격 확인,샐러드,샐러드,단품,사이드,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,유로피안 동파육,29000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,질소 티라미수 or 말차질소 티라미수,9500,표시 가격 확인,디저트,케이크,단품,디저트,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,차돌 마라크림 짬뽕,19500,표시 가격 확인,국수류,짬뽕,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,추가 빵,3000,표시 가격 확인,곁들임,난·빵,단품,사이드,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,크림리조또 공기밥,4500,표시 가격 확인,밥·죽류,리조또,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,탄탄파스타,21000,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,하우스 와인 (레드/화이트),9000,표시 가격 확인,음료,와인,단품,음료,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,해산물 바질 짬뽕,19500,표시 가격 확인,국수류,짬뽕,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,해산물 크림파스타,21900,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R172ccb955a96,중앙감속기,"서울 성동구 성수일로6길 7-1, 1층 (성수동2가)",37.5455723,127.0510932,기타,해산물 토마토 파스타,21900,표시 가격 확인,파스타류,파스타,단품,식사,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00,12:00-21:00\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,고기마요,5000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,나이스라이스,5000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,누들누들,5000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,오돌뼈,5000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,제육마요,5000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,참치김치마요,5000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'R2fb0ef15281b,지지고 한양대점,왕십리로 236 1층 2호,37.5578450,127.0413322,백반/한정식,참치마요,5000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-15:00,휴무\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,김치찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,닭볶음탕,25000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,돌솥비빔밥,6000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,동테찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,된장찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,부대찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,삼겹살,12000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,삼계탕,11000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,순두부찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,오징어덮밥,7000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,제육덮밥,6000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rab55a52cfb34,진선미식당,서울 성동구 왕십리로24길 12 (도선동),37.5635325,127.0342952,백반/한정식,청국장찌개,6000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,관자 버터 파스타,24000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,김페스토 들깨크림파스타,23000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,돈토로동,16900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,레몬사와,7900,표시 가격 확인,음료,칵테일,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,레몬시소에이드,6900,표시 가격 확인,음료,에이드,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,모듬카츠,18900,표시 가격 확인,튀김·커틀릿,돈가스,모둠,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,사시미 플레이트,36000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,사케동,17900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,싱하 탄산수,3000,표시 가격 확인,음료,탄산수,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,아부리사케동,17900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,어란 오일 파스타,24000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,연어 아보카도 타르타르,28000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,오키나와생맥주 400cc,8900,표시 가격 확인,음료,맥주,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,우나기동,35000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,우니바질파스타,24000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,우미마끼,25000,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,유즈에이드,6900,표시 가격 확인,음료,에이드,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,춘하추동,11900,표시 가격 확인,미확인,미확인,단품,미확인,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,춘하추동사와,7900,표시 가격 확인,음료,칵테일,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,카이센동,28900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,코카콜라/코카콜라제로/스프라이트,3000,표시 가격 확인,음료,탄산음료,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,토닉워터 300ml,3000,표시 가격 확인,음료,탄산음료,단품,음료,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,트러플 홍새우 파스타,24000,표시 가격 확인,파스타류,파스타,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,호르몬동,16900,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R5c48c21e8d77,진작 다이닝,"서울 성동구 왕십리로14나길 10, 1층 (성수동1가)",37.5493151,127.0456890,기타,후토마끼,17900,표시 가격 확인,초밥·회,초밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-20:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,직화낙지볶음(2인이상주문),12000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,직화삼겹살볶음(2인이상주문),11000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,직화쭈꾸미볶음(2인이상주문),11000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,철판낙지(2인),30000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,철판낙지(3인),45000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,철판닭갈비(2인),26000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,철판닭갈비(3인),39000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,철판쭈꾸미(2인),28000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R46679d1bdbe9,집밥쭈선생,"서울 성동구 용답19가길 23, 1층 (용답동)",37.5639441,127.0523890,한식,철판쭈꾸미(3인),42000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,10:30-22:30,휴무,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30,10:30-22:30\r\n'
    'R6d113f6745c9,쪽선생,"서울 성동구 용답29길 15-1, 1층 (용답동, 12-2 (101호))",37.5640979,127.0543743,한식,선생 쪽갈비 1인분,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,휴무,13:00-22:00\r\n'
    'R6d113f6745c9,쪽선생,"서울 성동구 용답29길 15-1, 1층 (용답동, 12-2 (101호))",37.5640979,127.0543743,한식,셀프주먹밥,5000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,휴무,13:00-22:00\r\n'
    'R6d113f6745c9,쪽선생,"서울 성동구 용답29길 15-1, 1층 (용답동, 12-2 (101호))",37.5640979,127.0543743,한식,청국장,7500,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,휴무,13:00-22:00\r\n'
    'R11898b362870,쪽선생쪽돼지,서울 성동구 용답29길 15-1 1층,37.5640979,127.0543743,돼지고기 구이/찜,선생 쪽갈비 1인분,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,휴무,13:00-22:00\r\n'
    'R11898b362870,쪽선생쪽돼지,서울 성동구 용답29길 15-1 1층,37.5640979,127.0543743,돼지고기 구이/찜,셀프주먹밥,5000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,휴무,13:00-22:00\r\n'
    'R11898b362870,쪽선생쪽돼지,서울 성동구 용답29길 15-1 1층,37.5640979,127.0543743,돼지고기 구이/찜,청국장,7500,표시 가격 확인,국물요리,찌개류,단품,식사,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,15:00-22:30,휴무,13:00-22:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,게살스프,7900,표시 가격 확인,국물요리,수프,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,고추잡채,25900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,깐풍마늘가지새우(4ps),24900,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,레몬크림새우 6ps,26000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,마라샹궈,30900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,마파두부,15900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,몽골리안비프,31900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,오렌지치킨,21900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,오이무침,7000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,유린군만두,7000,표시 가격 확인,분식,만두,단품,사이드,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,제네럴쏘치킨,21900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,짜장면,8500,표시 가격 확인,국수류,짜장면,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,짬뽕,12900,표시 가격 확인,국수류,짬뽕,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,차알 탕수육,20900,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,차알 팔보채,41000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,차알마라탕면,13900,표시 가격 확인,국물요리,마라탕,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,차알볶음밥,10900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,차알유린기,20900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,차우멘,13900,표시 가격 확인,국수류,볶음면,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,청경채마늘볶음,13000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,칠리새우6ps,26000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,토마토차우멘,13900,표시 가격 확인,국수류,볶음면,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,투움바차우멘,15900,표시 가격 확인,국수류,볶음면,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,해물 백짬뽕,15900,표시 가격 확인,국수류,짬뽕,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,해물누룽지탕,39500,표시 가격 확인,국물요리,국·탕,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,해물덮밥,16900,표시 가격 확인,밥·죽류,덮밥,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b2c5314c904,차알 청량리아트포레스트점,서울 동대문구 고산자로32길 78 2층 205~208호,37.5776524,127.0428595,중국집,해물짬뽕탕,22900,표시 가격 확인,국물요리,국·탕,단품,식사,휴무,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R6b03b96fe745,착한짜장,서울 성동구 아차산로7길 17-1 (성수동2가) 1층,37.5467302,127.0546677,중국식,자장면,5500,표시 가격 확인,국수류,짜장면,단품,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30\r\n'
    'R6b03b96fe745,착한짜장,서울 성동구 아차산로7길 17-1 (성수동2가) 1층,37.5467302,127.0546677,중국식,짬뽕,7000,표시 가격 확인,국수류,짬뽕,단품,식사,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30,10:30-20:30\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,감자탕(대),48000,표시 가격 확인,국물요리,감자탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,감자탕(소),32000,표시 가격 확인,국물요리,감자탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,감자탕(중),40000,표시 가격 확인,국물요리,감자탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,국내산 검은양선지,10000,표시 가격 확인,미확인,미확인,단품,미확인,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,국내산 검은양선지(특),13000,표시 가격 확인,미확인,미확인,단품,미확인,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,뼈해장국,10000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,뼈해장국(특),14000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,선지해장국,8000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,추어탕,11000,표시 가격 확인,국물요리,추어탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,콩나물해장국,8000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rd83727103bba,청진동해장국,"서울 성동구 마조로18길 21 (마장동,(지상1층))",37.5662699,127.0435989,한식,황태해장국,9000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,감자탕(대),48000,표시 가격 확인,국물요리,감자탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,감자탕(소),32000,표시 가격 확인,국물요리,감자탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,감자탕(중),40000,표시 가격 확인,국물요리,감자탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,국내산 검은양선지,10000,표시 가격 확인,미확인,미확인,단품,미확인,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,국내산 검은양선지(특),13000,표시 가격 확인,미확인,미확인,단품,미확인,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,뼈해장국,10000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,뼈해장국(특),14000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,선지해장국,8000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,추어탕,11000,표시 가격 확인,국물요리,추어탕,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,콩나물해장국,8000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'Rb054ff4e850f,청진동해장국 마장동점,서울 성동구 마조로18길 21 명신빌딩,37.5662699,127.0435989,국/탕/찌개류,황태해장국,9000,표시 가격 확인,국물요리,해장국,단품,식사,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00,06:00-24:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,계란찜,2000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,날치알 주먹밥,6000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,누룽지,6000,표시 가격 확인,밥·죽류,알밥·영양밥,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,매운 쪽갈비,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,생맥주 (300ml),3500,표시 가격 확인,음료,맥주,단품,음료,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,생맥주 (500ml),5500,표시 가격 확인,음료,맥주,단품,음료,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,양념 쪽갈비,19000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,용궁라면,7000,표시 가격 확인,국수류,라면,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,참갈비 (생쪽갈비),21000,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,청춘 하이볼,8000,표시 가격 확인,음료,하이볼,단품,음료,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,최루탄 주먹밥 中,10000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R6ee5b3dbcbca,청춘극장,서울 성동구 행당로 112-5 (행당동),37.5577422,127.0312644,한식,홍합탕,14000,표시 가격 확인,국물요리,국·탕,단품,식사,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00,15:00-23:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,[SET] 초선 마라탕+계란볶음밥,18900,표시 가격 확인,국물요리,마라탕,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,[SET] 초선 마라탕+꿔바로우,18900,표시 가격 확인,국물요리,마라탕,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,[SET] 초선 마라탕+꿔바로우+계란볶음밥,26900,표시 가격 확인,국물요리,마라탕,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,[SET] 초선 불맛 마라샹궈+꿔바로우,23900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,공기밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,단무지,500,표시 가격 확인,곁들임,반찬·무침,단품,사이드,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,땅콩소스,500,표시 가격 확인,미확인,미확인,단품,미확인,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,마라유,500,표시 가격 확인,미확인,미확인,단품,미확인,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,발효식초,500,표시 가격 확인,미확인,미확인,단품,미확인,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,빙탕설리(500ml),3000,표시 가격 확인,음료,기타 음료,단품,음료,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,빙홍차(490ml),3000,표시 가격 확인,음료,차,단품,음료,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,사이다(355ml),2000,표시 가격 확인,음료,탄산음료,단품,음료,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,제로콜라(펩시)(355ml),2000,표시 가격 확인,음료,탄산음료,단품,음료,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,초선 마라탕,11900,표시 가격 확인,국물요리,마라탕,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,초선 불맛 마라샹궈,16900,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,콜라(펩시)(355ml),2000,표시 가격 확인,음료,탄산음료,단품,음료,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,크림 쫀득 꿔바로우(대),24900,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,크림 쫀득 꿔바로우(소),9900,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R92ea72d31325,초선마라 왕십리점,살곶이8길 2 지하1층,37.5571944,127.0410122,중국식,크림 쫀득 꿔바로우(중),16900,표시 가격 확인,튀김·커틀릿,탕수육,단품,식사,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00,06:00-다음 날 03:00\r\n'
    'R0e5c4f28dc00,최영희 언양불고기,"서울 성동구 무학봉28길 21 (행당동, 286-12 지상1층)",37.5615915,127.0341305,한식,언양불고기(200g),17000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'R0e5c4f28dc00,최영희 언양불고기,"서울 성동구 무학봉28길 21 (행당동, 286-12 지상1층)",37.5615915,127.0341305,한식,언양불고기+차돌된장+공기밥,19000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'R0e5c4f28dc00,최영희 언양불고기,"서울 성동구 무학봉28길 21 (행당동, 286-12 지상1층)",37.5615915,127.0341305,한식,오징어매콤불고기,25000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'R0e5c4f28dc00,최영희 언양불고기,"서울 성동구 무학봉28길 21 (행당동, 286-12 지상1층)",37.5615915,127.0341305,한식,전골불고기,17000,표시 가격 확인,국물요리,전골,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'R0e5c4f28dc00,최영희 언양불고기,"서울 성동구 무학봉28길 21 (행당동, 286-12 지상1층)",37.5615915,127.0341305,한식,차돌된장,5000,표시 가격 확인,국물요리,찌개류,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'R0e5c4f28dc00,최영희 언양불고기,"서울 성동구 무학봉28길 21 (행당동, 286-12 지상1층)",37.5615915,127.0341305,한식,한우육회,25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,고등어 솥밥,18000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,스테이크 솥밥,25000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,언양식 납작불고기 솥밥,18000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,연어 솥밥,20000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,장어 솥밥,23000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,전복장 솥밥,20000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,쵸린기,22000,표시 가격 확인,구이·볶음·찜,중화 볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rbbef26979cdc,쵸리상경,서울 성동구 서울숲4길 18-8 2층 쵸리상경,37.5470517,127.0426738,솥밥,표고 멘보샤 4pc,14000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,고기만두/부추만두/만두반반,10000,표시 가격 확인,분식,만두,단품,사이드,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,고수한접시,500,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,공깃밥,1000,표시 가격 확인,밥·죽류,흰밥,단품,추가옵션,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,꼬마 물만두,7000,표시 가격 확인,분식,만두,단품,사이드,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,냉면/매운냉면/아기냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,마라냉면,9000,표시 가격 확인,국수류,냉면,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,마라비빔면,9000,표시 가격 확인,국수류,국수,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,마라온국밥,11000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,마라온면,10000,표시 가격 확인,국수류,국수,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,매운 온국밥,10000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,비빔면 / 매운비빔면/아기비빔면,8000,표시 가격 확인,국수류,국수,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,온국밥,10000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,온면 / 매운 온면,9000,표시 가격 확인,국수류,국수,단품,식사,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'Rd73fcff6998f,춘향미엔,"서울 성동구 왕십리로21길 26, 1층 (행당동)",37.5605459,127.0319995,중국식,차돌한접시,8000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 01:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-22:00\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,간장 스테이크,10000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,순살 데리갈릭 치킨,20900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,순살 로스트 치킨,19900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,순살 바베큐 치킨,20900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,양념 스테이크,10000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,웰빙 스테이크,9500,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,치즈 칠리 로스트 치킨,21900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,칠리 스테이크,10000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,카레 스테이크,10500,표시 가격 확인,커리·스튜,커리,단품,식사,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R355f9572c784,치폴레옹한양대점,"서울 성동구 마조로3길 2-1, 1층 (행당동)",37.5585015,127.0404140,호프/통닭,콘치즈소세지,13000,표시 가격 확인,미확인,미확인,단품,미확인,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,10:30-22:00,휴무\r\n'
    'R245ac72460e3,친친,"서울 동대문구 정릉천동로 58 (용두동, 롯데캐슬피렌체)",37.5769840,127.0357278,중국집,볶음밥,8000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R245ac72460e3,친친,"서울 동대문구 정릉천동로 58 (용두동, 롯데캐슬피렌체)",37.5769840,127.0357278,중국집,짜장면,6000,표시 가격 확인,국수류,짜장면,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R245ac72460e3,친친,"서울 동대문구 정릉천동로 58 (용두동, 롯데캐슬피렌체)",37.5769840,127.0357278,중국집,짬뽕,9000,표시 가격 확인,국수류,짬뽕,단품,식사,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,휴무,휴무\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,골든토리국밥(鷄),10000,표시 가격 확인,국물요리,국밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,골든토리라멘(鷄),13900,표시 가격 확인,국수류,일본식 라멘,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,더블연어덮밥한상,21900,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,돈마제소바,12900,표시 가격 확인,국수류,일본식 라멘,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,들기름메밀소바,9500,표시 가격 확인,국수류,모밀·소바,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,매콤돈마제소바,12900,표시 가격 확인,국수류,일본식 라멘,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,미소등심카츠동,13500,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,블랙니쿠국밥(牛),10000,표시 가격 확인,국물요리,국밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,블랙니쿠라멘(牛),13900,표시 가격 확인,국수류,일본식 라멘,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,산더미 야끼토리동(550g),17500,표시 가격 확인,구이·볶음·찜,꼬치구이,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,스리라차연어덮밥한상,16900,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,애호박카라이라멘,13900,표시 가격 확인,국수류,일본식 라멘,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,연어장부각덮밥한상,16900,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,우불고기 마제소바,13900,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,우삼겹덮밥,9300,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,우삼겹덮밥한상,13900,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,직화 통야끼토리동,14000,표시 가격 확인,구이·볶음·찜,꼬치구이,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,큐브연어장덮밥,13500,표시 가격 확인,밥·죽류,덮밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,큐브육회덮밥,11800,표시 가격 확인,밥·죽류,비빔밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,토로토로 게살 오므라이스,11900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'R93a144579bb1,카모메(한양대점),"서울 성동구 마조로 17 (행당동,4호 지상1층)",37.5595096,127.0409083,한식,트러플감자전,8900,표시 가격 확인,곁들임,전·부침,단품,사이드,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-21:00,09:50-17:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,단호박깜빠뉴,6800,표시 가격 확인,곁들임,빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,단호박치즈케이크 1호,42000,표시 가격 확인,디저트,케이크,단품,디저트,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,당근케이크1호,42000,표시 가격 확인,디저트,케이크,단품,디저트,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,마장불고기바게트,5800,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,명란 쌀 바게트,4200,표시 가격 확인,곁들임,난·빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,무화과 깜빠뉴,6800,표시 가격 확인,곁들임,빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,바닐라라떼,5500,표시 가격 확인,음료,커피,단품,음료,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,소금빵,3200,표시 가격 확인,곁들임,빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,아메리카노,4500,표시 가격 확인,음료,커피,단품,음료,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,앙버터 소금빵,4200,표시 가격 확인,곁들임,빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,얼그레이 가또 1호,32000,표시 가격 확인,디저트,케이크,단품,디저트,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,오레오 초코케이크 1호,36000,표시 가격 확인,디저트,케이크,단품,디저트,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,올리브 치아바타,4200,표시 가격 확인,곁들임,난·빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,카페라떼,5500,표시 가격 확인,음료,커피,단품,음료,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,카푸치노,5500,표시 가격 확인,음료,커피,단품,음료,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,크로와상,4300,표시 가격 확인,곁들임,빵,단품,사이드,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,포레누아 1호,34000,표시 가격 확인,디저트,케이크,단품,디저트,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'Rc4620b4f6053,카페드씨밀레,"서울 성동구 마장로44길 8-18, 1층, 2층 (마장동)",37.5665422,127.0451634,기타,헤이즐넛라떼,5500,표시 가격 확인,음료,커피,단품,음료,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00,08:30-21:00\r\n'
    'R9cadd9138eee,커피로 그리는 꿈 협동조합,서울 성동구 상원6나길 22-11 (성수동1가) 1층,37.5476908,127.0499912,기타,아메리카노,3000,표시 가격 확인,음료,커피,단품,음료,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,휴무,휴무\r\n'
    'Rb0d609139a32,통돼지두루치기김치찌개전문점,"서울 성동구 용답중앙15길 5-1, 지상1,2층 (용답동)",37.5641752,127.0531474,한식,두부사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rb0d609139a32,통돼지두루치기김치찌개전문점,"서울 성동구 용답중앙15길 5-1, 지상1,2층 (용답동)",37.5641752,127.0531474,한식,라면사리,2000,표시 가격 확인,국수류,라면,추가,추가옵션,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rb0d609139a32,통돼지두루치기김치찌개전문점,"서울 성동구 용답중앙15길 5-1, 지상1,2층 (용답동)",37.5641752,127.0531474,한식,볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rb0d609139a32,통돼지두루치기김치찌개전문점,"서울 성동구 용답중앙15길 5-1, 지상1,2층 (용답동)",37.5641752,127.0531474,한식,통돼지 두루치기,14000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Raf168f53de99,티엔리 베트남 쌀국수&반미,"서울 성동구 독서당로60길 2, 1층 (응봉동)",37.5497169,127.0307874,기타,반세오,12000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,분짜+미니쌀국수+새우월남쌈2p(특가할인)분짜하노이+미니쌀국수+새우월남쌈2p,26000,표시 가격 확인,국수류,국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,분짜+짜조 3p(한그릇할인)숯불 소제양념 고기+고급 버미셀리면+야채+소제 메콤달콤느억맘소스,15900,표시 가격 확인,국수류,국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,삼겹살 + 김치 볶음밥 + 계란++ 계란후라이1개기본 + 삼겹살 + 김치 볶음밥,13900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,삼겹살구이 250G (고기만)(1인분)양파 + 고기 + 쌈장,16900,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,새우 파인애플 볶밥+계란 달콤하게 바다의 향을 느끼고 싶은 분들에게 강추,10700,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,소고기볶음밥+계란(인기)소고기 ++ 마늘 듬뿍 넣고 같이 볶은 볶음밥 ++ 계란후라이,10700,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,소고기쌀국수+사이드 선택 구수한 소고기쌀국수와 맛보기 사이드메뉴까지 선택해서 주문할 수 있으니 건강하고 든든한 한끼 되세요:),12000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,스팸 + 김치 볶음밥 + 계란(특가 (NEW MENU)),9000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,스프라이트(355ml 일발맛),2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,스프링롤(닭가슴살 2P)새우.야채 등.면을 들어 있습니다.,5900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,스프링롤(닭가슴살 3P)새우.야채 등.면을 들어 있습니다.,8200,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,스프링롤(새우 2P)새우.야채 등.면을 들어 있습니다.,5900,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,스프링롤(새우 3P)새우.야채 등.면을 들어 있습니다.,8200,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,왕갈비 쌀국수(갈비+우삼겹 (프리미엄))넉넉한 갓성비 소왕갈비 쌀국수. 살살 녹는 왕갈비 한덩이 어떠세요,15400,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,왕갈비 쌀국수(소갈비)넉넉한 갓성비 소왕갈비 쌀국수. 살살 녹는 왕갈비 한덩이 어떠세요,13200,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,우삼겹 (소고기) 김치볶음밥+계란(1인분)우삼겹 + 김치볶음밥,13900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,제로콜라355ml ZERO COLA,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,직화쌀국수 [소고기뜸뿍](고기 없음)구수한 갓성비 양지 쌀국수. 12시간 이상 깊게 직접 우려낸 육수가 일품이에요,9700,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,직화쌀국수 [소고기뜸뿍](소고기)구수한 갓성비 양지 쌀국수. 12시간 이상 깊게 직접 우려낸 육수가 일품이에요,10700,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,직화쌀국수 [소고기뜸뿍](왕새우+우삼겹 (프리미엄))구수한 갓성비 양지 쌀국수. 12시간 이상 깊게 직접 우려낸 육수가 일품이에요,13200,표시 가격 확인,국수류,쌀국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,춘권 (바삭 튀김)(10P),5600,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,코카콜라355ml COLA,2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,팟타이+계란(새우 팟타이)쌀국수와 킹새우.계란.숙주와 함께 특제 매콤소스로 볶아낸 쌀국수,14500,표시 가격 확인,국수류,볶음면,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,팟타이+계란(소고기 팟타이)쌀국수와 킹새우.계란.숙주와 함께 특제 매콤소스로 볶아낸 쌀국수,14500,표시 가격 확인,국수류,볶음면,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,팟타이+계란후라이 (모듬 프리미엄)(새우+우삼겹 (프리미엄)),14900,표시 가격 확인,국수류,볶음면,모둠,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R1fa003a580b1,퍼녀,"서울 성동구 무학봉28길 4-1, 지2층 B203호 (하왕십리동)",37.5622223,127.0324799,기타,환타(355ml 파인애플),2000,표시 가격 확인,음료,탄산음료,단품,음료,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00\r\n'
    'R39867ede25bb,포레스트,"서울 성동구 서울숲4길 12-7, 스타힐스 1층 (성수동1가)",37.5472622,127.0423070,경양식,그릴드 바베큐 소스 포크 본,26000,표시 가격 확인,미확인,미확인,단품,미확인,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R39867ede25bb,포레스트,"서울 성동구 서울숲4길 12-7, 스타힐스 1층 (성수동1가)",37.5472622,127.0423070,경양식,부라타 치즈 샐러드,21000,표시 가격 확인,샐러드,샐러드,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R39867ede25bb,포레스트,"서울 성동구 서울숲4길 12-7, 스타힐스 1층 (성수동1가)",37.5472622,127.0423070,경양식,트러플 감자뇨끼,25000,표시 가격 확인,파스타류,뇨끼,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R39867ede25bb,포레스트,"서울 성동구 서울숲4길 12-7, 스타힐스 1층 (성수동1가)",37.5472622,127.0423070,경양식,트러플 화이트 라구 파스타,29000,표시 가격 확인,파스타류,파스타,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R39867ede25bb,포레스트,"서울 성동구 서울숲4길 12-7, 스타힐스 1층 (성수동1가)",37.5472622,127.0423070,경양식,해산물 리조또,26000,표시 가격 확인,밥·죽류,리조또,단품,식사,휴무,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00,11:30-21:00\r\n'
    'R376e422ad0e4,포림,"서울 성동구 행당로17길 18, 2층 (행당동)",37.5599831,127.0333734,기타,양지쌀국수,8900,표시 가격 확인,국수류,쌀국수,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-23:30,11:00-23:30,11:00-22:00\r\n'
    'R3531791a327f,포싸이공,서울 성동구 왕십리로 219-2,37.5570223,127.0426826,베트남음식,,7900,대표가 추정(메뉴 미등록),,,,,10:00-21:00,10:00-22:00,10:00-21:00,10:00-21:00,10:00-21:00,10:30-17:30,휴무\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,가브리살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,갈매기살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,생목살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,생삼겹살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,차돌박이,25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,푸줏간스페셜,188000,표시 가격 확인,미확인,미확인,단품,미확인,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,푸줏간스페셜반,98000,표시 가격 확인,미확인,미확인,단품,미확인,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우등심,49000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우모듬,128000,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우목심,25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우부채살,49000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우살치살,52000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우육회한접시,30000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,한우차돌박이,35000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R503cad585658,푸줏간생고기점,"서울 성동구 행당로17길 24 (행당동,지상1층)",37.5603885,127.0332117,한식,항정살,20000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-22:30\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,감바스,25000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,고르곤졸라피자,19000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,굴라쉬,28000,표시 가격 확인,커리·스튜,스튜,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,그릴드소세지,20000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,꼴레뇨,39000,표시 가격 확인,곁들임,곁들임,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,두부야채 모듬구이,14000,표시 가격 확인,미확인,미확인,모둠,미확인,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,리코타치즈 토마토 파스타,18900,표시 가격 확인,파스타류,파스타,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,머쉬룸크림파스타,18000,표시 가격 확인,파스타류,파스타,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,머쉬룸피자,20000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,먹태,13000,표시 가격 확인,곁들임,곁들임,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,바질치즈피자,19000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,버팔로윙,21000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,부각&야채칩,15000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,부라타 치즈 샐러드,18000,표시 가격 확인,샐러드,샐러드,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,비프 스테이크 샐러드,22000,표시 가격 확인,구이·볶음·찜,스테이크,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,살사 나쵸,13000,표시 가격 확인,곁들임,곁들임,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,쉬림프 바질 오일 파스타,19900,표시 가격 확인,파스타류,파스타,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,쉬림프 샐러드,20000,표시 가격 확인,샐러드,샐러드,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,쉬림프떡볶이(로제/크림),22000,표시 가격 확인,분식,떡볶이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,스테이크 로제 파스타,25000,표시 가격 확인,파스타류,파스타,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,스테이크 피자,25000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,스테이크&야채모듬,34000,표시 가격 확인,구이·볶음·찜,스테이크,모둠,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,애플시나몬피자,19000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,폴드포크 할라피뇨 피자,20000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,하몽 루꼴라 피자,23000,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,하몽&멜론,18000,표시 가격 확인,미확인,미확인,단품,미확인,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'R0680ce91ff70,프란츠카프카,"서울 성동구 왕십리로 311-1, 지하1층 (행당동)",37.5616358,127.0345954,경양식,허니 리코타치즈 크래커,13000,표시 가격 확인,곁들임,곁들임,단품,사이드,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00\r\n'
    'Rbb575a72ef5f,프란츠카프카 딜리버리,왕십리로 311-1,37.5616358,127.0345954,그 외 기타 간이 음식점,,16000,대표가 추정(메뉴 미등록),,,,,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-다음 날 01:00,17:00-23:00\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,고구마피자,10900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,고르곤졸라피자,12900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,까르보네피자,12900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,깐쇼새우피자,14900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,나폴리피자,12900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,닭안심살피자,14900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,더블갈릭바베큐피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,도이치바이트피자,14900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,멕시칸바이트피자,14900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,미트러버피자,15900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,불고기피자,11900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,불닭고구마피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,비프퀘사디아,13900,표시 가격 확인,피자·버거·샌드위치,랩·타코,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,스리라차마요소스,500,표시 가격 확인,미확인,미확인,단품,미확인,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,아이리쉬포테이토피자,12900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,야채퀘사디아피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,오지치즈포테이토피자,14900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,직화파인애플피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,직화홀릭바이트피자,14900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,치즈피자,8900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,치킨퀘사디아,13900,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,콘치즈피자,10900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,콤비네이션피자,10900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,크림치즈꽈배기,5900,표시 가격 확인,디저트,기타 디저트,단품,디저트,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,트러플머쉬룸피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,트러플포테이토피자,13900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,페퍼로니피자,9900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,포테이토피자,11900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,프리미엄스테이크피자,15900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R53f5d636c5b1,피자스쿨,서울 성동구 마장로 299 (마장동),37.5663174,127.0427291,패스트푸드,핫치킨피자,10900,표시 가격 확인,피자·버거·샌드위치,피자,단품,식사,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30,11:00-20:30\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,고기국수,10000,표시 가격 확인,국수류,국수,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,국밥사리 추가,2000,표시 가격 확인,국수류,면,추가,추가옵션,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,내장국밥,10000,표시 가격 확인,국물요리,국·탕,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,돼지국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,만두,6000,표시 가격 확인,분식,만두,단품,사이드,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,밀면,9000,표시 가격 확인,국수류,냉면,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,밀면수육세트,21000,표시 가격 확인,국수류,냉면,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,비빔면,9000,표시 가격 확인,국수류,국수,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,비빔면수육세트,21000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,섞어국밥,10000,표시 가격 확인,국물요리,국밥,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,소꼬리찜,59000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,수육 大,36000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,수육 中,28000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R51d4e022ec36,하동진,서울 성동구 행당로17길 17 한방양념구이 1층,37.5599257,127.0331577,한식,얼큰국밥,11000,표시 가격 확인,국물요리,국밥,단품,식사,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00,08:00-23:00\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,감자사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,감자탕 (1인식사),10000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,국물뼈찜 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,국물뼈찜 (3인),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,국물뼈찜 (4인),51000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,김치찜 (1인식사),12000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,깻잎사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,닭볶음탕 (2인),32000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,닭볶음탕 (3인),43000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,닭한마리 (2인),32000,표시 가격 확인,국물요리,닭한마리,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,닭한마리 (3인),43000,표시 가격 확인,국물요리,닭한마리,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,당면사리,3000,표시 가격 확인,국수류,당면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,떡사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,라면사리,3000,표시 가격 확인,국수류,라면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,명품감자탕 (2인),32000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,명품감자탕 (3인),39000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,명품감자탕 (4인),47000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,묵은지 닭볶음탕 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,묵은지 닭볶음탕 (3인),47000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,묵은지감자탕 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,묵은지감자탕 (3인),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,묵은지감자탕 (4인),51000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,묵은지사리,6000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,뼈찜 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,뼈찜 (3인),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,뼈찜 (4인),51000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,뼈추가,16000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,수제비사리,3000,표시 가격 확인,국수류,수제비,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,우거지사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,칼국수사리,3000,표시 가격 확인,국수류,칼국수,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R40f439c954bc,한남동 감자탕,"서울 성동구 무수막길 95, 솔베이그 지하1층 (금호동2가)",37.5542269,127.0198267,한식,콩나물사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,감자사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,감자탕 (1인식사),10000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,국물뼈찜 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,국물뼈찜 (3인),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,국물뼈찜 (4인),51000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,김치찜 (1인식사),12000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,깻잎사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,닭볶음탕 (2인),32000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,닭볶음탕 (3인),43000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,닭한마리 (2인),32000,표시 가격 확인,국물요리,닭한마리,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,닭한마리 (3인),43000,표시 가격 확인,국물요리,닭한마리,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,당면사리,3000,표시 가격 확인,국수류,당면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,떡사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,라면사리,3000,표시 가격 확인,국수류,라면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,명품감자탕 (2인),32000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,명품감자탕 (3인),39000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,명품감자탕 (4인),47000,표시 가격 확인,국물요리,감자탕,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,묵은지 닭볶음탕 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,묵은지 닭볶음탕 (3인),47000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,묵은지감자탕 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,묵은지감자탕 (3인),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,묵은지감자탕 (4인),51000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,묵은지사리,6000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,볶음밥,3000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,뼈찜 (2인),36000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,뼈찜 (3인),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,뼈찜 (4인),51000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,뼈추가,16000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,수제비사리,3000,표시 가격 확인,국수류,수제비,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,우거지사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,칼국수사리,3000,표시 가격 확인,국수류,칼국수,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'R760964ba1342,한남동감자탕 신금호역 직영점,서울 성동구 무수막길 95 솔베이그빌딩 지하1층,37.5542269,127.0198267,한식,콩나물사리,3000,표시 가격 확인,국수류,면,추가,추가옵션,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30,10:00-다음 날 01:30\r\n'
    'Rfaa9d7f13f50,한솥도시락한양대기숙사점,"서울 성동구 사근동길 68 (사근동,지상1층)",37.5610523,127.0483381,김밥(도시락),돈까스도련님,5200,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'Rfaa9d7f13f50,한솥도시락한양대기숙사점,"서울 성동구 사근동길 68 (사근동,지상1층)",37.5610523,127.0483381,김밥(도시락),빅치킨마요,4600,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'Rfaa9d7f13f50,한솥도시락한양대기숙사점,"서울 성동구 사근동길 68 (사근동,지상1층)",37.5610523,127.0483381,김밥(도시락),치킨제육,5500,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00,08:00-21:00\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"(한양SET) 곱도리탕(중) 세트 곱도리탕 + 전 택1 + 주류 택1 (소주, 맥주, 장수막걸리), 중독성 강한 인기 메뉴",56000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"(한양SET) 꽃도리탕(중) 세트 꽃도리탕 + 전 택1 + 주류 택1 (소주, 맥주, 장수막걸리), 한양 추천 세트",58000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"(한양SET) 닭도리탕(중) 세트 닭도리탕 + 전 택1 + 주류 택1 (소주, 맥주, 장수막걸리),부담없이 즐기는 인기 세트",48000,표시 가격 확인,구이·볶음·찜,닭볶음탕,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"(한양SET) 닭발 세트 매콤한 국물닭발 + 누룽지 계란찜탕 + 참치마요 주먹밥, 매운맛으로 스트레스 풀리는 조합",35000,표시 가격 확인,구이·볶음·찜,직화닭발,세트,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,(BESR)꽃도리탕 (중)연평도 꽃게와 부드러운 닭의 만남 비주얼부터 압도하는 깊은 감칠맛이 일품인 한양 시그니처 메뉴,39000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,(BEST)곱도리탕 (중)고소한 대창과 부드러운 닭의 중독성 강한 매콤 조합,37000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,고흥유자막걸리 유자향과 어울려 풍미가 좋은 막걸리,7000,표시 가격 확인,음료,막걸리,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,누룽지 계란찜탕 부드럽고 고소한 국물형 계란찜,9000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,달달 계란토스트 달달한 맛으로 인기 많은 사이드 메뉴,11000,표시 가격 확인,분식,토스트,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,달콤한 밤막걸리 달콤한 밤의 막걸리,6000,표시 가격 확인,음료,막걸리,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,닭도리탕 (중)매콤한 양념에 부드러운 닭과 아삭한 콩나물의 식감까지 더한 조합,29000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"닭한마리칼국수(중)국내산 닭을 푹 끓여낸 시원한 국물, 칼국수까지 즐기는 맑은 닭한마리",29000,표시 가격 확인,국수류,칼국수,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,대파닭구이 순살 닭다리살에 은은한 불향과 향긋한 대파를 더한 소금구이,17000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,덩더덕20%더덕의 향으로 풍미를 살린 술,20000,표시 가격 확인,음료,전통주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,돌문어숙회 & 미나리 명태초무침 쫄깃한 돌문어 숙회와 새콤달콤한 미나리 명태초무침의 환상적인 조화!,25000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,돌미나리 새우전 향긋한 미나리와 새우가 어우러진 전,17000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,들기름 막국수 고소함 살아있는 깔끔한 마무리 메뉴,9000,표시 가격 확인,국수류,막국수,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,모짜 감자전 고소한 감자에 모짜렐라 치즈를 더한 풍미 가득한 전,17000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"미나리 홍백탕 미나리향과 홍합, 백합을 더해 시원하게 끓여낸 국물",19900,표시 가격 확인,국물요리,국·탕,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"반건조오징어구이 노릇하게 구운 반건조 오징어, 쫄깃한 식감과 깊은 감칠맛이 일품인 술안주.",17000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,백세주 백세주,8000,표시 가격 확인,음료,전통주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,별빛청하,7000,표시 가격 확인,음료,전통주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,보해복분자,15000,표시 가격 확인,음료,전통주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,셀프 마무리 볶음 남은 국물에 볶아 즐기는 마무리 추천 메뉴,3000,표시 가격 확인,구이·볶음·찜,볶음요리,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,셀프 참치마요 주먹밥 간편하게 즐기는 고소한 한입 메뉴,5000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"소주/맥주 소주 : 참이슬, 참이슬 오리지널, 처음처럼, 진로, 새로맥주 : 카스, 테라",5000,표시 가격 확인,음료,소주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,순살마라닭도리탕 얼얼한 마라탕을 한양만의 방식으로 재해석한 일품요리,23000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,식혜막걸리 한양 시그니처 식혜와 막걸리의 만남,9000,표시 가격 확인,음료,막걸리,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"신당동 얼큰 국물 닭발 맵고 진한 국물, 스트레스 날리는 메뉴",23000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"아이스 황도 시원 달콤, 입가심 디저트",8000,표시 가격 확인,디저트,기타 디저트,단품,디저트,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,연평도 꽃게탕(중)연평도 꽃게로 끓여낸 구수한 맛이 일품! 푸짐한 꽃게로 만든 술 한잔이 절로 생각나는 맛!,52000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,오징어김치전 바삭한 식감에 오징어와 김치가 어우러진 전,17000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,일품진로 25%일품진로 25도,27000,표시 가격 확인,음료,소주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,장수막걸리 시원한 장수막걸리,5000,표시 가격 확인,음료,막걸리,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,직화불닭발 매콤한 닭발을 직화로 구워 화끈한 불향을 더한 한 접시,17000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,직화알곱창 쫄깃하고 고소한 알곱창에 직화 불향을 진하게 더한 한 접시,17000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,차돌된장술밥 차돌과 직접 만든 다대기를 더해 매콤하고 고소한 된장술밥,18000,표시 가격 확인,국물요리,국밥,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,초심 3대에 걸친 일본식 사케 방식으로 제조하여 목넘김이 부드럽고 깔끔한 한국형 프리미엄 사케,26000,표시 가격 확인,음료,사케,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,초코바닐라는 달고나 달콤한 달고나와 초코바닐라가 어우러진 디저트,6000,표시 가격 확인,디저트,기타 디저트,단품,디저트,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,"탄산음료 사이다,코카콜라,코카콜라제로",2000,표시 가격 확인,음료,탄산음료,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,토닉워터 토닉워터,2000,표시 가격 확인,음료,탄산음료,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,한양 마른안주 반란 다양한 건어물을 가성비 있게 즐길 수 있는 메뉴,17000,표시 가격 확인,곁들임,곁들임,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,한양 시그니처 식혜 (꼼자네 식혜)3대째 내려오는 비법 레시피로 제조하여 식사 후 즐기기 좋은 전통 음료,4000,표시 가격 확인,음료,전통음료,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,한양 한우육회 신선한 한우의 깊은 풍미를 느낄 수 있는 메뉴,25000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,해물파전 향긋한 파향에 오징어와 새우를 더해 바삭하게 부쳐낸 전,17000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,화요 25%화요 25도,25000,표시 가격 확인,음료,소주,단품,음료,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7c11b9e22172,한양 사람모이는 곳,"서울 성동구 무학봉28길 3-1, 1층 102호 (하왕십리동, 왕십리 지음재)",37.5624157,127.0326649,백반/한정식,황태해장국 황태를 진하게 우린 깊은 국물에 콩나물과 무를 더해 시원하고 깔끔하게 끓여낸 해장국,16000,표시 가격 확인,국물요리,해장국,단품,식사,11:30-24:00,11:30-24:00,11:30-24:00,11:30-24:00,11:30-다음 날 02:00,12:00-다음 날 02:00,휴무\r\n'
    'R7e4dfbe46799,한양대 신 닭발,서울 성동구 마조로 34 1층,37.5607016,127.0414769,,닭알찜,5000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R7e4dfbe46799,한양대 신 닭발,서울 성동구 마조로 34 1층,37.5607016,127.0414769,,무뼈닭발,10000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R7e4dfbe46799,한양대 신 닭발,서울 성동구 마조로 34 1층,37.5607016,127.0414769,,오돌뼈,12000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'R7e4dfbe46799,한양대 신 닭발,서울 성동구 마조로 34 1층,37.5607016,127.0414769,,주먹밥,2000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,휴무\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),모둠한판,55000,표시 가격 확인,미확인,미확인,모둠,미확인,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),소갈빗살,17900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),소곱창전골,29000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),양볶음밥 2인,11900,표시 가격 확인,밥·죽류,볶음밥,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),특양구이,22900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),한우곱창,23900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),한우대창,23900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),한우염통,14900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra529d1774b88,한양양곱창,"서울 성동구 마조로 17, 한양시장 1층 51, 52호 (행당동)",37.5595096,127.0409083,식육(숯불구이),황소막창,22900,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,17:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R25d34b3e8afe,한우공감,서울 성동구 용답중앙길 57-1 (용답동) 1층,37.5643138,127.0522983,한식,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00,11:00-23:00\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,갈비살 (한우 100g),17900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,갈비탕,12000,표시 가격 확인,국물요리,갈비탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,김치찌개,8000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,꽃등심 (한우 100g),17900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,낙지덮밥,10000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,냉면,7000,표시 가격 확인,국수류,냉면,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,된장찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,맥주,4000,표시 가격 확인,음료,맥주,단품,음료,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,복분자,10000,표시 가격 확인,음료,전통주,단품,음료,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,살치살 (한우 100g),23900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,생갈비 (한우 100g),23900,표시 가격 확인,구이·볶음·찜,갈비구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,소주,4000,표시 가격 확인,음료,소주,단품,음료,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,안심 (한우 100g),19900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,안창살 (한우 100g),27900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,우거지국밥,7000,표시 가격 확인,국물요리,국·탕,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,육사시미,30000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,육회,25000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,제육볶음,8000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,토시살 (한우 100g),25900,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rc27801676be0,한우마을,"서울 성동구 고산자로24길 11, 2층 (마장동, 509-11)",37.5700809,127.0387716,한식,특수모듬 (한우 100g),17900,표시 가격 확인,구이·볶음·찜,소고기구이,모둠,식사,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50,11:00-21:50\r\n'
    'Rdaf3113eb779,할로우치킨,"서울 성동구 용답19길 10-2, 1층 (용답동)",37.5630934,127.0522801,통닭(치킨),치킨,17000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,14:00-다음 날 02:00,14:00-다음 날 02:00,14:00-다음 날 02:00,14:00-다음 날 02:00,14:00-다음 날 02:00,13:00-다음 날 02:00,13:00-다음 날 02:00\r\n'
    'R9bb30c9dff4e,할루우치킨,서울 성동구 용답19길 10-2 (용답동) 1층,37.5630934,127.0522801,요리 주점,치킨,17000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,14:00-다음 날 02:00,14:00-다음 날 02:00,14:00-다음 날 02:00,14:00-다음 날 02:00,14:00-다음 날 02:00,13:00-다음 날 02:00,13:00-다음 날 02:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,간장돌게장,16000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,강경젓갈비빔밥,13000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,강경젓갈쌈밥,13000,표시 가격 확인,밥·죽류,쌈밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,골뱅이소면,25000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,버섯부추비빔밥,12000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,비빔밥불고기정식,17000,표시 가격 확인,밥·죽류,비빔밥,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,생선구이정식,14000,표시 가격 확인,구이·볶음·찜,생선구이,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,숨뼈국,14000,표시 가격 확인,국물요리,국밥,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,쌈밥정식,17000,표시 가격 확인,밥·죽류,쌈밥,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,저녁 생일상차림,30000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,제육볶음,25000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,직화바싹불고기,26000,표시 가격 확인,구이·볶음·찜,불고기,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,한우떡갈비,38000,표시 가격 확인,구이·볶음·찜,소고기구이,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R267217ee8b46,할머니의 레시피,"서울 성동구 서울숲2길 44-12, 지하1,1층 (성수동1가)",37.5461334,127.0431494,한식,한우떡갈비정식,17000,표시 가격 확인,구이·볶음·찜,소고기구이,세트,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R73613ac879de,해빈,"서울 성동구 용답중앙길 80, 1층 (용답동)",37.5627474,127.0540098,일식,[점심한정]모듬초밥:실속(10pcs),13000,표시 가격 확인,초밥·회,초밥,모둠,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,휴무\r\n'
    'R73613ac879de,해빈,"서울 성동구 용답중앙길 80, 1층 (용답동)",37.5627474,127.0540098,일식,숙성회 : 모듬 숙성회(28pcs),45000,표시 가격 확인,미확인,미확인,모둠,미확인,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,휴무\r\n'
    'R73613ac879de,해빈,"서울 성동구 용답중앙길 80, 1층 (용답동)",37.5627474,127.0540098,일식,숙성회 : 연어회(14pcs),25000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,휴무\r\n'
    'R73613ac879de,해빈,"서울 성동구 용답중앙길 80, 1층 (용답동)",37.5627474,127.0540098,일식,숙성회 : 혼술 숙성회(9pcs),19000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,11:30-23:00,휴무\r\n'
    'Rb58464e10ee8,해원횟집,서울 성동구 용답중앙15길 5 (용답동) 1층,37.5641202,127.0530845,일식,고등어구이,10000,표시 가격 확인,구이·볶음·찜,생선구이,단품,식사,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00\r\n'
    'Rb58464e10ee8,해원횟집,서울 성동구 용답중앙15길 5 (용답동) 1층,37.5641202,127.0530845,일식,대구탕,9000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00\r\n'
    'Rb58464e10ee8,해원횟집,서울 성동구 용답중앙15길 5 (용답동) 1층,37.5641202,127.0530845,일식,매운탕,8000,표시 가격 확인,국물요리,매운탕·해물탕,단품,식사,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00\r\n'
    'Rb58464e10ee8,해원횟집,서울 성동구 용답중앙15길 5 (용답동) 1층,37.5641202,127.0530845,일식,전어회,25000,표시 가격 확인,초밥·회,회·사시미,단품,식사,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00\r\n'
    'Rb58464e10ee8,해원횟집,서울 성동구 용답중앙15길 5 (용답동) 1층,37.5641202,127.0530845,일식,회덮밥,8000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00,11:00-다음 날 02:00\r\n'
    'Rf0d922e4a28d,해품장팔팔장어 성동점,서울 성동구 천호대로 244 판매시설동 2층 2208~2211호,37.5690508,127.0481414,해산물 구이/찜,특허받은 명품장어탕(공깃밥포함),10000,표시 가격 확인,국물요리,국·탕,단품,식사,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'Rf0d922e4a28d,해품장팔팔장어 성동점,서울 성동구 천호대로 244 판매시설동 2층 2208~2211호,37.5690508,127.0481414,해산물 구이/찜,풍천장어 2인분(실중량 540g),75000,표시 가격 확인,미확인,미확인,단품,미확인,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00,11:30-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,감자사리,2000,표시 가격 확인,국수류,면,추가,추가옵션,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,닭볶음탕,32000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,닭볶음탕한마리반,48000,표시 가격 확인,구이·볶음·찜,닭볶음탕,단품,식사,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,닭한마리,30000,표시 가격 확인,국물요리,닭한마리,단품,식사,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,닭한마리반,45000,표시 가격 확인,국물요리,닭한마리,단품,식사,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,떡/칼국수/라면 사리,2000,표시 가격 확인,국수류,라면,추가,추가옵션,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,볶음밥,2000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R63dfad5d5159,행당닭한마리,행당로17길 18,37.5599831,127.0333734,닭/오리고기 구이/찜,삼계탕,16000,표시 가격 확인,국물요리,삼계탕·백숙,단품,식사,11:00-22:00,휴무,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,감자만두(4알),3000,표시 가격 확인,분식,만두,단품,사이드,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,감자만두(8알),5000,표시 가격 확인,분식,만두,단품,사이드,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,들깨수제비,9000,표시 가격 확인,국수류,수제비,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,매콤비빔칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,바지락칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,보리비빔밥,9000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,안동찜닭(반마리),22000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,안동찜닭(한마리),44000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,안동찜닭(한마리반),65000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,얼큰행칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,콩칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,팥칼국수,10000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'R24ad63055ab3,행복한 칼국수,"서울 성동구 용답29길 10-1, 지상1층 (용답동, 8-10)",37.5638443,127.0550035,경양식,팥칼국수(옹심이만),12000,표시 가격 확인,국수류,칼국수,단품,식사,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-20:00,11:00-19:30,휴무\r\n'
    'Rbf7c5b347ca9,행운돈까스,서울 성동구 마조로1길 2,37.5583679,127.0403026,,돈까스 + 김치볶음밥,8000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무\r\n'
    'Rbf7c5b347ca9,행운돈까스,서울 성동구 마조로1길 2,37.5583679,127.0403026,,새우정식,9000,표시 가격 확인,여러 요리 구성,미확인,세트,식사 추정,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무\r\n'
    'Rbf7c5b347ca9,행운돈까스,서울 성동구 마조로1길 2,37.5583679,127.0403026,,치즈돈까스,9000,표시 가격 확인,튀김·커틀릿,돈가스,단품,식사,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,10:30-21:00,휴무\r\n'
    'R37ab954ad427,형제정육식당,서울 중구 다산로40길 56 1층(신당5동),37.5631846,127.0190717,소고기 구이/찜,김치찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R37ab954ad427,형제정육식당,서울 중구 다산로40길 56 1층(신당5동),37.5631846,127.0190717,소고기 구이/찜,된장찌개,7000,표시 가격 확인,국물요리,찌개류,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R37ab954ad427,형제정육식당,서울 중구 다산로40길 56 1층(신당5동),37.5631846,127.0190717,소고기 구이/찜,제육볶음,7000,표시 가격 확인,구이·볶음·찜,제육볶음,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'R8bbb10b5a1a5,호갱,"서울 성동구 왕십리로24길 18, 3층 (도선동)",37.5639992,127.0343461,경양식,머쉬룸 크림 숏파스타,16900,표시 가격 확인,파스타류,파스타,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:30-22:00,휴무\r\n'
    'R8bbb10b5a1a5,호갱,"서울 성동구 왕십리로24길 18, 3층 (도선동)",37.5639992,127.0343461,경양식,청양크림 파스타,16900,표시 가격 확인,파스타류,파스타,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:30-22:00,휴무\r\n'
    'R8bbb10b5a1a5,호갱,"서울 성동구 왕십리로24길 18, 3층 (도선동)",37.5639992,127.0343461,경양식,홍게 크림 페투치네,17900,표시 가격 확인,파스타류,파스타,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:30-22:00,휴무\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,냉채족발 대,54000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,냉채족발 중,48000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,막걸리,4000,표시 가격 확인,음료,막걸리,단품,음료,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,맥주,5000,표시 가격 확인,음료,맥주,단품,음료,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,미니족 추가,11000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,반반메뉴 대,58000,표시 가격 확인,미확인,미확인,단품,미확인,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,반반메뉴 소,44000,표시 가격 확인,미확인,미확인,단품,미확인,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,반반메뉴 중,52000,표시 가격 확인,미확인,미확인,단품,미확인,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,보쌈 대,48000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,보쌈 소,35000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,보쌈 중,42000,표시 가격 확인,구이·볶음·찜,보쌈·수육,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,보쌈+족발+쟁반막국수+음료 세트 대,96000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,보쌈+족발+쟁반막국수+음료 세트 소,66000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,보쌈+족발+쟁반막국수+음료 세트 중,84000,표시 가격 확인,구이·볶음·찜,보쌈·수육,세트,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,불족발 대,50000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,불족발 소,37000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,불족발 중,44000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,소주,5000,표시 가격 확인,음료,소주,단품,음료,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,어리굴젓,9000,표시 가격 확인,미확인,미확인,단품,미확인,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,음료수,2000,표시 가격 확인,음료,기타 음료,단품,음료,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,쟁반막국수 대,12000,표시 가격 확인,국수류,막국수,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,쟁반막국수 소,6000,표시 가격 확인,국수류,막국수,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,쟁반막국수 중,9000,표시 가격 확인,국수류,막국수,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,족발 대,48000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,족발 소,35000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,족발 중,42000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,주먹밥,3000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'Rc669c4804db3,호야네족발,무학봉26길 9 1층,37.5617778,127.0328334,한식,추가 계란찜,2000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,14:00-22:50,12:00-22:50,12:00-22:50\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,계란탕,5000,표시 가격 확인,국물요리,국·탕,단품,식사,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,뼈없는 닭발(2인분기본),30000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,뼈없는 닭발(3인분),45000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,뼈없는 닭발(4인분),60000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,뼈있는 닭발(2인분기본),28000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,뼈있는 닭발(3인분),42000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R003917291278,호호호닭발,"서울 성동구 무학로2길 31 (도선동,(지상1층))",37.5633706,127.0334463,닭/오리고기 구이/찜,주먹밥,5000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00,15:00-다음 날 02:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,막걸리,5000,표시 가격 확인,음료,막걸리,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,매운미니족,22000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,매운족발,39000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,사이다,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,새로,5000,표시 가격 확인,음료,소주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,쟁반국수,10000,표시 가격 확인,국수류,국수,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,전통족발,36000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,전통족발+매운족발,49000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,전통족발+튀김족발,49000,표시 가격 확인,구이·볶음·찜,족발,단품,식사,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,제로사이다,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,제로콜라,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,주먹밥,4000,표시 가격 확인,곁들임,주먹밥·곁들임밥,단품,사이드,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,진로,5000,표시 가격 확인,음료,소주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,참이슬 오리지널,5000,표시 가격 확인,음료,소주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,참이슬 후레시,5000,표시 가격 확인,음료,소주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,처음처럼,5000,표시 가격 확인,음료,소주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,청하,6000,표시 가격 확인,음료,전통주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,카스,5000,표시 가격 확인,음료,맥주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,켈리,5000,표시 가격 확인,음료,맥주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,콜라,2000,표시 가격 확인,음료,탄산음료,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,테라,5000,표시 가격 확인,음료,맥주,단품,음료,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R23eb8917766c,홍능족발1977,"서울 성동구 왕십리로 309, 왕십리역이스타빌 1층 107호 (행당동)",37.5611984,127.0346099,한식,튀김족발,39000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,휴무,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),냉면,8000,표시 가격 확인,국수류,냉면,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),냉모밀,8000,표시 가격 확인,국수류,모밀·소바,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),들깨 수제비,9000,표시 가격 확인,국수류,수제비,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),들깨칼국수,9000,표시 가격 확인,국수류,칼국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),수제비,6500,표시 가격 확인,국수류,수제비,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),열무비빔밥,8000,표시 가격 확인,밥·죽류,비빔밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),왕만두,6000,표시 가격 확인,분식,만두,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),제육덮밥,8000,표시 가격 확인,밥·죽류,덮밥,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),칼국수,6000,표시 가격 확인,국수류,칼국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),칼만두,9000,표시 가격 확인,국수류,칼국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),칼제비,6500,표시 가격 확인,국수류,칼국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),콩국수,10000,표시 가격 확인,국수류,국수,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'R2a63638872e1,홍두깨식당,"서울 성동구 용답중앙15길 1 (용답동, 용답동 57-20 지상1층)",37.5639311,127.0528466,통닭(치킨),홍어무침,15000,표시 가격 확인,곁들임,반찬·무침,단품,사이드,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무,10:00-21:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,순살찜닭 대,56000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,순살찜닭 소,32000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,순살찜닭 중,45000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,찜닭 대,54000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,찜닭 소,28000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,찜닭 중,41000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,치즈찜닭 소,31000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,치즈찜닭 중,45000,표시 가격 확인,구이·볶음·찜,찜닭,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,해물찜닭 대,61000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,해물찜닭 소,35000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rae4db466290e,홍수계찜닭 왕십리민자역사점,"서울 성동구 왕십리광장로 17, 4층 (행당동)",37.5612890,127.0383818,한식,해물찜닭 중,48000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00\r\n'
    'Rf06d4dfc96a9,황귀닭곰탕,"서울 성동구 마장로 310 (마장동,(1층))",37.5664695,127.0440043,한식,닭곰탕,8000,표시 가격 확인,국물요리,곰탕·설렁탕,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'Rf06d4dfc96a9,황귀닭곰탕,"서울 성동구 마장로 310 (마장동,(1층))",37.5664695,127.0440043,한식,닭다리 백반,10000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00,00:00-24:00\r\n'
    'R8691363ee6a8,황금 가든,"서울 성동구 고산자로6길 40, 행당동 레몬프라자 105호,106호 (행당동)",37.5582253,127.0354812,백반/한정식,곤드레밥,13000,표시 가격 확인,밥·죽류,솥밥,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'R8691363ee6a8,황금 가든,"서울 성동구 고산자로6길 40, 행당동 레몬프라자 105호,106호 (행당동)",37.5582253,127.0354812,백반/한정식,국내산 돼지갈비+목살(200g),16000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30,11:00-21:30\r\n'
    'Ra653a844ec9b,황금알,왕십리로21길 11 1층,37.5613084,127.0331460,백반/한정식,껍데기,10000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra653a844ec9b,황금알,왕십리로21길 11 1층,37.5613084,127.0331460,백반/한정식,꼼장어,15000,표시 가격 확인,구이·볶음·찜,해산물 볶음·구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra653a844ec9b,황금알,왕십리로21길 11 1층,37.5613084,127.0331460,백반/한정식,꽃닭발,15000,표시 가격 확인,구이·볶음·찜,직화닭발,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra653a844ec9b,황금알,왕십리로21길 11 1층,37.5613084,127.0331460,백반/한정식,똥집,10000,표시 가격 확인,미확인,미확인,단품,미확인,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Ra653a844ec9b,황금알,왕십리로21길 11 1층,37.5613084,127.0331460,백반/한정식,통갈매기살,17000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00,16:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,곱창구이,23000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,곱창구이모듬,43500,표시 가격 확인,구이·볶음·찜,곱창·막창구이,모둠,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,곱창전골,18000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,대창구이,23000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,대창전골,18000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,라면사리,2500,표시 가격 확인,국수류,라면,추가,추가옵션,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,막창구이,23000,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,벌집구이,23000,표시 가격 확인,미확인,미확인,단품,미확인,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,소모듬야채볶음,15000,표시 가격 확인,구이·볶음·찜,볶음요리,모둠,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,쓰리창,48500,표시 가격 확인,미확인,미확인,단품,미확인,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,알볶음밥,4000,표시 가격 확인,밥·죽류,볶음밥,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,염통구이,19500,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'Rb10ef0aa028c,황소곱창,서울 성동구 마조로5길 3 (행당동),37.5598997,127.0407133,한식,특양구이,27500,표시 가격 확인,구이·볶음·찜,곱창·막창구이,단품,식사,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00,12:00-24:00\r\n'
    'R1ce65aae0cec,후참치,"서울 성동구 용답19가길 24, 2층 (용답동)",37.5638682,127.0525415,일식,참치 코스 1인,45000,표시 가격 확인,여러 요리 구성,세부유형 미정,코스,식사,17:30-22:00,17:30-22:00,17:30-22:00,17:30-22:00,17:30-22:00,17:30-21:00,휴무\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,계란 추가,1000,표시 가격 확인,곁들임,달걀,추가,추가옵션,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,기린 병맥주 330ml,5000,표시 가격 확인,음료,맥주,단품,음료,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,닭껍질튀김,8000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,돈코츠라멘,11000,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,돈코츠카라이멘,11500,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,마제소바,12000,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,마제카라이소바,12500,표시 가격 확인,국수류,모밀·소바,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,음료수,2000,표시 가격 확인,음료,기타 음료,단품,음료,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,차슈 추가,2000,표시 가격 확인,추가옵션,토핑·추가,추가,추가옵션,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,치킨 가라아게,7000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,튀김만두,5000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,표고버섯튀김,3500,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,후쿠쵸 라멘,11500,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R23bdb8e83686,후쿠쵸라멘 본점,서울 성동구 왕십리로 410 센트라스 상가L동 지하1층 125호,37.5661105,127.0276465,일본라멘,후쿠쵸 카라이멘,12000,표시 가격 확인,국수류,일본식 라멘,단품,식사,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00,11:00-21:00\r\n'
    'R381bf0230b3d,흑돈연가,서울 중구 퇴계로88길 20 1층 (신당5동),37.5647610,127.0208831,백반/한정식,삼겹살(180g),15000,표시 가격 확인,구이·볶음·찜,돼지고기구이,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R381bf0230b3d,흑돈연가,서울 중구 퇴계로88길 20 1층 (신당5동),37.5647610,127.0208831,백반/한정식,점심백반,7000,표시 가격 확인,밥·죽류,백반·한상,단품,식사,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,10:00-21:00,휴무\r\n'
    'R65f512e838b1,희령이네 코다리찜,"서울 성동구 무학봉26길 2, 1층 101호 (하왕십리동, 가온팰리스)",37.5618753,127.0321327,해산물 구이/찜,코다리찜(대),68000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R65f512e838b1,희령이네 코다리찜,"서울 성동구 무학봉26길 2, 1층 101호 (하왕십리동, 가온팰리스)",37.5618753,127.0321327,해산물 구이/찜,코다리찜(소),43000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'R65f512e838b1,희령이네 코다리찜,"서울 성동구 무학봉26길 2, 1층 101호 (하왕십리동, 가온팰리스)",37.5618753,127.0321327,해산물 구이/찜,코다리찜(중),55000,표시 가격 확인,구이·볶음·찜,찜·조림,단품,식사,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,11:00-22:00,휴무\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,[평일런치] 가라아게 정식 구성 : 가라아게+밥+미소시루+샐러드+코바치,10000,표시 가격 확인,튀김·커틀릿,튀김,세트,사이드,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,생맥주(Cass),5000,표시 가격 확인,음료,맥주,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,솔티리치 맥주/츄하이,10000,표시 가격 확인,음료,주류,단품,음료,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,야끼오니기리(2개),6000,표시 가격 확인,초밥·회,초밥,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,야끼토리(4꼬치),10000,표시 가격 확인,구이·볶음·찜,꼬치구이,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,오야코동,13000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,오코노미야끼,13000,표시 가격 확인,곁들임,전·부침,단품,사이드,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,우엉튀김,10000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,치킨난반,18000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,치킨명란마요,18000,표시 가격 확인,튀김·커틀릿,치킨,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,카라아게,18000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,카라아게 카레/ 카츠카레,17000,표시 가격 확인,커리·스튜,커리,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,카츠동,15000,표시 가격 확인,밥·죽류,덮밥,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,히마와리고로케,15000,표시 가격 확인,튀김·커틀릿,튀김,단품,사이드,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,히마와리나베,28000,표시 가격 확인,국물요리,전골,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,히마와리샐러드,13000,표시 가격 확인,샐러드,샐러드,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,히마와리스키야키,28000,표시 가격 확인,국물요리,전골,단품,식사,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
    'Rdd9ce9f2848e,히마와리,"서울 성동구 무학로2길 33-2, 지상1층 (도선동)",37.5633078,127.0336047,일식,히마와리크래커,10000,표시 가격 확인,미확인,미확인,단품,미확인,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,11:00-24:00,18:00-다음 날 01:00,18:00-다음 날 01:00\r\n'
)


In [ ]:
from io import StringIO

TEST_MODE = True

def load_test_menu_csv():
    """노트북에 포함된 식당 CSV를 표로 불러온다."""
    import pandas as pd

    if 'TEST_MENU_CSV_TEXT' not in globals():
        raise RuntimeError('0번의 테스트용 CSV 원본 셀을 먼저 실행해 주세요.')
    return pd.read_csv(StringIO(TEST_MENU_CSV_TEXT), dtype=str, keep_default_na=False)


test_menu_data = load_test_menu_csv()
print(f'원본 CSV: {len(test_menu_data):,}행, {len(test_menu_data.columns)}열')


원본 CSV: 3,795행, 20열


## 1. 데이터 처리 함수

### `walking_route`
#### 입력
- `loc`, `restaurant`: 출발지와 식당의 좌표 (`x`: 경도, `y`: 위도)
- `kakao_rest_key`: 카카오 REST API 키

#### 출력
- `walk_distance_m`(m), `walk_seconds`(초), `walk_url`(경로 링크)을 담은 `dict`
  - 경로 조회 실패: `None`
  - 출발지와 도착지가 같을 때: 거리와 시간은 `0`

### `normalize_data`
#### 입력
- `restaurants`: 식당명, 업종, 도보 거리와 시간, 영업 여부, `menu_rows`를 담은 식당 목록

#### 출력
- 메뉴당 한 개의 `dict`를 담은 리스트
  - 식당명, 업종, 메뉴명, 가격(원), 음식유형, 도보 거리(m), 예상 도보 시간(분), 영업 여부
  - 가격: 정수 또는 `None`. 도보 거리와 시간: 조회값 또는 `None`
  - 영업 여부: `영업 중` / `영업시간 아님` / `확인 불가`

### `prepare_history`
#### 입력
- `history`: `{date, meal, status, menu, cuisine}` 형식의 식사 기록 목록
  - `date`: 날짜 (`YYYY-MM-DD`)
  - `meal`: 끼니 (`아침` / `점심` / `저녁`)
  - `status`: 식사 상태 (`eaten` / `unknown` / `skipped`)
  - `menu`, `cuisine`: 메뉴명과 음식 계열 (문자열 또는 `None`)
- `reference_date`: 기준 날짜 (`YYYY-MM-DD`)
- `reference_meal`: 기준 끼니 (`아침` / `점심` / `저녁`)

#### 출력
- 기준 끼니 직전 6끼에 해당하는 식사 기록 목록
  - `unknown`, `skipped`: 메뉴명과 음식 계열은 `None`
  - 누락 기록 추가 및 중복 제거 없음

### `straight_distance_m`
#### 입력
- `origin`, `target`: 출발지와 목적지의 `(경도, 위도)` 튜플 (°)

#### 출력
- 두 좌표 사이 직선거리(m)


In [ ]:
from datetime import date, timedelta
import json
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


def walking_route(loc, restaurant, kakao_rest_key):
    """카카오에서 출발지부터 식당까지의 도보 경로를 조회한다."""
    params = {'start_x': loc['x'], 'start_y': loc['y'],
              'end_x': restaurant['x'], 'end_y': restaurant['y']}
    url = 'https://dapi.kakao.com/v2/routing/walk?' + urlencode(params)
    req = Request(url, headers={'Authorization': 'KakaoAK ' + kakao_rest_key})
    try:
        with urlopen(req, timeout=5) as response:
            data = json.load(response)
        if data.get('status') == 'SAME_POINT':
            return {'walk_distance_m': 0, 'walk_seconds': 0, 'walk_url': None}
        if data.get('status') != 'OK':
            return None
        props = data['route']['properties']
    except (URLError, TimeoutError, ValueError, KeyError, TypeError):
        return None
    distance, seconds = props.get('totalDistance'), props.get('totalTime')
    return {
        'walk_distance_m': distance if type(distance) is int and distance >= 0 else None,
        'walk_seconds': seconds if type(seconds) is int and seconds >= 0 else None,
        'walk_url': props.get('landingUrl'),
    }


def normalize_data(restaurants):
    """식당별 메뉴를 한 행에 하나씩 펼쳐 결과표에 필요한 항목을 정리한다."""
    labels = {True: '영업 중', False: '영업시간 아님', None: '확인 불가'}
    return [
        {
            '식당명': restaurant['restaurant'],
            '업종': restaurant['category_name'],
            '메뉴명': menu['메뉴명'],
            '가격': int(menu['가격_원']) if menu['가격_원'] else None,
            '음식유형': menu['음식유형'],
            '도보 거리(m)': restaurant['walk_distance_m'],
            '예상 도보 시간(분)': restaurant['walk_min'],
            '영업 여부': labels[restaurant['is_open']],
        }
        for restaurant in restaurants
        for menu in restaurant['menu_rows']
    ]


def prepare_history(history, reference_date, reference_meal):
    """기준 끼니 직전 6끼에 해당하는 식사 기록을 추린다."""
    meals = ('아침', '점심', '저녁')
    end = date.fromisoformat(reference_date).toordinal() * 3 + meals.index(reference_meal)
    result = []
    for record in history:
        position = date.fromisoformat(record['date']).toordinal() * 3 + meals.index(record['meal'])
        if end - 6 <= position < end:
            status = record['status']
            result.append({
                'date': record['date'], 'meal': record['meal'], 'status': status,
                'menu': record['menu'] if status == 'eaten' else None,
                'cuisine': record['cuisine'] if status == 'eaten' else None,
            })
    return result

from math import radians, sin, cos, asin, sqrt


def straight_distance_m(origin, target):
    """두 좌표 사이의 직선거리를 미터로 계산한다."""
    lon1, lat1, lon2, lat2 = map(radians, (*origin, *target))
    h = sin((lat2 - lat1) / 2)**2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2)**2
    return 2 * 6_371_000 * asin(sqrt(min(1, h)))


## 2. 최근 6끼 입력

### 끼니 분류 기준
- 22:30부터 다음 날 10:30 전까지: 아침 (22:30부터 기준 날짜는 다음 날)
- 10:30부터 16:00 전까지: 점심
- 16:00부터 22:30 전까지: 저녁

### 식사 기록
#### 입력
- 현재 한국 시각, `TEST_MODE` (`False`: 직접 입력, `True`: 예시 기록)
- 끼니별 메뉴명, `모름` 또는 `안 먹음`
  - 빈 입력: `모름`
  - 메뉴 입력 시 음식 계열 추가 입력

#### 출력
- `REFERENCE_DATE`, `REFERENCE_MEAL`: 기준 날짜와 끼니
- `recent_history`: 기준 끼니 직전 6끼 기록 (오늘 점심 기준, 엊그제 점심부터 오늘 아침까지)
- `recommendation_history`: `recent_history`에서 먹은 끼니만 추린 목록


In [ ]:
from datetime import datetime, timezone

kst = timezone(timedelta(hours=9))
now = datetime.now(kst)
reference = now.date()
minutes = now.hour * 60 + now.minute
if minutes >= 22 * 60 + 30:
    REFERENCE_MEAL = '아침'
    reference += timedelta(days=1)
elif minutes < 10 * 60 + 30:
    REFERENCE_MEAL = '아침'
elif minutes < 16 * 60:
    REFERENCE_MEAL = '점심'
else:
    REFERENCE_MEAL = '저녁'
REFERENCE_DATE = reference.isoformat()
meals = ('아침', '점심', '저녁')
print(f"한국 시간 {now:%Y-%m-%d %H:%M} / 기준: {REFERENCE_DATE} {REFERENCE_MEAL}")

if TEST_MODE:
    sample_meals = [
        ('eaten', '김밥', '분식'),
        ('eaten', '된장찌개', '한식'),
        ('unknown', None, None),
        ('eaten', '짜장면', '중식'),
        ('skipped', None, None),
        ('eaten', '비빔밥', '한식'),
    ]
else:
    print('직전 6끼를 입력하세요. 기억이 안 나면 Enter 또는 모름, 거른 끼니는 안 먹음을 입력하세요.')

history = []
for offset in range(-6, 0):
    day_offset, meal_index = divmod(meals.index(REFERENCE_MEAL) + offset, 3)
    meal_date = reference + timedelta(days=day_offset)
    meal = meals[meal_index]
    if TEST_MODE:
        status, menu, cuisine = sample_meals[offset + 6]
    else:
        menu = input(f'{meal_date.isoformat()} {meal}: ').strip()
        if menu in ('', '모름'):
            status, menu, cuisine = 'unknown', None, None
        elif menu == '안 먹음':
            status, menu, cuisine = 'skipped', None, None
        else:
            status = 'eaten'
            cuisine = input('음식 계열 (한식/중식/일식/양식/분식/기타): ').strip() or None
    history.append({'date': meal_date.isoformat(), 'meal': meal, 'status': status,
                    'menu': menu, 'cuisine': cuisine})

recent_history = prepare_history(history, REFERENCE_DATE, REFERENCE_MEAL)
# 추천에는 실제로 먹은 메뉴만 사용.
recommendation_history = [record for record in recent_history if record['status'] == 'eaten']
print('예시 식사 기록:' if TEST_MODE else '입력한 식사 기록:')
for record in recent_history:
    description = record['menu'] if record['status'] == 'eaten' else {'unknown': '모름', 'skipped': '안 먹음'}[record['status']]
    print(f"{record['date']} {record['meal']}: {description}")


한국 시간 2026-11-22 15:42 / 기준: 2026-11-22 점심
예시 식사 기록:
2026-11-20 점심: 김밥
2026-11-20 저녁: 된장찌개
2026-11-21 아침: 모름
2026-11-21 점심: 짜장면
2026-11-21 저녁: 안 먹음
2026-11-22 아침: 비빔밥


## 3. 주변 식당 검색

### `location_from_address`
#### 입력
- `key`: 카카오 REST API 키
- 실행 중 입력한 도로명 주소 또는 지번 주소 (카카오로 전송)

#### 출력
- 출발지 좌표 `{x: 경도, y: 위도}`
  - 검색 결과가 없거나 여러 건: 주소 재입력
  - 빈 주소 또는 API 호출 실패: 검색 중단

### `csv_opening_status`
#### 입력
- `menu_rows`: 한 식당의 CSV 원본 행 목록 (월요일~일요일 열 사용)
- `current_time`: 현재 한국 시각 (`datetime`)

#### 출력
- 영업 여부 (`True`: 영업 중, `False`: 영업시간 아님, `None`: 확인 불가)
  - 오늘 영업시간과 전날 자정을 넘긴 영업시간 기준

### `csv_restaurant_candidates`
#### 입력
- `menu_data`: 최종 CSV를 읽은 `DataFrame`
- `origin`: 출발지의 `(경도, 위도)` 튜플 (°)
- `current_time`: 현재 한국 시각 (`datetime`)
- `radius_m`: 직선거리 검색 반경(m), 현재 2,000 m

#### 출력
- 직선거리순 식당 후보 목록
  - 식당 ID, 이름, 업종, 좌표, 직선거리(m), 영업 여부, `menu_rows`
  - `menu_rows`: 메뉴명이 있는 `식사` 행 (대분류 `여러 요리 구성` 제외, 원본 열 유지)
  - 반경 밖, 남은 메뉴가 없는 식당, 술집·간식·요리 주점 제외

### 검색 셀 결과
- `data`: 식당 후보 `candidates`와 2번 셀의 `recent_history`
  - 직선거리 기준 가까운 식당 최대 100곳의 카카오 도보 경로 조회 (출발·도착 좌표 전송)
  - 도보 거리(m), 예상 시간(초), 분으로 올림한 예상 시간, 경로 링크. 미조회 값은 `None`
- `menu_results`: `normalize_data`로 정리한 메뉴별 8열 결과표 (직선거리 미표시)


In [ ]:
# 1. API 키와 위치 준비
import re


def location_from_address(key):
    """입력한 출발지 주소를 카카오에서 좌표로 변환한다."""
    while True:
        address = input('출발지 도로명/지번 주소 (카카오로 전송, Enter: 종료): ').strip()
        if not address:
            raise RuntimeError('주소를 입력하지 않아 검색을 중단했습니다.')
        url = 'https://dapi.kakao.com/v2/local/search/address.json?' + urlencode({'query': address})
        request = Request(url, headers={'Authorization': 'KakaoAK ' + key})
        try:
            with urlopen(request, timeout=15) as response:
                documents = json.load(response).get('documents', [])
        except (URLError, TimeoutError, ValueError) as error:
            raise RuntimeError('주소 검색에 실패했습니다. API 키와 인터넷 연결을 확인하세요.') from error
        if not documents:
            print('주소를 찾지 못했습니다. 도로명/지번 주소와 건물 번호를 다시 입력하세요.')
            continue
        if len(documents) != 1:
            print('여러 주소가 검색되었습니다. 건물 번호까지 입력하세요.')
            continue
        return {'x': float(documents[0]['x']), 'y': float(documents[0]['y'])}


def csv_opening_status(menu_rows, current_time):
    """CSV 영업시간과 현재 시각을 비교해 영업 여부를 판단한다."""
    days = ('월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일')

    def hours(day):
        """지정한 요일의 영업 시작과 종료 시각을 분 단위로 바꾼다."""
        values = {row.get(day, '').strip() for row in menu_rows}
        if values == {'휴무'}:
            return ()
        if len(values) != 1:
            return None
        value = next(iter(values))
        match = re.fullmatch(r'(\d{2}):(\d{2})\s*-\s*(?:(다음\s*날)\s*)?(\d{2}):(\d{2})', value)
        if not match:
            return None
        sh, sm, overnight, eh, em = match.groups()
        sh, sm, eh, em = map(int, (sh, sm, eh, em))
        if sh > 23 or sm > 59 or eh > 24 or em > 59 or (eh == 24 and em):
            return None
        start = sh * 60 + sm
        end = eh * 60 + em + (1440 if overnight else 0)
        return (start, end) if start < end <= start + 1440 else None

    weekday = current_time.weekday()
    minute = current_time.hour * 60 + current_time.minute
    today = hours(days[weekday])
    yesterday = hours(days[(weekday - 1) % 7])
    if today and today[0] <= minute < today[1]:
        return True
    if yesterday and yesterday[0] <= minute + 1440 < yesterday[1]:
        return True
    if today is None or yesterday is None:
        return None
    return False


def csv_restaurant_candidates(menu_data, origin, current_time, radius_m):
    """검색 반경 안에서 식사 메뉴가 있는 식당을 추린다."""
    rows = []
    for restaurant_id, group in menu_data.groupby('식당_ID', sort=False):
        source_rows = group.to_dict('records')
        menu_rows = [row for row in source_rows
                     if row['메뉴역할'].strip() == '식사'
                     and row['메뉴_대분류'].strip() != '여러 요리 구성'
                     and row['메뉴명'].strip()]
        if not menu_rows:
            continue
        first = source_rows[0]
        category = first['업종'] or None
        if {part.strip() for part in (category or '').split('>')} & {'술집', '간식', '요리 주점'}:
            continue
        try:
            x, y = float(first['경도']), float(first['위도'])
        except (TypeError, ValueError):
            continue
        if not (-180 <= x <= 180 and -90 <= y <= 90):
            continue
        distance = straight_distance_m(origin, (x, y))
        if distance > radius_m:
            continue
        rows.append({
            'id': restaurant_id, 'restaurant': first['식당명'], 'category_name': category,
            'x': x, 'y': y, 'distance_m': distance, 'menu_rows': menu_rows,
            'walk_min': None, 'walk_seconds': None,
            'walk_distance_m': None, 'walk_url': None,
            'is_open': csv_opening_status(source_rows, current_time),
        })
    return sorted(rows, key=lambda row: (row['distance_m'], row['id']))


import pandas as pd
from IPython.display import display
from google.colab import userdata
from google.colab.data_table import DataTable

if globals().get('test_menu_data') is None:
    if 'load_test_menu_csv' not in globals():
        raise RuntimeError('0번의 CSV 설정 셀을 먼저 실행해 주세요.')
    test_menu_data = load_test_menu_csv()

SEARCH_RADIUS_M = 2000
WALK_LIMIT = 500
try:
    api_key = (userdata.get('KAKAO_REST_KEY') or '').strip()
except Exception:
    api_key = ''

if not api_key and TEST_MODE:
    # 왕십리로 222에서 조회했을 때 영업 중이었던 세 곳 (식당 ID, 도보 m, 도보 분)
    sample = [('R5d66493feb24', 392, 8), ('Rdd1c4f5c648b', 392, 8), ('R86c646cae74a', 865, 15)]
    print('API 키가 없어 저장된 식당 3곳으로 예시를 실행합니다.')
    rows = []
    for restaurant_id, meters, minutes in sample:
        group = test_menu_data[test_menu_data['식당_ID'] == restaurant_id]
        menus = group[(group['메뉴역할'] == '식사') & (group['메뉴_대분류'] != '여러 요리 구성') & (group['메뉴명'] != '')].to_dict('records')
        if menus:
            rows.append({'id': restaurant_id, 'restaurant': menus[0]['식당명'],
                         'category_name': menus[0]['업종'], 'menu_rows': menus,
                         'walk_distance_m': meters, 'walk_min': minutes, 'is_open': True})
else:
    if not api_key:
        from getpass import getpass
        api_key = getpass('카카오 REST API 키: ').strip()
        if not api_key:
            raise RuntimeError('직접 주소를 조회하려면 카카오 REST API 키가 필요합니다.')
    location = location_from_address(api_key)
    print('출발 위치를 확인했어요.')

    rows = csv_restaurant_candidates(test_menu_data, (location['x'], location['y']),
                                     datetime.now(kst), SEARCH_RADIUS_M)
    route_cache = {}
    for row in rows[:WALK_LIMIT]:
        target = (row['x'], row['y'])
        if target not in route_cache:
            route_cache[target] = walking_route(location, row, api_key)
        route = route_cache[target]
        if route:
            row.update(route)
            if row['walk_seconds'] is not None:
                row['walk_min'] = (row['walk_seconds'] + 59) // 60

candidates = rows
data = {'candidates': candidates, 'history': recent_history}
columns = [
    '식당명', '업종', '메뉴명', '가격', '음식유형',
    '도보 거리(m)', '예상 도보 시간(분)', '영업 여부',
]
menu_results = pd.DataFrame(normalize_data(candidates), columns=columns)

if menu_results.empty:
    print(f'반경 {SEARCH_RADIUS_M:,} m 안에 식사 메뉴가 없어요.')
else:
    print(f'반경 {SEARCH_RADIUS_M:,} m: 식당 {len(candidates):,}곳, 식사 메뉴 {len(menu_results):,}개')
    display(DataTable(menu_results, include_index=False))


출발지 도로명/지번 주소 (카카오로 전송, Enter: 종료): 왕십리로 222
출발 위치를 확인했어요.
반경 2,000 m: 식당 262곳, 식사 메뉴 2,556개


,식당명,업종,메뉴명,가격,음식유형,도보 거리(m),예상 도보 시간(분),영업 여부
0,면24 한양대점,한식·분식,원조김밥,3300,김밥,392,8,영업시간 아님
1,면24 한양대점,한식·분식,참치김밥,4000,김밥,392,8,영업시간 아님
2,면24 한양대점,한식·분식,소고기김밥,4300,김밥,392,8,영업시간 아님
3,면24 한양대점,한식·분식,치즈김밥,3700,김밥,392,8,영업시간 아님
4,면24 한양대점,한식·분식,고추김밥,3700,김밥,392,8,영업시간 아님
...,...,...,...,...,...,...,...,...
2551,변산반도 고향집,None,한우 토시살 (300g),70000,소고기구이,3023,48,영업 중
2552,변산반도 고향집,None,한우갈비살 (300g),80000,소고기구이,3023,48,영업 중
2553,변산반도 고향집,None,한우등심 (300g),90000,소고기구이,3023,48,영업 중
2554,변산반도 고향집,None,한우안심 (300g),90000,소고기구이,3023,48,영업 중


# C2 검증, 필터 (C2-1~2)

IN → data: C1에서 구성된 딕셔너리 (candidates 리스트 + history 리스트)

OUT → data: 조건을 통과한 후보만 남긴 candidates 리스트 + 변경 없는 history

---

C2-1 입력 검증

validata_data: 후보 목록이 존재하는지 확인한다.

candidates가 빈 리스트이면, "근처에 식당이 없습니다."를 출력하고 None을 반환해 이후 단계를 중단한다.

후보가 하나 이상 존재하는 경우 data를 그대로 반환해 C2-2를 진행한다.

---

C2-2 조건 검사

filter_candidates: 후보 목록에서 현재 영업중이고 도보 이동 가능한 식당만 선별한다.

선별 조건: is_open == True, walk_min is not None and walk_min <= maximum(=15)

기존 후보 목록을 선별된 목록으로 갱신하고 data를 반환한다.

In [ ]:
# C2-1 입력 검증
def validate_data(data):
  if not data['candidates']:            # 후보 목록이 비어있는 경우
    print("근처에 식당이 없습니다.")
    return
  return data

# C2-2 조건 필터링
def filter_candidates(data, maximum=15):
  # 후보 목록(candidates)에서 is_open == True, walk_min <= 15인 후보들만 suggests 리스트로 생성
  suggests = [cand for cand in data['candidates']
              if cand['is_open'] and cand['walk_min'] is not None and cand['walk_min'] <= maximum]

  data['candidates'] = suggests # 기존 후보 목록을 suggests로 변경

  return data

# 테스트
import copy

test_data = copy.deepcopy(data)
total = len(test_data['candidates'])

result = validate_data(test_data)
if result is None:
    print("validate_data: 후보 없음으로 종료")
else:
    result = filter_candidates(result)
    passed = len(result['candidates'])
    print(f"필터 통과: {passed}곳 / 전체: {total}곳")

filtered = pd.DataFrame(normalize_data(result["candidates"]), columns=columns)
display(DataTable(filtered, include_index=False))

필터 통과: 7곳 / 전체: 262곳


,식당명,업종,메뉴명,가격,음식유형,도보 거리(m),예상 도보 시간(분),영업 여부
0,프레퍼스 한양대플라자점,샐러드,비프 플레이트 샐러드,12900,샐러드,392,8,영업 중
1,프레퍼스 한양대플라자점,샐러드,치킨 콥 플레이트,11900,치킨,392,8,영업 중
2,프레퍼스 한양대플라자점,샐러드,치킨 플레이트 샐러드,10900,치킨,392,8,영업 중
3,프레퍼스 한양대플라자점,샐러드,포크 플레이트 샐러드,11900,샐러드,392,8,영업 중
4,엄마손한식뷔페,한식뷔페,제육볶음,6000,제육볶음,848,14,영업 중
...,...,...,...,...,...,...,...,...
102,버거리 한양대점,버거,프레쉬버거 세트,9000,버거,856,14,영업 중
103,버거리 한양대점,버거,하와이안버거,7700,버거,856,14,영업 중
104,버거리 한양대점,버거,하와이안버거 세트,11400,버거,856,14,영업 중
105,버거리 한양대점,버거,핫치폴레버거,7500,버거,856,14,영업 중


# C3 메뉴 점수 계산, 정렬

IN → result: C2를 통과한 식당 목록(candidates)과 최근 6끼 기록(history)

OUT → ranked: 메뉴별 점수를 담고 추천 점수가 높은 순서로 정렬한 리스트

---

최근 식사와 비교

score_candidates: 각 후보 메뉴를 최근 6끼 중 실제로 먹은 메뉴와 하나씩 비교한다. 식사 기록의 음식유형과 대분류는 메뉴 목록(catalog)에서 찾는다.

overlap_points: 메뉴명이 같으면 10점, 음식유형이 같으면 6점, 대분류만 같으면 3점, 모두 다르면 0점을 준다. 여러 조건이 맞으면 가장 높은 점수 하나만 사용한다.

직전 끼니부터 차례로 10, 9, 8, 7, 6, 5를 곱한다. ‘모름’과 ‘안 먹음’은 해당 자리만 건너뛴다.

가장 큰 값을 100으로 나눠 유사도로 저장하고, 그때의 식사 이름도 함께 저장한다. 동점이면 더 최근 식사를 선택한다. 비교할 기록이 없으면 두 값 모두 None으로 둔다.

결과를 메뉴별 리스트 table_rows로 반환해 C3-2에 넘긴다.

In [ ]:
def clean_category(value):
    if not value:
        return ''
    value = value.strip()
    if value in ['미확인', '세부유형 미정']:
        return ''
    return value


def lookup_category(name, catalog):
    exact = []
    partial = []
    for row in catalog:
        title = row['메뉴명'].strip()
        if title == name:
            exact.append(row)
        elif name in title:
            partial.append(row)

    matches = exact
    if not matches:
        matches = partial

    counts = {}
    for row in matches:
        pair = (clean_category(row['메뉴_대분류']), clean_category(row['음식유형']))
        if pair not in counts:
            counts[pair] = 0
        counts[pair] += 1

    best_pair = ('', '')
    best_count = 0
    for pair in counts:
        if counts[pair] > best_count:
            best_pair = pair
            best_count = counts[pair]
    return best_pair


def overlap_points(menu, past):
    if menu['메뉴명'].strip() == past['메뉴명']:
        return 10
    if past['음식유형'] and clean_category(menu['음식유형']) == past['음식유형']:
        return 6
    if past['대분류'] and clean_category(menu['메뉴_대분류']) == past['대분류']:
        return 3
    return 0


def meal_number(day, meal):
    order = {'아침': 0, '점심': 1, '저녁': 2}
    return date.fromisoformat(day).toordinal() * 3 + order[meal]


def score_candidates(data, catalog, reference_date, reference_meal):
    current = meal_number(reference_date, reference_meal)
    history = []
    for record in data['history']:
        if record['status'] != 'eaten' or not record['menu']:
            continue
        ago = current - meal_number(record['date'], record['meal'])
        if 1 <= ago <= 6:
            name = record['menu'].strip()
            if name:
                group, kind = lookup_category(name, catalog)
                history.append({'메뉴명': name, '대분류': group, '음식유형': kind,
                                '가중치': 11 - ago})

    rows = []
    for place in data['candidates']:
        place_rows = normalize_data([place])
        for i in range(len(place_rows)):
            menu = place['menu_rows'][i]
            row = place_rows[i]
            row['유사도'] = None
            row['가장 비슷한 식사'] = None
            best = -1
            best_weight = 0
            for past in history:
                weight = past['가중치']
                points = overlap_points(menu, past) * weight
                if points > best or (points == best and weight > best_weight):
                    best = points
                    best_weight = weight
                    row['유사도'] = points / 100
                    row['가장 비슷한 식사'] = past['메뉴명']
            rows.append(row)
    return rows


catalog = []
for row in test_menu_data.to_dict('records'):
    if row['메뉴역할'].strip() == '식사':
        catalog.append(row)

table_rows = score_candidates(result, catalog, REFERENCE_DATE, REFERENCE_MEAL)
similarity_results = pd.DataFrame(table_rows, columns=columns + ['유사도', '가장 비슷한 식사'])
display(similarity_results.head(10))

,식당명,업종,메뉴명,가격,음식유형,도보 거리(m),예상 도보 시간(분),영업 여부,유사도,가장 비슷한 식사
0,프레퍼스 한양대플라자점,샐러드,비프 플레이트 샐러드,12900,샐러드,392,8,영업 중,0.00,비빔밥
1,프레퍼스 한양대플라자점,샐러드,치킨 콥 플레이트,11900,치킨,392,8,영업 중,0.00,비빔밥
2,프레퍼스 한양대플라자점,샐러드,치킨 플레이트 샐러드,10900,치킨,392,8,영업 중,0.00,비빔밥
3,프레퍼스 한양대플라자점,샐러드,포크 플레이트 샐러드,11900,샐러드,392,8,영업 중,0.00,비빔밥
4,엄마손한식뷔페,한식뷔페,제육볶음,6000,제육볶음,848,14,영업 중,0.00,비빔밥
5,우동 가조쿠 본점,일식 면 요리,가마타마,9000,우동,812,14,영업 중,0.24,짜장면
6,우동 가조쿠 본점,일식 면 요리,가조쿠우동,9000,우동,812,14,영업 중,0.24,짜장면
7,우동 가조쿠 본점,일식 면 요리,라유멘,10000,일본식 라멘,812,14,영업 중,0.24,짜장면
8,우동 가조쿠 본점,일식 면 요리,붓가케우동,7500,우동,812,14,영업 중,0.24,짜장면
9,우동 가조쿠 본점,일식 면 요리,육육우동,9500,우동,812,14,영업 중,0.24,짜장면


---

추천 점수 계산

score_sort: table_rows의 각 메뉴에 중복회피 점수, 도보 점수, 가격 점수를 매긴다.

중복회피 점수는 1에서 유사도를 뺀 값이다. 도보 점수는 0분일 때 1점이고 15분까지 일정하게 줄어든다. 가격 점수는 최저가일 때 1점, 최고가일 때 0점이며 가격이 모두 같으면 0.5점이다.

추천 점수 = 중복회피 점수 × 0.5 + 도보 점수 × 0.3 + 가격 점수 × 0.2

유사도나 가격이 없으면 해당 항목을 제외하고, 남은 항목의 가중치 합으로 나눈다.

---

점수순 정렬

ranking_order: 추천 점수가 높은 메뉴를 먼저 두고, 동점이면 도보시간이 짧은 메뉴를 먼저 둔다.

score_sort는 이 기준으로 정렬한 리스트 ranked를 반환한다.

In [ ]:
def ranking_order(row):
    # 추천 점수가 높은 순서, 동점이면 도보시간이 짧은 순서
    return (-row['추천 점수'], row['예상 도보 시간(분)'])


def score_sort(menu_rows):
    prices = []
    for row in menu_rows:
        if row['가격'] is not None:
            prices.append(row['가격'])
    if prices:
        low = min(prices)
        high = max(prices)
    ranked = []

    for row in menu_rows:
        walk = 1 - row['예상 도보 시간(분)'] / 15
        total = 0.3 * walk
        weight = 0.3

        variety = None
        if row['유사도'] is not None:
            variety = 1 - row['유사도']
            total += 0.5 * variety
            weight += 0.5

        price = None
        if row['가격'] is not None:
            if high == low:
                price = 0.5
            else:
                price = (high - row['가격']) / (high - low)
            total += 0.2 * price
            weight += 0.2

        item = row.copy()
        item['중복회피 점수'] = variety
        item['도보 점수'] = walk
        item['가격 점수'] = price
        item['추천 점수'] = total / weight
        ranked.append(item)

    ranked.sort(key=ranking_order)
    return ranked


ranked = score_sort(table_rows)
if ranked:
    best = ranked[0]
    print(f"1위 메뉴 · {best['식당명']} / {best['메뉴명']}")
    for field in ['중복회피 점수', '도보 점수', '가격 점수', '추천 점수']:
        value = best[field]
        if value is None:
            print(field, '제외')
        else:
            print(field, round(value, 3))

1위 메뉴 · 프레퍼스 한양대플라자점 / 치킨 플레이트 샐러드
중복회피 점수 1.0
도보 점수 0.467
가격 점수 0.819
추천 점수 0.804


# C4 상위 메뉴 선택, 출력

IN → ranked: C3에서 추천 점수순으로 정렬한 메뉴 리스트

OUT → recs: 추천 이유를 붙인 메뉴 최대 3개. 선택한 메뉴를 화면에 출력한다.

---

상위 메뉴 선택

select_top3: 점수순으로 보면서 같은 식당명에서 첫 메뉴만 고른다. 서로 다른 식당을 최대 3곳까지 추천한다.

각 메뉴에 최근 식사와 얼마나 겹치는지, 걸어서 몇 분 걸리는지를 적는다. 가격 점수가 1이면 후보 중 가장 저렴하다는 설명도 붙인다.

추천 이유를 추가한 리스트 recs를 반환한다.

---

결과 출력

display_results: recs에 있는 각 메뉴의 식당명, 메뉴명, 가격, 도보거리, 도보시간과 추천 이유를 순서대로 출력한다.

recs가 빈 리스트이면 "조건에 맞는 메뉴가 없습니다."를 출력하고 종료한다.

In [ ]:
def select_top3(ranked):
    recs = []
    selected_names = []
    for row in ranked:
        if row['식당명'] in selected_names:
            continue
        selected_names.append(row['식당명'])
        reasons = []
        if row['중복회피 점수'] is None:
            reasons.append('비교할 최근 식사 기록이 없어요.')
        elif row['중복회피 점수'] >= 0.5:
            reasons.append('최근에 먹은 음식과 겹치는 정도가 낮아요.')
        else:
            reasons.append(f"최근 식사 중 {row['가장 비슷한 식사']}와 가장 비슷해요.")
        reasons.append(f"걸어서 {row['예상 도보 시간(분)']}분이면 갈 수 있어요.")
        if row['가격 점수'] == 1:
            reasons.append('후보 중 가장 저렴해요.')
        item = row.copy()
        item['추천 이유'] = ' '.join(reasons)
        recs.append(item)
        if len(recs) == 3:
            break
    return recs


def display_results(recs):
    if not recs:
        print('조건에 맞는 메뉴가 없습니다.')
        return

    for i in range(len(recs)):
        row = recs[i]
        rank = i + 1
        price = '가격 미확인'
        if row['가격'] is not None:
            price = f"{row['가격']:,}원"
        meters = row['도보 거리(m)']
        distance = '거리 미확인'
        if meters is not None:
            distance = f'{meters:,.0f}m'
        print(f"추천 {rank} · {row['식당명']} / {row['메뉴명']}")
        print(f"  {price} · 도보 {distance}, {row['예상 도보 시간(분)']}분")
        print(f"  {row['추천 이유']}")


recs = select_top3(ranked)
display_results(recs)

추천 1 · 프레퍼스 한양대플라자점 / 치킨 플레이트 샐러드
  10,900원 · 도보 392m, 8분
  최근에 먹은 음식과 겹치는 정도가 낮아요. 걸어서 8분이면 갈 수 있어요.
추천 2 · 버거리 한양대점 / 코울슬로
  2,500원 · 도보 856m, 14분
  최근에 먹은 음식과 겹치는 정도가 낮아요. 걸어서 14분이면 갈 수 있어요. 후보 중 가장 저렴해요.
추천 3 · 엄마손한식뷔페 / 제육볶음
  6,000원 · 도보 848m, 14분
  최근에 먹은 음식과 겹치는 정도가 낮아요. 걸어서 14분이면 갈 수 있어요.
